# Day 08 - 2교시: Day 07 Kafka 실습 복습
> 검증된 CLI 실습 코드와 예상 출력으로 Day07 핵심 개념 복습

## 🎯 학습 목표
이 파트를 마치면 다음을 할 수 있습니다:

- 파티션 분배 메커니즘(Sticky Partitioner)을 이해하고 설명할 수 있다
- Consumer Group과 Offset의 동작 원리를 실습으로 확인할 수 있다
- 멀티 브로커 클러스터에서 Leader 재선출 과정을 관찰할 수 있다
- 실제 운영에서 발생할 수 있는 이슈와 해결방법을 알 수 있다

---

## 📚 전체 학습 흐름

| 순서 | 내용 | 시간 |
|------|------|------|
| 1 | 파티션 분배 실습 | 20분 |
| 2 | Consumer Group 병렬 처리 실습 | 20분 |
| 3 | Offset 관리 실습 | 25분 |
| 4 | 멀티 브로커 클러스터 실습 | 25분 |

---

## 🔧 사전 준비

### Docker Compose 환경 시작

```bash
# Day07에서 사용한 kafka-ui-demo 폴더로 이동
cd kafka-ui-demo

# Kafka 및 Kafka UI 시작
docker compose up -d

# 컨테이너 상태 확인
docker ps
```

**예상 출력:**
```
CONTAINER ID   IMAGE                           STATUS         PORTS                    NAMES
xxxxxxxxxxxx   apache/kafka:latest            Up ...         0.0.0.0:9092->9092/tcp   broker
xxxxxxxxxxxx   provectuslabs/kafka-ui:latest  Up ...         0.0.0.0:8080->8080/tcp   kafka-ui
```

---
## 1. 파티션 분배 실습

### 1-1. 토픽 생성 (파티션 3개)

```bash
docker exec broker /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --create \
  --topic partition-demo \
  --partitions 3
```

**예상 출력:**
```
Created topic partition-demo.
```

### 1-2. 토픽 정보 확인

```bash
docker exec broker /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --describe \
  --topic partition-demo
```

**예상 출력:**
```
Topic: partition-demo   TopicId: xxxxx   PartitionCount: 3   ReplicationFactor: 1
    Topic: partition-demo   Partition: 0    Leader: 1   Replicas: 1   Isr: 1
    Topic: partition-demo   Partition: 1    Leader: 1   Replicas: 1   Isr: 1
    Topic: partition-demo   Partition: 2    Leader: 1   Replicas: 1   Isr: 1
```

### 🚨 중요: Sticky Partitioner 이해하기

#### 문제 상황

Key 없이 메시지를 전송할 때, 라운드로빈 방식으로 균등하게 분배될 것으로 예상하지만,
실제로는 **Sticky Partitioner** 때문에 모든 메시지가 하나의 파티션에 몰릴 수 있습니다.

```
┌──────────────────────────────────────────────────────────────────────────┐
│                      Sticky Partitioner 동작 원리                         │
├──────────────────────────────────────────────────────────────────────────┤
│                                                                          │
│  Key가 없는 메시지 전송 시:                                                │
│                                                                          │
│  ❌ 예상 (라운드로빈):                                                     │
│     msg1 → P0, msg2 → P1, msg3 → P2, msg4 → P0 ...                       │
│                                                                          │
│  ✅ 실제 (Sticky Partitioner):                                            │
│     msg1, msg2, msg3, msg4, msg5, msg6 → 모두 P2 (한 배치로 묶임)          │
│                                                                          │
│  💡 이유:                                                                  │
│     - Kafka 2.4+에서 도입된 성능 최적화 기능                                │
│     - 같은 배치의 메시지를 같은 파티션으로 전송 → 네트워크 효율 ↑            │
│     - linger.ms 동안 모인 메시지가 하나의 배치로 처리됨                     │
│                                                                          │
└──────────────────────────────────────────────────────────────────────────┘
```

#### 기본 설정으로 메시지 전송 (Sticky Partitioner 동작)

```bash
# 메시지 6개 한 번에 전송 (기본 설정)
docker exec -i broker /opt/kafka/bin/kafka-console-producer.sh \
  --bootstrap-server localhost:9092 \
  --topic partition-demo << 'EOF'
주문1-아이폰
주문2-맥북
주문3-아이패드
주문4-에어팟
주문5-애플워치
주문6-맥미니
EOF
```

**실제 결과 (모든 메시지가 한 파티션에 몰림):**
```
# Partition 0: (비어있음)
# Partition 1: (비어있음)
# Partition 2: 모든 메시지 6개 (주문1~주문6)
```

### 1-3. Key 없이 메시지 전송 (파티션 분산 방법)

배치 크기를 작게 설정하고 메시지를 천천히 전송하면 분산됩니다.

```bash
# 배치 크기를 작게 설정하여 파티션 분산
for i in {1..6}; do
  echo "주문$i" | docker exec -i broker /opt/kafka/bin/kafka-console-producer.sh \
    --bootstrap-server localhost:9092 \
    --topic partition-demo \
    --producer-property batch.size=1 \
    --producer-property linger.ms=0
  sleep 0.1
done
```

**설정 설명:**
- `batch.size=1`: 배치 크기를 1로 설정 (메시지마다 별도 배치)
- `linger.ms=0`: 대기 시간 없이 즉시 전송
- `sleep 0.1`: 메시지 간 0.1초 간격으로 전송

### 1-4. 파티션별 메시지 확인

> 💡 **Consumer 종료 팁**: `--timeout-ms` 옵션을 사용하면 자동으로 종료됩니다.

```bash
# Partition 0 확인
docker exec broker /opt/kafka/bin/kafka-console-consumer.sh \
  --bootstrap-server localhost:9092 \
  --topic partition-demo \
  --partition 0 \
  --from-beginning \
  --timeout-ms 3000 2>/dev/null || true
```

**예상 출력 (Partition 0):**
```
주문1
주문3
주문5
```

```bash
# Partition 1 확인
docker exec broker /opt/kafka/bin/kafka-console-consumer.sh \
  --bootstrap-server localhost:9092 \
  --topic partition-demo \
  --partition 1 \
  --from-beginning \
  --timeout-ms 3000 2>/dev/null || true
```

**예상 출력 (Partition 1):**
```
주문4
```

```bash
# Partition 2 확인
docker exec broker /opt/kafka/bin/kafka-console-consumer.sh \
  --bootstrap-server localhost:9092 \
  --topic partition-demo \
  --partition 2 \
  --from-beginning \
  --timeout-ms 3000 2>/dev/null || true
```

**예상 출력 (Partition 2):**
```
주문2
주문6
```

> ⚠️ **참고**: 실제 파티션 분배는 네트워크 상황과 배치 처리에 따라 달라질 수 있습니다.

### 1-5. Key를 지정하여 메시지 전송

Key가 있으면 **Hash(Key) % 파티션 수**로 파티션이 결정되어 같은 Key는 항상 같은 파티션으로 갑니다.

```bash
docker exec -i broker /opt/kafka/bin/kafka-console-producer.sh \
  --bootstrap-server localhost:9092 \
  --topic partition-demo \
  --property parse.key=true \
  --property key.separator=: << 'EOF'
user-A:주문-A1
user-B:주문-B1
user-A:주문-A2
user-C:주문-C1
user-B:주문-B2
user-A:주문-A3
EOF
```

**설정 설명:**
- `parse.key=true`: Key:Value 형식으로 파싱
- `key.separator=:`: Key와 Value 구분자 지정

**예상 결과:**
- `user-A`의 모든 메시지 (주문-A1, A2, A3) → 같은 파티션
- `user-B`의 모든 메시지 (주문-B1, B2) → 같은 파티션
- `user-C`의 메시지 (주문-C1) → 또 다른 파티션

### 1-6. 실습 정리

```bash
# 토픽 삭제
docker exec broker /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --delete \
  --topic partition-demo
```

### 📝 파티션 분배 핵심 요약

| 조건 | 동작 | 사용 사례 |
|------|------|----------|
| Key 없음 (기본) | Sticky Partitioner (한 배치 → 한 파티션) | 순서 무관한 대량 처리 |
| Key 없음 + batch.size=1 | 라운드로빈에 가깝게 분산 | 테스트/학습용 |
| Key 있음 | 같은 Key → 같은 파티션 | 사용자별 순서 보장 |

---
## 2. Consumer Group 병렬 처리 실습

### 2-1. 실습용 토픽 생성

```bash
docker exec broker /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --create \
  --topic consumer-demo \
  --partitions 3
```

**예상 출력:**
```
Created topic consumer-demo.
```

### 2-2. 테스트 메시지 전송

```bash
docker exec -i broker /opt/kafka/bin/kafka-console-producer.sh \
  --bootstrap-server localhost:9092 \
  --topic consumer-demo << 'EOF'
msg-1
msg-2
msg-3
msg-4
msg-5
msg-6
msg-7
msg-8
msg-9
EOF
```

### 2-3. 단일 Consumer로 읽기

```bash
docker exec broker /opt/kafka/bin/kafka-console-consumer.sh \
  --bootstrap-server localhost:9092 \
  --topic consumer-demo \
  --group my-consumer-group \
  --from-beginning \
  --timeout-ms 5000 2>/dev/null || true
```

**예상 출력:**
```
msg-1
msg-2
msg-3
msg-4
msg-5
msg-6
msg-7
msg-8
msg-9
```

> ⚠️ **주의**: 메시지 순서는 **파티션 간에 섞일 수 있습니다**. 각 파티션 내에서만 순서가 보장됩니다.

### 2-4. Consumer Group 상태 확인

```bash
docker exec broker /opt/kafka/bin/kafka-consumer-groups.sh \
  --bootstrap-server localhost:9092 \
  --describe \
  --group my-consumer-group
```

**예상 출력:**
```
GROUP             TOPIC           PARTITION  CURRENT-OFFSET  LOG-END-OFFSET  LAG
my-consumer-group consumer-demo   0          0               0               0
my-consumer-group consumer-demo   1          9               9               0
my-consumer-group consumer-demo   2          0               0               0

Consumer group 'my-consumer-group' has no active members.
```

> 💡 **설명**: Sticky Partitioner로 인해 모든 메시지가 Partition 1에 저장됨

### 2-5. 새 메시지 전송 (파티션에 분산)

```bash
for i in {1..6}; do
  echo "new-$i" | docker exec -i broker /opt/kafka/bin/kafka-console-producer.sh \
    --bootstrap-server localhost:9092 \
    --topic consumer-demo \
    --producer-property batch.size=1
  sleep 0.1
done
```

### 2-6. LAG 확인

```bash
docker exec broker /opt/kafka/bin/kafka-consumer-groups.sh \
  --bootstrap-server localhost:9092 \
  --describe \
  --group my-consumer-group
```

**예상 출력:**
```
GROUP             TOPIC           PARTITION  CURRENT-OFFSET  LOG-END-OFFSET  LAG
my-consumer-group consumer-demo   0          0               2               2
my-consumer-group consumer-demo   1          9               9               0
my-consumer-group consumer-demo   2          0               4               4

Consumer group 'my-consumer-group' has no active members.
```

```
┌──────────────────────────────────────────────────────────────────────────┐
│                        LAG (Consumer Lag) 이해                            │
├──────────────────────────────────────────────────────────────────────────┤
│                                                                          │
│  LAG = LOG-END-OFFSET - CURRENT-OFFSET                                   │
│                                                                          │
│  • CURRENT-OFFSET: Consumer가 마지막으로 읽은 위치                         │
│  • LOG-END-OFFSET: 파티션에 저장된 마지막 메시지 위치                       │
│  • LAG: 아직 처리되지 않은 메시지 수                                       │
│                                                                          │
│  예시:                                                                    │
│    Partition 0: CURRENT=0, LOG-END=2 → LAG=2 (2개 메시지 밀림)            │
│    Partition 1: CURRENT=9, LOG-END=9 → LAG=0 (모두 처리됨)                │
│    Partition 2: CURRENT=0, LOG-END=4 → LAG=4 (4개 메시지 밀림)            │
│                                                                          │
└──────────────────────────────────────────────────────────────────────────┘
```

### 2-7. 같은 그룹으로 새 메시지만 읽기

```bash
docker exec broker /opt/kafka/bin/kafka-console-consumer.sh \
  --bootstrap-server localhost:9092 \
  --topic consumer-demo \
  --group my-consumer-group \
  --timeout-ms 5000 2>/dev/null || true
```

**예상 출력:**
```
new-1
new-2
new-3
new-4
new-5
new-6
```

> 💡 **핵심**: 같은 Consumer Group은 마지막으로 읽은 Offset부터 이어서 읽습니다!
> (순서는 파티션 간에 섞일 수 있습니다)

### 2-8. Consumer 수 > Partition 수 상황

```
┌──────────────────────────────────────────────────────────────────────────┐
│  ⚠️ Consumer 수 > Partition 수                                           │
│                                                                          │
│  리밸런싱 발생 후:                                                        │
│  ┌──────────────────────────────────────────────────────────────────┐   │
│  │  Partition 0 ──────▶ Consumer 1                                  │   │
│  │  Partition 1 ──────▶ Consumer 2                                  │   │
│  │  Partition 2 ──────▶ Consumer 3                                  │   │
│  │                                                                  │   │
│  │  Consumer 4 ──── (파티션 미할당, 활성 멤버이지만 작업 없음)         │   │
│  └──────────────────────────────────────────────────────────────────┘   │
│                                                                          │
│  💡 Consumer 4는 그룹에 참여하지만 할당된 파티션이 없음                     │
│  💡 다른 Consumer 장애 시 자동으로 파티션 할당받음 (예비 대기)              │
│                                                                          │
└──────────────────────────────────────────────────────────────────────────┘
```

### 2-9. 실습 정리

```bash
# 토픽 삭제
docker exec broker /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --delete \
  --topic consumer-demo

# Consumer Group 삭제
docker exec broker /opt/kafka/bin/kafka-consumer-groups.sh \
  --bootstrap-server localhost:9092 \
  --delete \
  --group my-consumer-group
```

---
## 3. Consumer Group과 Offset 실습

### 3-1. 실습용 토픽 생성 (파티션 1개)

> 💡 **이유**: Offset 흐름을 명확히 관찰하기 위해 파티션 1개로 생성

```bash
docker exec broker /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --create \
  --topic offset-demo \
  --partitions 1
```

**예상 출력:**
```
Created topic offset-demo.
```

### 3-2. 테스트 메시지 10개 전송

```bash
docker exec -i broker /opt/kafka/bin/kafka-console-producer.sh \
  --bootstrap-server localhost:9092 \
  --topic offset-demo << 'EOF'
msg-00
msg-01
msg-02
msg-03
msg-04
msg-05
msg-06
msg-07
msg-08
msg-09
EOF
```

### 3-3. Group-A로 모든 메시지 읽기

```bash
docker exec broker /opt/kafka/bin/kafka-console-consumer.sh \
  --bootstrap-server localhost:9092 \
  --topic offset-demo \
  --group group-A \
  --from-beginning \
  --timeout-ms 5000 2>/dev/null || true
```

**예상 출력:**
```
msg-00
msg-01
msg-02
msg-03
msg-04
msg-05
msg-06
msg-07
msg-08
msg-09
```

### 3-4. Group-A로 다시 읽기 (아무것도 안 나옴)

```bash
docker exec broker /opt/kafka/bin/kafka-console-consumer.sh \
  --bootstrap-server localhost:9092 \
  --topic offset-demo \
  --group group-A \
  --timeout-ms 3000 2>/dev/null || true
```

**예상 출력:**
```
(아무것도 출력되지 않음)
```

> 💡 **이유**: Group-A는 이미 Offset 10까지 읽었으므로 새 메시지가 없음

### 3-5. Group-B로 읽기 (처음부터 다시)

```bash
docker exec broker /opt/kafka/bin/kafka-console-consumer.sh \
  --bootstrap-server localhost:9092 \
  --topic offset-demo \
  --group group-B \
  --from-beginning \
  --timeout-ms 5000 2>/dev/null || true
```

**예상 출력:**
```
msg-00
msg-01
msg-02
msg-03
msg-04
msg-05
msg-06
msg-07
msg-08
msg-09
```

> 💡 **핵심**: 다른 Consumer Group은 **독립적인 Offset**을 가집니다!

### 3-6. 새 메시지 5개 추가

```bash
docker exec -i broker /opt/kafka/bin/kafka-console-producer.sh \
  --bootstrap-server localhost:9092 \
  --topic offset-demo << 'EOF'
new-01
new-02
new-03
new-04
new-05
EOF
```

### 3-7. Group-A의 Offset 및 LAG 확인

```bash
docker exec broker /opt/kafka/bin/kafka-consumer-groups.sh \
  --bootstrap-server localhost:9092 \
  --describe \
  --group group-A
```

**예상 출력:**
```
GROUP           TOPIC           PARTITION  CURRENT-OFFSET  LOG-END-OFFSET  LAG
group-A         offset-demo     0          10              15              5

Consumer group 'group-A' has no active members.
```

| 항목 | 값 | 의미 |
|------|-----|------|
| **CURRENT-OFFSET** | 10 | 마지막으로 읽은 위치 |
| **LOG-END-OFFSET** | 15 | 파티션의 마지막 메시지 위치 |
| **LAG** | 5 | 처리 안 된 메시지 수 (15 - 10) |

### 3-8. Offset 리셋 (처음으로) - Dry Run

```bash
docker exec broker /opt/kafka/bin/kafka-consumer-groups.sh \
  --bootstrap-server localhost:9092 \
  --group group-A \
  --topic offset-demo \
  --reset-offsets \
  --to-earliest \
  --dry-run
```

**예상 출력:**
```
GROUP                          TOPIC                          PARTITION  NEW-OFFSET
group-A                        offset-demo                    0          0
```

> 💡 **--dry-run**: 실제 실행하지 않고 미리보기만

### 3-9. Offset 리셋 실행

```bash
docker exec broker /opt/kafka/bin/kafka-consumer-groups.sh \
  --bootstrap-server localhost:9092 \
  --group group-A \
  --topic offset-demo \
  --reset-offsets \
  --to-earliest \
  --execute
```

**예상 출력:**
```
GROUP                          TOPIC                          PARTITION  NEW-OFFSET
group-A                        offset-demo                    0          0
```

> ⚠️ **주의**: Consumer가 실행 중일 때는 리셋이 실패합니다. 모든 Consumer를 종료 후 실행하세요!

### 3-10. 리셋 후 다시 읽기 (모든 메시지 15개)

```bash
docker exec broker /opt/kafka/bin/kafka-console-consumer.sh \
  --bootstrap-server localhost:9092 \
  --topic offset-demo \
  --group group-A \
  --timeout-ms 5000 2>/dev/null || true
```

**예상 출력:**
```
msg-00
msg-01
msg-02
msg-03
msg-04
msg-05
msg-06
msg-07
msg-08
msg-09
new-01
new-02
new-03
new-04
new-05
```

> 💡 **결과**: Offset을 0으로 리셋했으므로 처음부터 15개 메시지 모두 다시 읽음!

### 3-11. 다양한 Offset 리셋 옵션

#### 최신으로 이동 (과거 메시지 무시)
```bash
docker exec broker /opt/kafka/bin/kafka-consumer-groups.sh \
  --bootstrap-server localhost:9092 \
  --group group-A \
  --topic offset-demo \
  --reset-offsets \
  --to-latest \
  --execute
```

#### 특정 Offset으로 이동
```bash
docker exec broker /opt/kafka/bin/kafka-consumer-groups.sh \
  --bootstrap-server localhost:9092 \
  --group group-A \
  --topic offset-demo \
  --reset-offsets \
  --to-offset 5 \
  --execute
```

#### 현재 위치에서 뒤로 3칸
```bash
docker exec broker /opt/kafka/bin/kafka-consumer-groups.sh \
  --bootstrap-server localhost:9092 \
  --group group-A \
  --topic offset-demo \
  --reset-offsets \
  --shift-by -3 \
  --execute
```

### 3-12. 실습 정리

```bash
# 토픽 삭제
docker exec broker /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --delete \
  --topic offset-demo

# Consumer Group 삭제
docker exec broker /opt/kafka/bin/kafka-consumer-groups.sh \
  --bootstrap-server localhost:9092 \
  --delete \
  --group group-A

docker exec broker /opt/kafka/bin/kafka-consumer-groups.sh \
  --bootstrap-server localhost:9092 \
  --delete \
  --group group-B
```

---
## 4. 멀티 브로커 클러스터 실습

### 4-1. 기존 환경 정리

```bash
cd kafka-ui-demo
docker compose down -v
```

### 4-2. 클러스터용 폴더 및 compose.yml 생성

```bash
mkdir kafka-cluster-demo
cd kafka-cluster-demo
```

**compose.yml:**

```yaml
# compose.yml - Kafka 3-Broker Cluster
services:
  broker-1:
    image: apache/kafka:latest
    container_name: broker-1
    ports:
      - "9092:9092"
    environment:
      KAFKA_NODE_ID: 1
      KAFKA_PROCESS_ROLES: broker,controller
      KAFKA_LISTENERS: PLAINTEXT://0.0.0.0:9092,CONTROLLER://0.0.0.0:9093
      KAFKA_ADVERTISED_LISTENERS: PLAINTEXT://broker-1:9092
      KAFKA_CONTROLLER_LISTENER_NAMES: CONTROLLER
      KAFKA_LISTENER_SECURITY_PROTOCOL_MAP: CONTROLLER:PLAINTEXT,PLAINTEXT:PLAINTEXT
      KAFKA_CONTROLLER_QUORUM_VOTERS: 1@broker-1:9093,2@broker-2:9093,3@broker-3:9093
      KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR: 3
      KAFKA_TRANSACTION_STATE_LOG_REPLICATION_FACTOR: 3
      KAFKA_TRANSACTION_STATE_LOG_MIN_ISR: 2
      KAFKA_DEFAULT_REPLICATION_FACTOR: 3
      KAFKA_MIN_INSYNC_REPLICAS: 2
      KAFKA_GROUP_INITIAL_REBALANCE_DELAY_MS: 0
      CLUSTER_ID: MkU3OEVBNTcwNTJENDM2Qk

  broker-2:
    image: apache/kafka:latest
    container_name: broker-2
    ports:
      - "9093:9092"
    environment:
      KAFKA_NODE_ID: 2
      KAFKA_PROCESS_ROLES: broker,controller
      KAFKA_LISTENERS: PLAINTEXT://0.0.0.0:9092,CONTROLLER://0.0.0.0:9093
      KAFKA_ADVERTISED_LISTENERS: PLAINTEXT://broker-2:9092
      KAFKA_CONTROLLER_LISTENER_NAMES: CONTROLLER
      KAFKA_LISTENER_SECURITY_PROTOCOL_MAP: CONTROLLER:PLAINTEXT,PLAINTEXT:PLAINTEXT
      KAFKA_CONTROLLER_QUORUM_VOTERS: 1@broker-1:9093,2@broker-2:9093,3@broker-3:9093
      KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR: 3
      KAFKA_TRANSACTION_STATE_LOG_REPLICATION_FACTOR: 3
      KAFKA_TRANSACTION_STATE_LOG_MIN_ISR: 2
      KAFKA_DEFAULT_REPLICATION_FACTOR: 3
      KAFKA_MIN_INSYNC_REPLICAS: 2
      KAFKA_GROUP_INITIAL_REBALANCE_DELAY_MS: 0
      CLUSTER_ID: MkU3OEVBNTcwNTJENDM2Qk

  broker-3:
    image: apache/kafka:latest
    container_name: broker-3
    ports:
      - "9094:9092"
    environment:
      KAFKA_NODE_ID: 3
      KAFKA_PROCESS_ROLES: broker,controller
      KAFKA_LISTENERS: PLAINTEXT://0.0.0.0:9092,CONTROLLER://0.0.0.0:9093
      KAFKA_ADVERTISED_LISTENERS: PLAINTEXT://broker-3:9092
      KAFKA_CONTROLLER_LISTENER_NAMES: CONTROLLER
      KAFKA_LISTENER_SECURITY_PROTOCOL_MAP: CONTROLLER:PLAINTEXT,PLAINTEXT:PLAINTEXT
      KAFKA_CONTROLLER_QUORUM_VOTERS: 1@broker-1:9093,2@broker-2:9093,3@broker-3:9093
      KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR: 3
      KAFKA_TRANSACTION_STATE_LOG_REPLICATION_FACTOR: 3
      KAFKA_TRANSACTION_STATE_LOG_MIN_ISR: 2
      KAFKA_DEFAULT_REPLICATION_FACTOR: 3
      KAFKA_MIN_INSYNC_REPLICAS: 2
      KAFKA_GROUP_INITIAL_REBALANCE_DELAY_MS: 0
      CLUSTER_ID: MkU3OEVBNTcwNTJENDM2Qk

  kafka-ui:
    image: provectuslabs/kafka-ui:latest
    container_name: kafka-ui
    depends_on:
      - broker-1
      - broker-2
      - broker-3
    ports:
      - "8080:8080"
    environment:
      KAFKA_CLUSTERS_0_NAME: local-cluster
      KAFKA_CLUSTERS_0_BOOTSTRAPSERVERS: broker-1:9092,broker-2:9092,broker-3:9092
```

### 4-3. 클러스터 실행 및 상태 확인

```bash
docker compose up -d
```

**예상 출력:**
```
[+] Running 4/4
 ✔ Container broker-1  Started
 ✔ Container broker-2  Started
 ✔ Container broker-3  Started
 ✔ Container kafka-ui  Started
```

```bash
docker ps
```

**예상 출력:**
```
CONTAINER ID   IMAGE                           STATUS         PORTS                    NAMES
xxxxxxxxxxxx   apache/kafka:latest            Up 30 seconds   0.0.0.0:9092->9092/tcp   broker-1
xxxxxxxxxxxx   apache/kafka:latest            Up 30 seconds   0.0.0.0:9093->9092/tcp   broker-2
xxxxxxxxxxxx   apache/kafka:latest            Up 30 seconds   0.0.0.0:9094->9092/tcp   broker-3
xxxxxxxxxxxx   provectuslabs/kafka-ui:latest  Up 25 seconds   0.0.0.0:8080->8080/tcp   kafka-ui
```

### 4-4. Replication Factor 3 토픽 생성

```bash
docker exec broker-1 /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --create \
  --topic replicated-topic \
  --partitions 3 \
  --replication-factor 3
```

**예상 출력:**
```
Created topic replicated-topic.
```

### 4-5. 토픽 상세 정보 확인

```bash
docker exec broker-1 /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --describe \
  --topic replicated-topic
```

**예상 출력:**
```
Topic: replicated-topic   TopicId: xxxxx   PartitionCount: 3   ReplicationFactor: 3
    Topic: replicated-topic   Partition: 0    Leader: 1   Replicas: 1,2,3   Isr: 1,2,3
    Topic: replicated-topic   Partition: 1    Leader: 2   Replicas: 2,3,1   Isr: 2,3,1
    Topic: replicated-topic   Partition: 2    Leader: 3   Replicas: 3,1,2   Isr: 3,1,2
```

| 항목 | 의미 |
|------|------|
| **Leader** | 읽기/쓰기를 담당하는 브로커 |
| **Replicas** | 복제본을 가진 브로커 목록 |
| **Isr** | 동기화된 복제본 (In-Sync Replicas) |

### 4-6. 메시지 전송 및 읽기

```bash
# broker-1에서 메시지 전송
docker exec -i broker-1 /opt/kafka/bin/kafka-console-producer.sh \
  --bootstrap-server localhost:9092 \
  --topic replicated-topic << 'EOF'
message-1
message-2
message-3
message-4
message-5
EOF

# broker-2에서 메시지 읽기 (복제 확인)
docker exec broker-2 /opt/kafka/bin/kafka-console-consumer.sh \
  --bootstrap-server localhost:9092 \
  --topic replicated-topic \
  --from-beginning \
  --timeout-ms 5000 2>/dev/null || true
```

**예상 출력:**
```
message-1
message-2
message-3
message-4
message-5
```

> 💡 **핵심**: broker-1에서 전송한 메시지를 broker-2에서 읽을 수 있음! (복제 완료)

### 4-7. 브로커 장애 시뮬레이션

```bash
# Partition 0의 Leader (broker-1) 종료
docker stop broker-1
```

**예상 출력:**
```
broker-1
```

### 4-8. Leader 재선출 확인

```bash
sleep 5
docker exec broker-2 /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --describe \
  --topic replicated-topic
```

**예상 출력:**
```
Topic: replicated-topic   PartitionCount: 3   ReplicationFactor: 3
    Topic: replicated-topic   Partition: 0    Leader: 2   Replicas: 1,2,3   Isr: 2,3
    Topic: replicated-topic   Partition: 1    Leader: 2   Replicas: 2,3,1   Isr: 2,3
    Topic: replicated-topic   Partition: 2    Leader: 3   Replicas: 3,1,2   Isr: 3,2
```

```
┌──────────────────────────────────────────────────────────────────────────┐
│                       Leader 재선출 과정                                   │
├──────────────────────────────────────────────────────────────────────────┤
│                                                                          │
│  Before (broker-1 정상):                                                  │
│    Partition 0: Leader=1, Isr=[1,2,3]                                    │
│                                                                          │
│  After (broker-1 다운):                                                   │
│    Partition 0: Leader=2, Isr=[2,3]   ← 자동 재선출!                      │
│                                                                          │
│  💡 핵심 포인트:                                                          │
│    • Leader가 다운되면 Isr 중에서 새 Leader 선출                          │
│    • 다운된 브로커는 Isr에서 제외                                         │
│    • 서비스 중단 없이 계속 동작!                                          │
│                                                                          │
└──────────────────────────────────────────────────────────────────────────┘
```

### 4-9. 장애 상태에서 메시지 전송 및 읽기

```bash
# broker-1이 다운된 상태에서 메시지 전송
docker exec -i broker-2 /opt/kafka/bin/kafka-console-producer.sh \
  --bootstrap-server localhost:9092 \
  --topic replicated-topic << 'EOF'
after-failure-1
after-failure-2
after-failure-3
EOF

# broker-3에서 메시지 읽기
docker exec broker-3 /opt/kafka/bin/kafka-console-consumer.sh \
  --bootstrap-server localhost:9092 \
  --topic replicated-topic \
  --from-beginning \
  --timeout-ms 5000 2>/dev/null || true
```

**예상 출력:**
```
message-1
message-2
message-3
message-4
message-5
after-failure-1
after-failure-2
after-failure-3
```

> 💡 **중요**: broker-1이 다운되었지만 서비스는 정상 동작!

### 4-10. 브로커 복구 및 동기화 확인

```bash
# broker-1 복구
docker start broker-1

# 동기화 대기
sleep 10

# 상태 확인
docker exec broker-1 /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --describe \
  --topic replicated-topic
```

**예상 출력:**
```
Topic: replicated-topic   PartitionCount: 3   ReplicationFactor: 3
    Topic: replicated-topic   Partition: 0    Leader: 2   Replicas: 1,2,3   Isr: 2,3,1
    Topic: replicated-topic   Partition: 1    Leader: 2   Replicas: 2,3,1   Isr: 2,3,1
    Topic: replicated-topic   Partition: 2    Leader: 3   Replicas: 3,1,2   Isr: 3,2,1
```

| 항목 | 상태 |
|------|------|
| **Isr** | broker-1이 다시 포함됨 ✅ |
| **Leader** | 현재 Leader 유지 (자동 복귀 안 함) |
| **데이터** | 다운 중 전송된 메시지도 자동 동기화 |

### 4-11. 클러스터 종료

```bash
docker compose down -v
```

---
## 📚 자주 사용하는 명령어 정리

### 토픽 관리
```bash
# 토픽 생성
docker exec broker /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --create --topic <토픽명> --partitions <개수>

# 토픽 목록
docker exec broker /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --list

# 토픽 상세 정보
docker exec broker /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --describe --topic <토픽명>

# 토픽 삭제
docker exec broker /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server localhost:9092 \
  --delete --topic <토픽명>
```

### Producer/Consumer
```bash
# Producer (메시지 전송)
docker exec -i broker /opt/kafka/bin/kafka-console-producer.sh \
  --bootstrap-server localhost:9092 \
  --topic <토픽명>

# Consumer (메시지 읽기, 자동 종료)
docker exec broker /opt/kafka/bin/kafka-console-consumer.sh \
  --bootstrap-server localhost:9092 \
  --topic <토픽명> \
  --group <그룹명> \
  --from-beginning \
  --timeout-ms 5000 2>/dev/null || true
```

### Consumer Group 관리
```bash
# Consumer Group 목록
docker exec broker /opt/kafka/bin/kafka-consumer-groups.sh \
  --bootstrap-server localhost:9092 \
  --list

# Consumer Group 상세 정보 (LAG 확인)
docker exec broker /opt/kafka/bin/kafka-consumer-groups.sh \
  --bootstrap-server localhost:9092 \
  --describe --group <그룹명>

# Offset 리셋
docker exec broker /opt/kafka/bin/kafka-consumer-groups.sh \
  --bootstrap-server localhost:9092 \
  --group <그룹명> \
  --topic <토픽명> \
  --reset-offsets --to-earliest \
  --execute

# Consumer Group 삭제
docker exec broker /opt/kafka/bin/kafka-consumer-groups.sh \
  --bootstrap-server localhost:9092 \
  --delete --group <그룹명>
```

---
## 🛠️ 문제 해결 팁

### 1. Consumer가 종료되지 않을 때
```bash
# 방법 1: --timeout-ms 옵션 사용 (권장)
--timeout-ms 5000 2>/dev/null || true

# 방법 2: -it 옵션으로 실행 후 Ctrl+C
docker exec -it broker /opt/kafka/bin/kafka-console-consumer.sh ...
```

### 2. 메시지가 한 파티션에 몰릴 때
```bash
# 배치 크기를 작게 설정하고 천천히 전송
for i in {1..10}; do
  echo "msg-$i" | docker exec -i broker /opt/kafka/bin/kafka-console-producer.sh \
    --bootstrap-server localhost:9092 \
    --topic <토픽명> \
    --producer-property batch.size=1
  sleep 0.1
done
```

### 3. Offset 리셋이 실패할 때
- 모든 Consumer를 종료한 후 실행
- Consumer가 연결된 상태에서는 리셋 불가

### 4. LAG 모니터링 명령어 오류
- ❌ `kafka.tools.GetOffsetShell` (최신 Kafka에서 없음)
- ✅ `kafka-consumer-groups.sh --describe` (권장)

---
## 📝 퀴즈

### Q1. Sticky Partitioner의 동작으로 올바른 것은?

- A) 모든 메시지를 순차적으로 라운드로빈 분배한다
- B) 같은 배치의 메시지를 같은 파티션으로 전송한다
- C) Key가 있는 메시지만 파티션을 고정한다
- D) 가장 비어있는 파티션으로 메시지를 전송한다

<details>
<summary>정답 확인</summary>

**정답: B**

Sticky Partitioner는 Kafka 2.4+에서 도입된 기능으로, Key가 없는 메시지의 경우 같은 배치의 메시지를 같은 파티션으로 전송합니다. 이를 통해 네트워크 효율을 높이고 배치 처리 성능을 향상시킵니다.
</details>

---

### Q2. Consumer Group에서 LAG의 의미는?

- A) Consumer의 처리 속도
- B) 아직 처리되지 않은 메시지 수
- C) 파티션의 총 메시지 수
- D) Consumer의 대기 시간

<details>
<summary>정답 확인</summary>

**정답: B**

LAG = LOG-END-OFFSET - CURRENT-OFFSET 로 계산되며, Consumer가 아직 처리하지 못한 메시지의 수를 나타냅니다. LAG이 계속 증가하면 Consumer의 처리 속도가 메시지 유입 속도를 따라가지 못하는 것입니다.
</details>

---

### Q3. Offset 리셋 시 주의사항으로 올바른 것은?

- A) Consumer가 실행 중일 때만 리셋 가능
- B) 모든 Consumer를 종료한 후 리셋해야 함
- C) 리셋 후 자동으로 Consumer가 재시작됨
- D) 리셋은 한 번만 실행 가능

<details>
<summary>정답 확인</summary>

**정답: B**

Offset 리셋은 해당 Consumer Group의 모든 Consumer가 종료된 상태에서만 가능합니다. 활성 Consumer가 있으면 리셋이 실패합니다. 이는 실행 중인 Consumer와의 충돌을 방지하기 위함입니다.
</details>

---

### Q4. 멀티 브로커 클러스터에서 Leader가 다운되면?

- A) 모든 서비스가 중단됨
- B) 수동으로 새 Leader를 지정해야 함
- C) ISR 중에서 새 Leader가 자동 선출됨
- D) 데이터가 모두 유실됨

<details>
<summary>정답 확인</summary>

**정답: C**

Leader 브로커가 다운되면 ISR(In-Sync Replicas) 중에서 자동으로 새로운 Leader가 선출됩니다. 이 과정은 자동으로 진행되며, 서비스 중단 없이 계속 동작합니다. 다운된 브로커는 ISR에서 제외됩니다.
</details>

---
## ✏️ 과제

### 과제 1: 파티션 분배 테스트 (난이도: ⭐)

1. 파티션 3개짜리 토픽 생성
2. Key 없이 10개 메시지 전송 (기본 설정)
3. 각 파티션별 메시지 확인
4. batch.size=1 설정으로 10개 메시지 재전송
5. 파티션 분배 차이 비교

---

### 과제 2: Consumer Group 실습 (난이도: ⭐⭐)

1. 토픽 생성 후 메시지 10개 전송
2. Group-A로 메시지 읽기
3. LAG 확인
4. 새 메시지 5개 추가
5. LAG 변화 확인
6. Offset 리셋 후 전체 메시지 다시 읽기

---

### 과제 3: 장애 복구 테스트 (난이도: ⭐⭐⭐)

1. 3-브로커 클러스터 실행
2. Replication Factor 3 토픽 생성
3. 메시지 전송 및 복제 확인
4. Leader 브로커 종료
5. Leader 재선출 확인
6. 장애 상태에서 메시지 전송/읽기 테스트
7. 브로커 복구 후 동기화 확인

---
## 🎯 핵심 요약

### 1. Sticky Partitioner
- Kafka 2.4+에서 도입된 성능 최적화 기능
- 같은 배치의 메시지 → 같은 파티션
- 분산이 필요하면 `batch.size=1` 설정

### 2. Consumer Group
- 같은 `group.id` → 파티션 분담 처리
- 다른 `group.id` → 각각 독립적으로 모든 메시지 처리
- Consumer 수 > Partition 수 → 일부 Consumer는 대기 상태

### 3. Offset 관리
- CURRENT-OFFSET: 마지막으로 읽은 위치
- LOG-END-OFFSET: 파티션의 마지막 메시지 위치
- LAG = LOG-END-OFFSET - CURRENT-OFFSET
- 리셋 시 Consumer 종료 필수

### 4. 멀티 브로커 클러스터
- Replication Factor: 데이터 복제본 수
- Leader: 읽기/쓰기 담당
- ISR: 동기화된 복제본 목록
- 자동 Leader 재선출로 고가용성 보장

### 5. CLI 팁
- Consumer 종료: `--timeout-ms 5000 2>/dev/null || true`
- LAG 확인: `kafka-consumer-groups.sh --describe`
- 파티션 분산: `--producer-property batch.size=1`

---


# Day 8 - 1교시: Kafka란 무엇인가 + Python 환경 설정

---

## 🎯 수업 목표

이 교시를 마치면 다음을 할 수 있습니다:

- ✅ Apache Kafka가 무엇인지, 왜 필요한지 명확히 이해한다
- ✅ 데이터 통합의 문제점과 Kafka의 해결책을 설명할 수 있다
- ✅ 실제 기업들이 Kafka를 어떻게 활용하는지 안다
- ✅ 데이터 엔지니어에게 Kafka가 왜 필수 스킬인지 이해한다
- ✅ Day06-07의 CLI 명령어를 Python 코드로 연결할 수 있다
- ✅ venv와 uv의 차이를 이해하고 적절히 선택할 수 있다
- ✅ Docker 기반 Python Kafka 환경을 구축할 수 있다
- ✅ 첫 Producer/Consumer를 Python으로 작성하고 실행할 수 있다

---

## 📋 Part 1: What is Apache Kafka? (20분)

> 📖 **공식 문서**: [Apache Kafka Documentation](https://kafka.apache.org/documentation/)
> 📖 **참고 자료**: [Conduktor - What is Kafka](https://www.conduktor.io/kafka/what-is-apache-kafka/)

### 1.1 Data Integration Challenges (데이터 통합의 문제)

![](https://substackcdn.com/image/fetch/$s_!-az8!,w_1456,c_limit,f_webp,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2F44381f1d-2081-42cf-b9b7-b1200d42284d_1999x750.png)

**현대 조직의 데이터 환경**:

회사에는 다양한 시스템들이 있습니다:

- **CRM** (Customer Relationship Management): 고객 정보 관리
- **Billing System**: 결제/청구 시스템
- **Accounting**: 회계 시스템
- **Website**: 웹사이트/모바일 앱
- **Analytics**: 분석 시스템
- **Email System**: 이메일 발송

이 시스템들은 서로 데이터를 주고받아야 합니다.
예를 들어, 웹사이트에서 결제가 일어나면 → Billing에 알려주고 → 회계 시스템에도 기록해야 합니다.

---

**직접 통합(Direct Integration)의 문제점 - 스파게티 코드**:

![](https://mintcdn.com/conduktor/uDLwmpxGlqlGuZu3/learn/images/What_is_Apache_Kafka_Part_1_-_Data_Integration_Challenges.png?w=1650&fit=max&auto=format&n=uDLwmpxGlqlGuZu3&q=85&s=d55069807e238b0dfb78d952fb48596d)

처음엔 단순하게 **"필요할 때마다 직접 연결"**하는 방식을 씁니다:

```
Website ←→ Billing
Website ←→ CRM
Billing ←→ Accounting
CRM ←→ Accounting
...
```

스파게티처럼 엉키는 문제들:

**1. 연결 수가 폭발적으로 증가**

| 시스템 개수 | 필요한 연결 수 (N × (N-1)) |
|------------|--------------------------|
| 4개 | 4 × 3 = **12개** 연결 |
| 6개 | 6 × 5 = **30개** 연결 |
| 10개 | 10 × 9 = **90개** 연결 |
| 100개 | 100 × 99 = **9,900개** 연결! |

**2. 대화 방식(프로토콜)이 제각각**

- 어떤 시스템은 **HTTP**로 말하고
- 어떤 시스템은 **JDBC**로 말하고
- 어떤 시스템은 **FTP**로 말합니다
- TCP, REST API, ODBC...

**3. 데이터 형식도 제각각**

- **JSON**으로 보내는 곳
- **CSV**로 보내는 곳
- **XML**로 보내는 곳
- Binary, Avro, Parquet...

**4. 스키마 변경 관리의 어려움**

- 한 시스템이 변경되면 연결된 **모든 시스템 수정 필요**
- 유지보수 비용 폭증

새 시스템 하나 추가할 때마다 기존 모든 시스템과 맞춰야 해서 **악몽**이 됩니다.

```
직접 통합의 악몽 (스파게티 아키텍처):

[CRM] ←→ [Billing]
  ↕         ↕
[Website] ←→ [Analytics]
  ↕         ↕
[Email] ←→ [Accounting]

6개 시스템 = 30개 연결!
유지보수 불가능한 스파게티 코드
```

---

### 1.2 Kafka를 통한 Decoupling (분리)

**비유: 마을의 중앙 우체국**

Kafka가 없을 때는 마을 사람들이 서로에게 **직접 편지를 전달**해야 했습니다.
100명이 있으면 각자 99명의 집 위치를 알아야 하고, 직접 찾아가야 했죠.

Kafka가 생기면? **중앙 우체국**이 생긴 겁니다!

```
[보내는 사람들]     →    📮 Kafka    →    [받는 사람들]
 (Producers)           (우체국)          (Consumers)
```

- **보내는 사람**: 우체국에 편지만 넣으면 끝
- **받는 사람**: 우체국에서 자기 편지만 가져가면 끝

서로 상대방의 집 위치를 몰라도 됩니다!

---

**Kafka를 중앙 데이터 허브로 활용**:

- 모든 시스템이 Kafka를 통해서만 통신
- Source → **Kafka** → Target 구조

![](https://mintcdn.com/conduktor/uDLwmpxGlqlGuZu3/learn/images/What_is_Apache_Kafka_Part_1_-_Decoupling_Different_Data_Systems.png?w=1650&fit=max&auto=format&n=uDLwmpxGlqlGuZu3&q=85&s=6a31c40d792a129d5fc5858f204ae79a)

```
Kafka 중심 아키텍처:

   [CRM]    [Billing]   [Website]
     ↓         ↓           ↓
     ↓         ↓           ↓
 ┌─────────────────────────────┐
 │         📮 Kafka            │
 │       (중앙 우체국)          │
 └─────────────────────────────┘
     ↓         ↓           ↓
     ↓         ↓           ↓
[Analytics] [Email]  [Accounting]
```

---

**복잡도가 확 줄어드는 마법**:

| 방식 | 시스템 4개 | 시스템 10개 | 시스템 100개 |
|------|-----------|------------|-------------|
| **직접 연결** (N×(N-1)) | 12개 | 90개 | **9,900개** |
| **Kafka 사용** (2×N) | 8개 | 20개 | **200개** |

**공식으로 보면**:
- 직접 연결: **N × (N-1)** → 시스템이 늘수록 기하급수적 증가
- Kafka: **2 × N** (보내기 N개 + 받기 N개) → 선형 증가

10개 시스템 기준으로 **90개 → 20개**로 감소! (78% 감소)

---

**Kafka 도입의 장점**:

- ✅ 시스템 간 **독립성** 확보 (서로 몰라도 됨)
- ✅ 새 시스템 추가가 **쉬움** (Kafka에만 연결하면 끝)
- ✅ 한 시스템 장애가 **다른 시스템에 영향 없음**
- ✅ 표준화된 **인터페이스** (모두 Kafka 프로토콜 사용)
- ✅ **메시지 보관** (우체국이 편지를 보관해둠)

### 1.3 Data Streaming이란?

![](https://mintcdn.com/conduktor/uDLwmpxGlqlGuZu3/learn/images/What_is_Apache_Kafka_Part_1_-_Use_Cases_and_Applications.png?w=1650&fit=max&auto=format&n=uDLwmpxGlqlGuZu3&q=85&s=6c78877ef60bb24f377828ddad980239)

**정의**:
- 잠재적으로 **무한한** 데이터의 연속적인 흐름
- 데이터가 **생성되는 즉시** 사용 가능
- **실시간 처리**가 핵심

**Batch vs Streaming**:

| 구분 | Batch 처리 | Streaming 처리 |
|------|-----------|---------------|
| **시간** | 일/시간 단위 | 초/밀리초 단위 |
| **데이터** | 과거 데이터 | 실시간 데이터 |
| **처리** | 모아서 한번에 | 들어오는 즉시 |
| **예시** | 일일 매출 보고서 | 실시간 주가 변동 |

**예시**:
- **Batch**: 밤 12시에 하루 주문을 모아서 처리
- **Streaming**: 주문이 들어올 때마다 즉시 처리

---

**실제 Data Stream 예시**:

회사들이 실제로 처리하는 데이터 스트림의 예시입니다:

**1. 로그 분석 (Log Analysis)**

```
[마이크로서비스 1] ──┐
[마이크로서비스 2] ──┼──→ 📮 Kafka ──→ [로그 분석 시스템]
[마이크로서비스 3] ──┤                       ↓
       ...         ──┘               [대시보드/알림]
(수천 개의 서비스)
```

- 현대 애플리케이션은 **수십~수천 개의 마이크로서비스**로 구성
- 각 서비스가 **끊임없이 로그를 생성** (초당 수천~수백만 개)
- 이 로그에는 **비즈니스 인사이트**, **장애 예측**, **디버깅** 정보가 가득

**문제**: 이렇게 대량으로 생성되는 로그 데이터를 어떻게 한 곳에서 처리할까?

**해결**: 로그 데이터를 **Kafka(데이터 스트림)**에 푸시하여 **스트림 처리** 수행

```
예시 - 에러 로그 실시간 감지:

서비스A 로그: [INFO] 사용자 로그인 성공
서비스B 로그: [ERROR] 데이터베이스 연결 실패  ← 즉시 감지!
서비스C 로그: [INFO] 주문 처리 완료
서비스B 로그: [ERROR] 타임아웃 발생           ← 패턴 분석!

→ Kafka로 모든 로그 수집 → 실시간 분석 → "서비스B 장애 발생!" 알림
```

**2. 웹 분석 (Web Analytics)**

```
사용자 행동 데이터:

👤 사용자A: 페이지 방문 → 버튼 클릭 → 상품 조회 → 장바구니 추가
👤 사용자B: 페이지 방문 → 검색 → 상품 클릭 → 이탈
👤 사용자C: 로그인 → 구매 완료 → 리뷰 작성
          ...
      (초당 수천~수만 건)
```

- 현대 웹 애플리케이션은 **거의 모든 사용자 활동을 측정**
  - 버튼 클릭, 페이지 조회, 스크롤, 마우스 움직임...
- 이 이벤트들은 **빠르게 누적** (대형 서비스는 초당 수만 건)

**Batch 방식의 문제**:
- 데이터를 모아서 **몇 시간 후에** 분석
- "어제 이탈률이 높았네" → 이미 늦음!

**Streaming 방식의 장점**:
- 데이터가 **생성되는 즉시** 처리
- "지금 이탈률이 급증 중!" → 즉시 대응 가능

```
Streaming 처리 예시:

10:00:00 - 페이지뷰 100건/초 (정상)
10:00:30 - 페이지뷰 20건/초   ← 이상 감지!
10:00:31 - 알림: "트래픽 급감! 서버 문제 확인 필요"

→ 문제 발생 1초 만에 감지 (Batch였다면 다음날 알았을 것)
```

### 1.4 실제 사용 사례

**🚗 Uber (우버)**:
- **사용**: 실시간 요금 책정 파이프라인
- **규모**: 매일 수십억 건의 이벤트 처리
- **효과**:
  - 실시간으로 수요/공급 파악
  - 동적 요금제 (서지 프라이싱)
  - 운전자와 승객 실시간 매칭

**📺 Netflix (넷플릭스)**:
- **사용**: 사용자 행동 분석, 추천 시스템
- **규모**: 하루 **5000억 개** 이벤트, **1.3 PB** 데이터 처리
- **효과**:
  - 실시간 추천 업데이트
  - 시청 패턴 분석
  - A/B 테스트 즉시 반영

**📊 로그 분석**:
- **사용**: 수천 개 마이크로서비스의 로그 실시간 수집
- **효과**:
  - 장애 즉시 감지
  - 실시간 모니터링 대시보드
  - 보안 이슈 빠른 대응

**🌐 웹 분석**:
- **사용**: 사용자 행동 데이터 실시간 수집
- **효과**:
  - 실시간 방문자 통계
  - 클릭스트림 분석
  - 개인화 콘텐츠 제공

### 1.5 데이터 엔지니어에게 왜 중요한가?

**필수 스킬인 이유**:

1. **실시간 데이터 파이프라인 구축의 핵심**
   - Batch ETL → **Real-time Streaming ETL** 전환
   - 데이터 레이크/웨어하우스로 실시간 수집

2. **빅데이터 기술과의 통합**
   - **Apache Spark**: Kafka → Spark Streaming 파이프라인
   - **Apache Flink**: 복잡한 이벤트 처리
   - **Elasticsearch**: 로그 분석

3. **산업 표준**
   - **Fortune 100 기업의 80% 이상**이 사용
   - LinkedIn, Airbnb, Spotify, Twitter...

4. **채용 공고 필수 요구사항**
   - "Kafka 경험자 우대" 또는 "필수"
   - 데이터 엔지니어 채용의 90% 이상

5. **클라우드 서비스**
   - **AWS MSK** (Managed Streaming for Kafka)
   - **Confluent Cloud**
   - **Azure Event Hubs**

---

## 🔄 Part 2: Day06-07 복습 - CLI 명령어와 Python 개념 연결 (15분)

### 2.1 CLI 명령어 분해

**Day06-07에서 사용한 Producer CLI**:

```bash
kafka-console-producer \
  --bootstrap-server localhost:9092 \  # Broker 주소
  --topic orders                        # 토픽 이름
```

**Python 코드로 변환**:

```python
from confluent_kafka import Producer

# CLI의 --bootstrap-server
config = {
    'bootstrap.servers': 'broker:9092'
}

producer = Producer(config)

# CLI의 --topic orders
producer.produce(topic='orders', value=b'message')
producer.flush()  # CLI의 Enter 키 역할
```

**Day06-07에서 사용한 Consumer CLI**:

```bash
kafka-console-consumer \
  --bootstrap-server localhost:9092 \  # Broker 주소
  --topic orders \                      # 토픽 이름
  --from-beginning \                    # 처음부터 읽기
  --group my-group                      # Consumer Group
```

**Python 코드로 변환**:

```python
from confluent_kafka import Consumer

config = {
    'bootstrap.servers': 'broker:9092',         # --bootstrap-server
    'group.id': 'my-group',                     # --group
    'auto.offset.reset': 'earliest'             # --from-beginning
}

consumer = Consumer(config)
consumer.subscribe(['orders'])  # --topic orders

msg = consumer.poll(timeout=5.0)
if msg:
    print(msg.value().decode('utf-8'))
consumer.close()
```

### 2.2 핵심 개념 복습

**Day06-07에서 배운 핵심 개념**:

| 개념 | 설명 | 비유 |
|------|------|------|
| **Topic** | 메시지를 담는 카테고리 | 카카오톡 채팅방 |
| **Partition** | 병렬 처리 단위 | 채팅방 스레드 |
| **Producer** | 메시지 생산자 | 메시지 보내는 사람 |
| **Consumer** | 메시지 소비자 | 메시지 읽는 사람 |
| **Broker** | Kafka 서버 | 카카오톡 서버 |
| **Offset** | 파티션 내 메시지 위치 | 메시지 번호 |
| **Consumer Group** | 파티션 분담 처리 | 팀 단톡방 |

### 2.3 CLI vs Python 대응표

**완전 대응표**:

| CLI 옵션 | Python 설정 | 의미 | 예시 |
|---------|------------|------|------|
| `--bootstrap-server` | `bootstrap.servers` | Kafka 브로커 주소 | `'broker:9092'` |
| `--topic` | `topic` 파라미터 | 메시지 카테고리 | `'orders'` |
| `--group` | `group.id` | Consumer Group ID | `'my-group'` |
| `--from-beginning` | `auto.offset.reset='earliest'` | 처음부터 읽기 | - |
| Enter 입력 | `producer.flush()` | 전송 완료 대기 | - |

**핵심 포인트**:
- CLI 옵션의 `--`는 Python에서 `.`으로 변환
- 예: `--bootstrap-server` → `bootstrap.servers`

---

## 🐍 Part 3: Python 환경 설정 (uv + Docker) (30분)

### 3.1 uv 소개 및 설치

**uv란?**
- **초고속** Python 패키지 매니저
- **Rust 언어**로 작성 (pip보다 10-100배 빠름)
- 자동 가상환경 관리

> 📖 **공식 문서**: [uv Documentation](https://github.com/astral-sh/uv)

**왜 빠른가?**
- pip: 순수 Python 구현 → 느림
- uv: Rust 구현 + 병렬 다운로드 → 매우 빠름

**설치 방법 (WSL/Ubuntu)**:

```bash
# uv 설치
curl -LsSf https://astral.sh/uv/install.sh | sh

# 설치 확인
uv --version
# 출력: uv 0.x.x
```

**기본 사용법**:

```bash
# 패키지 설치 (가상환경 자동 생성)
uv pip install confluent-kafka

# 여러 패키지 한번에 설치
uv pip install confluent-kafka requests pandas

# requirements.txt에서 설치
uv pip install -r requirements.txt
```

### 3.2 venv vs uv 비교

**가상환경 도구 비교표**:

| 기능 | venv (기본) | uv (권장) |
|-----|-----------|---------|
| **설치 방법** | Python 내장 (별도 설치 불필요) | 별도 설치 필요 |
| **속도** | 느림 (순수 Python) | 매우 빠름 (Rust 기반) |
| **패키지 설치** | `pip install` 사용 | `uv pip install` 사용 |
| **설치 속도** | 기준 (1x) | 10-100배 빠름 |
| **가상환경 생성** | `python -m venv .venv` | 자동 생성 (`uv run` 사용 시) |
| **활성화** | `source .venv/bin/activate` | 자동 활성화 또는 `source .venv/bin/activate` |
| **종속성 해결** | 느림, 때때로 충돌 발생 | 빠르고 정확함 |
| **프로덕션 준비** | 적합 | 적합 (더 빠름) |
| **안정성** | 매우 안정적 | 안정적 (신기술) |

**언제 venv를 쓰고 언제 uv를 쓸까?**

**venv 사용 시나리오**:
- Python 기본 환경만 사용하고 싶을 때
- 추가 설치 없이 바로 사용해야 할 때
- 간단한 프로젝트 (패키지 10개 미만)
- 전통적인 Python 환경 선호

**uv 사용 시나리오** (추천):
- 패키지가 많은 프로젝트 (특히 데이터 과학)
- CI/CD 환경 (빠른 빌드가 중요)
- Docker 이미지 빌드 (빌드 시간 단축)
- 빠른 설치가 필요한 모든 경우

**실습에서는 uv 사용 이유**:
1. **confluent-kafka는 librdkafka(C 라이브러리) 의존성**
   - C 컴파일이 필요하여 설치 시간이 김
   - pip: 2-3분 소요
   - uv: 20-30초 소요 (10배 빠름)

2. **Docker 이미지 빌드 시간 단축**
   - 강의 중 빠른 환경 구축 가능
   - 학생들 대기 시간 최소화

3. **실무 트렌드**
   - 최신 Python 프로젝트에서 uv 도입 증가
   - 빠른 개발 사이클 구축

### 3.3 Docker 기반 Python 환경 구축

**왜 Docker를 사용하나?**
- ✅ 모든 학생이 **동일한 환경**
- ✅ 로컬 환경 **오염 방지**
- ✅ Kafka + Python을 **한번에 실행**
- ✅ 실무와 **동일한 환경**

**Dockerfile 작성** (`Dockerfile`):

```dockerfile
# Python 3.11 slim 이미지 사용 (경량)
FROM python:3.11-slim

# uv 설치 (멀티스테이지 빌드)
COPY --from=ghcr.io/astral-sh/uv:latest /uv /usr/local/bin/uv

# 작업 디렉토리 설정
WORKDIR /app

# confluent-kafka 설치에 필요한 시스템 패키지
RUN apt-get update && apt-get install -y \
    gcc \
    librdkafka-dev \
    && rm -rf /var/lib/apt/lists/*

# Python 패키지 설치 (uv 사용)
RUN uv pip install --system confluent-kafka

# 애플리케이션 코드 복사
COPY . .

# 기본 실행 명령
CMD ["python", "producer.py"]
```

**compose.yml 작성** (`compose.yml`):

```yaml
services:
  # Kafka Broker (KRaft 모드 - Zookeeper 불필요)
  broker:
    image: apache/kafka:latest
    ports:
      - "9092:9092"
    environment:
      KAFKA_NODE_ID: 1
      KAFKA_PROCESS_ROLES: broker,controller
      KAFKA_LISTENERS: PLAINTEXT://0.0.0.0:9092,CONTROLLER://0.0.0.0:9093
      KAFKA_ADVERTISED_LISTENERS: PLAINTEXT://broker:9092
      KAFKA_LISTENER_SECURITY_PROTOCOL_MAP: PLAINTEXT:PLAINTEXT,CONTROLLER:PLAINTEXT
      KAFKA_CONTROLLER_LISTENER_NAMES: CONTROLLER
      KAFKA_CONTROLLER_QUORUM_VOTERS: 1@broker:9093
      KAFKA_AUTO_CREATE_TOPICS_ENABLE: "true"
      # 단일 브로커 환경에서 Consumer Group 동작을 위한 필수 설정
      KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR: 1
      KAFKA_TRANSACTION_STATE_LOG_REPLICATION_FACTOR: 1
      KAFKA_TRANSACTION_STATE_LOG_MIN_ISR: 1

  # Kafka UI (웹 기반 관리 도구)
  kafka-ui:
    image: provectuslabs/kafka-ui:latest
    ports:
      - "8080:8080"
    environment:
      KAFKA_CLUSTERS_0_NAME: local
      KAFKA_CLUSTERS_0_BOOTSTRAPSERVERS: broker:9092

  # Python Producer 애플리케이션
  python-producer:
    build: .
    depends_on:
      - broker
    environment:
      KAFKA_BOOTSTRAP_SERVERS: broker:9092
```

**핵심 포인트**:
- **`broker:9092`**: Docker 네트워크 내에서는 **서비스 이름**으로 접근
- **`localhost:9092`**: 호스트에서 접근할 때 사용
- **Kafka UI**: `http://localhost:8080`에서 메시지 확인 가능

### 3.4 confluent-kafka vs kafka-python 비교

**Python Kafka 라이브러리 선택지**:

| 라이브러리 | 특징 | 성능 | 장점 | 단점 | 프로덕션 적합 |
|-----------|------|------|------|------|-------------|
| **kafka-python** | 순수 Python 구현 | 느림 (1x) | 설치 쉬움, 순수 Python | 느림, 기능 제한 | 낮음 |
| **confluent-kafka** | librdkafka (C) 기반 | 빠름 (10x+) | 매우 빠름, 풍부한 기능 | C 의존성 | 높음 ✅ |
| **aiokafka** | asyncio 기반 | 중간 (5x) | async/await 지원 | 비동기만 가능 | 중간 |

**confluent-kafka 선택 이유** (우리 강의):

1. **Confluent 공식 지원**
   - Confluent = Kafka 창시자들이 만든 회사
   - 공식 Python 클라이언트

2. **고성능**
   - C로 작성된 librdkafka 기반
   - 대용량 메시지 처리 가능

3. **프로덕션 검증**
   - LinkedIn, Uber, Netflix 등 사용
   - 실무에서 가장 많이 사용

4. **풍부한 기능**
   - 고급 Producer/Consumer 설정
   - Avro, Protobuf 직렬화 지원
   - Exactly-once 시맨틱

> 📖 **공식 문서**: [confluent-kafka-python](https://docs.confluent.io/kafka-clients/python/current/overview.html)

---

## 🚀 Part 4: Hello Kafka in Python (Docker 환경) (15분)

### 실습 준비

**1단계: 프로젝트 폴더 생성**

```bash
# WSL/Ubuntu 터미널에서
mkdir ~/kafka-python-hello
cd ~/kafka-python-hello
```

**2단계: 파일 생성**

다음 3개 파일을 생성합니다:
- `Dockerfile`
- `compose.yml`
- `producer.py`
- `consumer.py`

### 4.1 첫 Producer 작성

**`producer.py` 파일 내용**:

In [ ]:
# producer.py - 첫 번째 Kafka Producer
# 이 파일을 ~/kafka-python-hello/producer.py로 저장하세요

from confluent_kafka import Producer

# -----------------------------------------------------------------------------
# Producer 설정
# -----------------------------------------------------------------------------
# Kafka 브로커에 연결하기 위한 설정
config = {
    # Docker 네트워크 내에서는 서비스 이름(broker)으로 접근
    # 호스트에서는 localhost:9092로 접근
    "bootstrap.servers": "broker:9092",
    # Producer를 식별하는 ID (로그/모니터링에서 유용)
    "client.id": "hello-producer",
}

# -----------------------------------------------------------------------------
# Producer 인스턴스 생성
# -----------------------------------------------------------------------------
# config 딕셔너리를 전달하여 Producer 객체 생성
producer = Producer(config)

# -----------------------------------------------------------------------------
# 메시지 전송
# -----------------------------------------------------------------------------
# produce() 메서드로 메시지 전송
# - topic: 메시지를 보낼 토픽 이름
# - value: 메시지 내용 (bytes 타입이어야 함)
producer.produce(
    topic="hello-topic",
    value=b"Hello Kafka from Python!",  # b'...' = bytes 리터럴
)

# -----------------------------------------------------------------------------
# 전송 완료 대기
# -----------------------------------------------------------------------------
# flush(): 버퍼의 모든 메시지를 강제로 전송하고 완료될 때까지 대기
# 이 호출이 없으면 메시지가 전송되지 않을 수 있음!
producer.flush()

print("✅ 메시지 전송 완료")
print("   토픽: hello-topic")
print("   내용: Hello Kafka from Python!")

**예상 출력**:

```
✅ 메시지 전송 완료
   토픽: hello-topic
   내용: Hello Kafka from Python!
```

### 4.2 첫 Consumer 작성

**`consumer.py` 파일 내용**:

In [ ]:
# consumer.py - 첫 번째 Kafka Consumer
# 이 파일을 ~/kafka-python-hello/consumer.py로 저장하세요

from confluent_kafka import Consumer

# -----------------------------------------------------------------------------
# Consumer 설정
# -----------------------------------------------------------------------------
config = {
    # Kafka 브로커 주소
    "bootstrap.servers": "broker:9092",
    # Consumer Group ID
    # 같은 group.id를 가진 Consumer들은 파티션을 나눠서 처리
    "group.id": "hello-group",
    # 처음 시작할 때 어디서부터 읽을지 결정
    # 'earliest': 처음부터 읽기 (CLI의 --from-beginning)
    # 'latest': 최신 메시지부터 읽기
    "auto.offset.reset": "earliest",
}

# -----------------------------------------------------------------------------
# Consumer 인스턴스 생성
# -----------------------------------------------------------------------------
consumer = Consumer(config)

# -----------------------------------------------------------------------------
# 토픽 구독
# -----------------------------------------------------------------------------
# subscribe(): 읽고 싶은 토픽 목록을 리스트로 전달
# 여러 토픽 구독 가능: ['hello-topic', 'orders', 'logs']
consumer.subscribe(["hello-topic"])

print("📨 메시지 수신 대기 중...")
print("   토픽: hello-topic")
print("   Group: hello-group\n")

# -----------------------------------------------------------------------------
# 메시지 읽기
# -----------------------------------------------------------------------------
# poll(): 메시지를 하나 읽음
# timeout: 대기 시간 (초 단위)
msg = consumer.poll(timeout=5.0)

# 메시지 확인
if msg is None:
    # timeout 시간 내에 메시지가 없으면 None 반환
    print("⏰ 타임아웃: 메시지가 없습니다")
elif msg.error():
    # 에러 발생 시
    print(f"❌ 에러 발생: {msg.error()}")
else:
    # 메시지 정상 수신
    # msg.value(): bytes 타입
    # .decode('utf-8'): bytes → 문자열 변환
    message_text = msg.value().decode("utf-8")
    print(f"✅ 메시지 수신: {message_text}")
    print(f"   파티션: {msg.partition()}")
    print(f"   Offset: {msg.offset()}")

# -----------------------------------------------------------------------------
# Consumer 종료
# -----------------------------------------------------------------------------
# close(): Consumer를 정상 종료 (리소스 해제)
consumer.close()
print("\n✅ Consumer 종료 완료")

**예상 출력**:

```
📨 메시지 수신 대기 중...
   토픽: hello-topic
   Group: hello-group

✅ 메시지 수신: Hello Kafka from Python!
   파티션: 2
   Offset: 0

✅ Consumer 종료 완료
```

> **참고**: 파티션 번호는 토픽의 파티션 설정에 따라 달라질 수 있습니다.

### 4.3 실행 방법

**터미널에서 실행**:

```bash
# 1. Docker Compose로 전체 환경 실행
cd ~/kafka-python-hello
docker compose up -d

# 2. Kafka가 시작될 때까지 잠깐 대기 (10초)
sleep 10

# 3. Producer 실행 (메시지 전송)
docker compose run --rm python-producer python producer.py
# 출력: ✅ 메시지 전송 완료

# 4. Consumer 실행 (메시지 읽기)
docker compose run --rm python-producer python consumer.py
# 출력: ✅ 메시지 수신: Hello Kafka from Python!

# 5. Kafka UI에서 확인
# 브라우저에서 http://localhost:8080 접속
# - Topics 탭에서 hello-topic 선택
# - Messages 탭에서 메시지 확인

# 6. 환경 종료
docker compose down
```

**Kafka UI 확인 방법**:

1. 브라우저에서 `http://localhost:8080` 접속
2. 왼쪽 메뉴에서 **Topics** 클릭
3. `hello-topic` 클릭
4. **Messages** 탭에서 메시지 확인
   - Value: `Hello Kafka from Python!`
   - Partition: 0
   - Offset: 0

---

## ❓ FAQ

### Q1. uv와 pip의 차이가 뭔가요?

**A**: 기능은 거의 동일하지만 **속도**가 크게 다릅니다.

- **pip**: 순수 Python으로 작성 → 느림
- **uv**: Rust로 작성 + 병렬 다운로드 → 10-100배 빠름

**벤치마크 예시** (pandas + numpy + scikit-learn 설치):
- pip: 45초
- uv: 4.5초 (10배 빠름)

### Q2. Docker 없이 로컬에서 Kafka 설치해도 되나요?

**A**: 가능하지만 Docker 사용을 강력 권장합니다.

**로컬 설치의 문제점**:
- Java 설치 필요
- 복잡한 설정
- OS마다 설치 방법 다름
- 로컬 환경 오염

**Docker의 장점**:
- 환경 통일 (모든 학생이 동일한 환경)
- 깔끔한 설치/삭제
- 실무와 동일한 환경

### Q3. bootstrap.servers에서 'broker:9092'는 뭔가요?

**A**: Docker 네트워크 내에서 사용하는 **서비스 이름**입니다.

- **Docker 네트워크 내부**: `broker:9092` (서비스 이름)
- **호스트 (내 컴퓨터)**: `localhost:9092`

**이유**:
- Docker Compose는 자동으로 서비스 이름을 DNS 이름으로 등록
- `python-producer` 서비스에서 `broker` 서비스에 접근 가능

### Q4. Consumer의 auto.offset.reset이 뭔가요?

**A**: Consumer가 **처음 시작할 때** 어디서부터 읽을지 결정합니다.

- **`earliest`**: 토픽의 **처음부터** 읽기 (CLI의 `--from-beginning`)
- **`latest`**: **최신 메시지부터** 읽기 (기본값)

**예시**:
```python
# 토픽에 메시지 100개 있을 때

# earliest: 1번부터 읽음
config = {'auto.offset.reset': 'earliest'}

# latest: 101번부터 읽음 (기존 메시지 무시)
config = {'auto.offset.reset': 'latest'}
```

### Q5. flush()를 왜 호출해야 하나요?

**A**: Producer는 성능을 위해 메시지를 **버퍼에 모았다가** 전송합니다.

**flush() 없으면**:
- `produce()` 호출 후 **즉시 전송되지 않음**
- 버퍼에만 저장됨
- 프로그램이 바로 종료되면 **메시지 유실**!

**flush() 있으면**:
- 버퍼의 **모든 메시지를 강제 전송**
- 전송 완료까지 **대기**
- 안전하게 종료 가능

**비유**: 택배 발송
- `produce()`: 택배 상자에 물건 담기
- `flush()`: 택배 발송하기

---

## 📝 핵심 요약

### Kafka의 가치

| 문제 | Kafka 해결책 |
|------|-----------|
| N×(N-1) 통합 복잡도 | 2N으로 감소 |
| 시스템 간 강한 결합 | 느슨한 결합 (Decoupling) |
| 배치 처리의 지연 | 실시간 스트리밍 처리 |
| 장애 전파 | 시스템 독립성 확보 |

### venv vs uv

| 특징 | venv | uv |
|------|------|-----|
| **속도** | 1x | 10-100x |
| **설치** | Python 내장 | 별도 설치 |
| **사용 시나리오** | 간단한 프로젝트 | 복잡한 프로젝트, CI/CD |

### CLI → Python 대응

| CLI | Python |
|-----|--------|
| `--bootstrap-server localhost:9092` | `'bootstrap.servers': 'broker:9092'` |
| `--topic orders` | `topic='orders'` |
| `--from-beginning` | `'auto.offset.reset': 'earliest'` |
| `--group my-group` | `'group.id': 'my-group'` |
| Enter 입력 | `producer.flush()` |

### Hello World 코드 패턴

**Producer**:
```python
producer = Producer({'bootstrap.servers': 'broker:9092'})
producer.produce(topic='hello-topic', value=b'message')
producer.flush()
```

**Consumer**:
```python
consumer = Consumer({
    'bootstrap.servers': 'broker:9092',
    'group.id': 'my-group',
    'auto.offset.reset': 'earliest'
})
consumer.subscribe(['hello-topic'])
msg = consumer.poll(timeout=5.0)
if msg:
    print(msg.value().decode('utf-8'))
consumer.close()
```

---

## 🎯 다음 교시 예고

**2교시: Producer 깊이 있게 알기**
- Producer 설정 옵션 상세
- JSON 메시지 전송 (실무 패턴!)
- Key 지정으로 파티션 제어
- Delivery Callback으로 전송 확인

이제 Kafka의 필요성과 기본 환경을 이해했습니다.
다음 시간에는 Producer를 제대로 활용하는 방법을 배웁니다!

---


# Day 8 - 2교시: Producer 깊이 있게 알기

---

## 🎯 수업 목표

이 교시를 마치면 다음을 할 수 있습니다:

- ✅ Kafka CLI 명령을 Python Producer 코드로 전환할 수 있다
- ✅ confluent-kafka-python Producer의 주요 설정 옵션을 이해한다
- ✅ JSON 데이터를 직렬화하여 Kafka로 전송할 수 있다
- ✅ Key를 지정하여 특정 파티션으로 메시지를 보낼 수 있다
- ✅ Delivery Callback으로 전송 성공/실패를 확인할 수 있다
- ✅ flush()의 역할과 중요성을 이해한다

---

## 🔗 1교시 복습

### Day06-07 CLI 명령어 복습

```bash
# Producer CLI 명령어
kafka-console-producer \
  --bootstrap-server localhost:9092 \
  --topic my-topic
```

### Python으로 전환하면?

```python
from confluent_kafka import Producer

# Producer 설정
config = {
    'bootstrap.servers': 'broker:9092'  # --bootstrap-server에 대응
}

# Producer 생성
producer = Producer(config)

# 메시지 전송 (my-topic에 대응)
producer.produce(topic='my-topic', value=b'Hello')
producer.flush()  # CLI의 Enter 역할 (전송 완료 대기)
```

---

## 📋 Part 1: Producer 기본 설정 (30분)

### Producer 설정 딕셔너리

**필수 설정**:
- **`bootstrap.servers`**: Kafka 브로커 주소
  - 형식: `'host:port'` 또는 `'host1:port1,host2:port2'` (여러 브로커)
  - Docker 환경: 서비스 이름 사용 (`'broker:9092'`)

**권장 설정**:
- **`client.id`**: Producer 식별자
  - 로그/모니터링에서 어떤 Producer인지 확인 가능
  - 예: `'order-producer'`, `'log-collector'`

**신뢰성 설정** (6교시에서 자세히):
- **`acks`**: 메시지 전송 확인 수준
  - `0`: 확인 안 함 (가장 빠름)
  - `1`: Leader만 확인 (기본값)
  - `'all'` 또는 `-1`: 모든 복제본 확인 (가장 안전)

### Producer 생명 주기

```
1. Producer 생성       producer = Producer(config)
   ↓
2. 메시지 전송         producer.produce(topic, value)
   ↓
3. 전송 완료 대기      producer.flush()
   ↓
4. 종료               producer.close() (선택적)
```

In [ ]:
# Producer 기본 설정 예제
from confluent_kafka import Producer

# -----------------------------------------------------------------------------
# Producer 설정
# -----------------------------------------------------------------------------
config = {
    # 필수: Kafka 브로커 주소
    # Docker 환경에서는 서비스 이름 사용
    "bootstrap.servers": "broker:9092",
    # 권장: Producer 식별자
    # 로그에서 "order-producer가 메시지 전송"이라고 표시됨
    "client.id": "order-producer",
}

# -----------------------------------------------------------------------------
# Producer 객체 생성
# -----------------------------------------------------------------------------
# config 딕셔너리를 전달하여 Producer 인스턴스 생성
producer = Producer(config)

print("✅ Producer 생성 완료")
print(f"   연결된 브로커: {config['bootstrap.servers']}")
print(f"   Client ID: {config['client.id']}")

**예상 출력**:

```
✅ Producer 생성 완료
   연결된 브로커: broker:9092
   Client ID: order-producer
```

### flush()의 역할

**왜 flush()가 필요한가?**
- Producer는 성능을 위해 메시지를 **버퍼에 모았다가 한번에 전송**
- `produce()` 호출 후 **즉시 전송되지 않음** (비동기)
- `flush()`를 호출해야 **버퍼의 모든 메시지를 강제 전송**하고 완료 대기

**비유**:
- `produce()`: 편지를 우체통에 넣기
- `flush()`: 우체부가 편지를 다 수거할 때까지 기다리기

```python
# 버퍼에만 저장 (아직 전송 안 됨)
producer.produce('my-topic', b'message1')
producer.produce('my-topic', b'message2')

# 이 시점에 프로그램 종료하면? → 메시지 유실!

# 버퍼의 모든 메시지를 전송하고 완료 대기
producer.flush()  # ← 이제 안전하게 종료 가능
```

**timeout 옵션**:
```python
# 최대 5초 대기, 그 이후엔 전송 안 된 메시지 수 반환
remaining = producer.flush(timeout=5.0)
if remaining > 0:
    print(f"⚠️ {remaining}개 메시지 전송 실패")
```

In [ ]:
# flush() 실습
from confluent_kafka import Producer
import time

# Producer 생성
producer = Producer(
    {"bootstrap.servers": "broker:9092", "client.id": "flush-test-producer"}
)

# -----------------------------------------------------------------------------
# 실험 1: flush() 없이 전송
# -----------------------------------------------------------------------------
print("📤 실험 1: flush() 없이 메시지 전송")
producer.produce(topic="test-topic", value=b"Message without flush")
print("   produce() 호출 완료 (아직 전송 안 됨)")
time.sleep(0.1)  # 잠깐 대기

# -----------------------------------------------------------------------------
# 실험 2: flush()로 강제 전송
# -----------------------------------------------------------------------------
print("\n📤 실험 2: flush()로 강제 전송")
producer.produce(topic="test-topic", value=b"Message with flush")
print("   produce() 호출 완료")
print("   flush() 호출 중...")
producer.flush()  # 전송 완료까지 대기
print("   ✅ 전송 완료!")

# -----------------------------------------------------------------------------
# 실험 3: flush() with timeout
# -----------------------------------------------------------------------------
print("\n📤 실험 3: timeout 설정")
for i in range(5):
    producer.produce(topic="test-topic", value=f"Message {i}".encode("utf-8"))

# 최대 3초 대기
remaining = producer.flush(timeout=3.0)
if remaining == 0:
    print(f"   ✅ 모든 메시지 전송 완료")
else:
    print(f"   ⚠️ {remaining}개 메시지 아직 전송 중")

**예상 출력**:

```
📤 실험 1: flush() 없이 메시지 전송
   produce() 호출 완료 (아직 전송 안 됨)

📤 실험 2: flush()로 강제 전송
   produce() 호출 완료
   flush() 호출 중...
   ✅ 전송 완료!

📤 실험 3: timeout 설정
   ✅ 모든 메시지 전송 완료
```

---

## 🔄 Part 2: CLI → Python 변환 실습 (30분)

### CLI 명령어 분해

```bash
# CLI 명령어
echo "message1" | kafka-console-producer \
  --bootstrap-server localhost:9092 \
  --topic orders
```

### Python 코드로 변환

| CLI 요소 | Python 대응 | 설명 |
|---------|-----------|------|
| `--bootstrap-server localhost:9092` | `'bootstrap.servers': 'broker:9092'` | 브로커 주소 |
| `--topic orders` | `topic='orders'` | 토픽 이름 |
| `echo "message1"` | `value=b'message1'` | 메시지 내용 |
| Enter 입력 | `producer.flush()` | 전송 완료 대기 |

### bytes 타입 주의사항

**중요**: confluent-kafka는 메시지를 **bytes 타입**으로만 받음

```python
# ❌ 문자열 그대로 전송 불가
producer.produce('orders', 'message1')  # TypeError!

# ✅ bytes로 변환 필요
producer.produce('orders', b'message1')  # OK

# ✅ 또는 encode() 사용
producer.produce('orders', 'message1'.encode('utf-8'))  # OK
```

In [ ]:
# CLI → Python 변환 실습
from confluent_kafka import Producer

# -----------------------------------------------------------------------------
# 1. Producer 설정 (CLI의 --bootstrap-server)
# -----------------------------------------------------------------------------
config = {
    "bootstrap.servers": "broker:9092",  # CLI: --bootstrap-server localhost:9092
    "client.id": "cli-migration-demo",
}

producer = Producer(config)

# -----------------------------------------------------------------------------
# 2. 단일 메시지 전송 (CLI: echo "message1")
# -----------------------------------------------------------------------------
print("📤 메시지 1개 전송 (CLI 방식)")
producer.produce(
    topic="orders",  # CLI: --topic orders
    value=b"message1",  # CLI: echo "message1"
)
producer.flush()  # CLI: Enter 키 (전송 완료 대기)
print("   ✅ 전송 완료")

# -----------------------------------------------------------------------------
# 3. 여러 메시지 전송 (CLI에서 여러 번 Enter)
# -----------------------------------------------------------------------------
print("\n📤 메시지 3개 연속 전송")
messages = ["order-001", "order-002", "order-003"]

for msg in messages:
    # 문자열을 bytes로 변환 (encode)
    producer.produce(topic="orders", value=msg.encode("utf-8"))
    print(f"   버퍼에 추가: {msg}")

# 모든 메시지 한번에 전송
producer.flush()
print("   ✅ 3개 메시지 전송 완료")

**예상 출력**:

```
📤 메시지 1개 전송 (CLI 방식)
   ✅ 전송 완료

📤 메시지 3개 연속 전송
   버퍼에 추가: order-001
   버퍼에 추가: order-002
   버퍼에 추가: order-003
   ✅ 3개 메시지 전송 완료
```

### 실습: CLI 명령을 Python으로 변환하기

**문제**: 다음 CLI 명령을 Python 코드로 변환하세요.

```bash
kafka-console-producer \
  --bootstrap-server localhost:9092 \
  --topic user-events \
  < messages.txt
```

**힌트**:
1. `bootstrap.servers` 설정
2. `topic='user-events'`
3. 파일 읽기: `with open('messages.txt') as f:`
4. 각 줄마다 `produce()` 호출
5. 마지막에 `flush()`

In [ ]:
# 실습 답안
from confluent_kafka import Producer

# Producer 생성
producer = Producer(
    {
        "bootstrap.servers": "broker:9092",  # --bootstrap-server localhost:9092
        "client.id": "file-sender",
    }
)

# messages.txt 파일 내용 (시뮬레이션)
# 실제로는 파일 읽기: with open('messages.txt') as f:
messages_from_file = [
    "user-login: user123",
    "page-view: /products",
    "user-logout: user123",
]

print("📂 파일에서 메시지 읽어서 전송")
for line in messages_from_file:
    # 각 줄을 bytes로 변환하여 전송
    producer.produce(
        topic="user-events",  # --topic user-events
        value=line.encode("utf-8"),  # < messages.txt (각 줄)
    )
    print(f"   전송: {line}")

# 모든 메시지 전송 완료 대기
producer.flush()
print("✅ 파일 내용 전송 완료")

**예상 출력**:

```
📂 파일에서 메시지 읽어서 전송
   전송: user-login: user123
   전송: page-view: /products
   전송: user-logout: user123
✅ 파일 내용 전송 완료
```

---

## 📦 Part 3: JSON 메시지 전송 (30분)

### 왜 JSON인가?

**실무에서는 단순 문자열이 아닌 구조화된 데이터를 전송**:
- 주문 정보: `{"order_id": "ORD-001", "customer": "홍길동", "amount": 50000}`
- 로그 데이터: `{"timestamp": "2026-01-14", "level": "ERROR", "message": "..."}`
- 이벤트 데이터: `{"event_type": "purchase", "user_id": 123, "product_id": 456}`

### 직렬화 단계

```
1. Python dict        {"name": "홍길동"}
   ↓ json.dumps()
2. JSON 문자열        '{"name": "홍길동"}'
   ↓ .encode('utf-8')
3. bytes             b'{"name": "홍길동"}'
   ↓ producer.produce()
4. Kafka 전송        [Broker에 저장]
```

### JSON 직렬화 코드 패턴

```python
import json

# 1단계: Python dict 준비
order = {
    "order_id": "ORD-001",
    "customer": "홍길동",
    "amount": 50000
}

# 2단계: JSON 문자열로 변환
order_json = json.dumps(order)

# 3단계: bytes로 변환
order_bytes = order_json.encode('utf-8')

# 4단계: Kafka로 전송
producer.produce('orders', value=order_bytes)
producer.flush()
```

### 한 줄로 줄이기 (실무 패턴)

```python
# 위 4단계를 한 줄로
producer.produce(
    topic='orders',
    value=json.dumps(order).encode('utf-8')
)
```

### 한글 처리 주의사항

```python
# ❌ 한글이 깨질 수 있음
json.dumps(order)

# ✅ ensure_ascii=False 옵션 필수
json.dumps(order, ensure_ascii=False)
```

In [ ]:
# JSON 메시지 전송 예제
from confluent_kafka import Producer
import json

# Producer 생성
producer = Producer({"bootstrap.servers": "broker:9092", "client.id": "json-sender"})

# -----------------------------------------------------------------------------
# 예제 1: 단일 JSON 메시지 전송
# -----------------------------------------------------------------------------
print("📦 예제 1: 주문 정보 전송")

# 1단계: Python dict 준비
order = {
    "order_id": "ORD-001",
    "customer": "홍길동",
    "product": "노트북",
    "quantity": 1,
    "amount": 1500000,
}

# 2단계: JSON 문자열로 변환 (ensure_ascii=False: 한글 유지)
order_json = json.dumps(order, ensure_ascii=False)
print(f"   JSON 문자열: {order_json}")

# 3단계: bytes로 변환
order_bytes = order_json.encode("utf-8")
print(f"   bytes 타입: {type(order_bytes)}")

# 4단계: Kafka로 전송
producer.produce(topic="orders", value=order_bytes)
producer.flush()
print("   ✅ 전송 완료")

# -----------------------------------------------------------------------------
# 예제 2: 여러 JSON 메시지 한번에 전송 (실무 패턴)
# -----------------------------------------------------------------------------
print("\n📦 예제 2: 10개 주문 전송 (실무 패턴)")

for i in range(1, 11):
    # Python dict
    order = {
        "order_id": f"ORD-{i:03d}",  # ORD-001, ORD-002, ...
        "customer": f"고객{i}",
        "product": "상품A",
        "quantity": i,
        "amount": 10000 * i,
    }

    # dict → JSON → bytes → Kafka (한 줄로)
    producer.produce(
        topic="orders", value=json.dumps(order, ensure_ascii=False).encode("utf-8")
    )

    print(f"   버퍼 추가: {order['order_id']}")

# 모든 메시지 전송 완료 대기
producer.flush()
print("✅ 10개 주문 전송 완료")

**예상 출력**:

```
📦 예제 1: 주문 정보 전송
   JSON 문자열: {"order_id": "ORD-001", "customer": "홍길동", "product": "노트북", "quantity": 1, "amount": 1500000}
   bytes 타입: <class 'bytes'>
   ✅ 전송 완료

📦 예제 2: 10개 주문 전송 (실무 패턴)
   버퍼 추가: ORD-001
   버퍼 추가: ORD-002
   버퍼 추가: ORD-003
   버퍼 추가: ORD-004
   버퍼 추가: ORD-005
   버퍼 추가: ORD-006
   버퍼 추가: ORD-007
   버퍼 추가: ORD-008
   버퍼 추가: ORD-009
   버퍼 추가: ORD-010
✅ 10개 주문 전송 완료
```

### Key 지정으로 파티션 제어

**Key란?**
- 메시지의 **식별자** 역할
- 같은 Key를 가진 메시지는 **항상 같은 파티션**으로 전송
- 순서 보장이 필요할 때 사용

**사용 사례**:
- **사용자 ID를 Key로**: 특정 사용자의 이벤트를 순서대로 처리
- **주문 ID를 Key로**: 같은 주문의 상태 변경을 순서대로 처리
- **디바이스 ID를 Key로**: IoT 센서 데이터를 순서대로 처리

### Key가 없으면? (Key=None)

```
토픽: orders (3개 파티션)

Key 없음 (Round-robin 방식):
Message 1 → Partition 0
Message 2 → Partition 1
Message 3 → Partition 2
Message 4 → Partition 0  (순서 보장 X)
```

### Key가 있으면?

```
토픽: orders (3개 파티션)

Key 있음 (Hash 기반):
Message (key="user-123") → Partition 1
Message (key="user-123") → Partition 1 (같은 파티션!)
Message (key="user-456") → Partition 0
Message (key="user-123") → Partition 1 (순서 보장!)
```

### Key 전송 방법

```python
# Key도 bytes 타입으로 전달
producer.produce(
    topic='orders',
    key=b'user-123',              # ← Key 지정
    value=b'{"order_id": "001"}'
)
```

In [ ]:
# Key를 사용한 파티션 제어
from confluent_kafka import Producer
import json

# Producer 생성
producer = Producer(
    {"bootstrap.servers": "broker:9092", "client.id": "key-demo-producer"}
)

# -----------------------------------------------------------------------------
# 시나리오: 3명의 고객이 각각 3개씩 주문
# -----------------------------------------------------------------------------
print("📦 Key를 사용한 주문 전송")
print("   목표: 같은 고객의 주문은 같은 파티션으로\n")

customers = ["customer-A", "customer-B", "customer-C"]

for customer in customers:
    print(f"👤 {customer}의 주문 3건 전송")

    for order_num in range(1, 4):
        order = {
            "order_id": f"{customer}-{order_num}",
            "customer": customer,
            "product": f"상품-{order_num}",
            "amount": 10000 * order_num,
        }

        # Key에 customer 지정 → 같은 고객은 같은 파티션으로
        producer.produce(
            topic="orders",
            key=customer.encode("utf-8"),  # ← Key: 고객 ID
            value=json.dumps(order, ensure_ascii=False).encode("utf-8"),
        )

        print(f"   전송: {order['order_id']} (key={customer})")

# 모든 메시지 전송 완료
producer.flush()
print("\n✅ 전송 완료")
print("💡 Kafka UI에서 확인해보세요:")
print("   - 같은 customer의 메시지가 같은 파티션에 있는지")
print("   - http://localhost:8080")

**예상 출력**:

```
📦 Key를 사용한 주문 전송
   목표: 같은 고객의 주문은 같은 파티션으로

👤 customer-A의 주문 3건 전송
   전송: customer-A-1 (key=customer-A)
   전송: customer-A-2 (key=customer-A)
   전송: customer-A-3 (key=customer-A)
👤 customer-B의 주문 3건 전송
   전송: customer-B-1 (key=customer-B)
   전송: customer-B-2 (key=customer-B)
   전송: customer-B-3 (key=customer-B)
👤 customer-C의 주문 3건 전송
   전송: customer-C-1 (key=customer-C)
   전송: customer-C-2 (key=customer-C)
   전송: customer-C-3 (key=customer-C)

✅ 전송 완료
💡 Kafka UI에서 확인해보세요:
   - 같은 customer의 메시지가 같은 파티션에 있는지
   - http://localhost:8080
```

### 실습: 사용자 이벤트 전송 (Key 사용)

**시나리오**: 웹사이트에서 3명의 사용자가 각각 다음 순서로 행동합니다.

- user-001: 로그인 → 상품 조회 → 로그아웃
- user-002: 로그인 → 장바구니 추가 → 구매
- user-003: 로그인 → 상품 조회 → 상품 조회

**요구사항**:
1. 각 사용자 ID를 Key로 사용
2. 이벤트 정보를 JSON으로 전송
3. 같은 사용자의 이벤트는 순서 보장

**힌트**:
```python
event = {
    "user_id": "user-001",
    "event_type": "login",
    "timestamp": "2026-01-14 10:00:00"
}
producer.produce(
    topic='user-events',
    key=event['user_id'].encode('utf-8'),
    value=json.dumps(event, ensure_ascii=False).encode('utf-8')
)
```

In [ ]:
# 실습 답안
from confluent_kafka import Producer
import json
from datetime import datetime

# Producer 생성
producer = Producer(
    {"bootstrap.servers": "broker:9092", "client.id": "user-event-producer"}
)

# -----------------------------------------------------------------------------
# 사용자별 이벤트 시퀀스
# -----------------------------------------------------------------------------
user_events = [
    # user-001 이벤트
    ("user-001", "login", "10:00:00"),
    ("user-001", "view-product", "10:01:00"),
    ("user-001", "logout", "10:02:00"),
    # user-002 이벤트
    ("user-002", "login", "10:05:00"),
    ("user-002", "add-to-cart", "10:06:00"),
    ("user-002", "purchase", "10:07:00"),
    # user-003 이벤트
    ("user-003", "login", "10:10:00"),
    ("user-003", "view-product", "10:11:00"),
    ("user-003", "view-product", "10:12:00"),
]

print("📊 사용자 이벤트 전송 (Key 사용)")
print("=" * 60)

# 각 이벤트 전송
for user_id, event_type, timestamp in user_events:
    # JSON 이벤트 생성
    event = {
        "user_id": user_id,
        "event_type": event_type,
        "timestamp": f"2026-01-14 {timestamp}",
        "session_id": f"{user_id}-session-1",
    }

    # Kafka로 전송 (Key: user_id)
    producer.produce(
        topic="user-events",
        key=user_id.encode("utf-8"),  # ← Key: 사용자 ID
        value=json.dumps(event, ensure_ascii=False).encode("utf-8"),
    )

    print(f"✉️  {user_id:10s} | {event_type:15s} | {timestamp}")

# 전송 완료 대기
producer.flush()
print("=" * 60)
print("✅ 전송 완료")
print("\n💡 Kafka UI 확인:")
print("   1. http://localhost:8080")
print("   2. user-events 토픽 선택")
print("   3. 같은 user_id가 같은 파티션에 있는지 확인")

**예상 출력**:

```
📊 사용자 이벤트 전송 (Key 사용)
============================================================
✉️  user-001   | login           | 10:00:00
✉️  user-001   | view-product    | 10:01:00
✉️  user-001   | logout          | 10:02:00
✉️  user-002   | login           | 10:05:00
✉️  user-002   | add-to-cart     | 10:06:00
✉️  user-002   | purchase        | 10:07:00
✉️  user-003   | login           | 10:10:00
✉️  user-003   | view-product    | 10:11:00
✉️  user-003   | view-product    | 10:12:00
============================================================
✅ 전송 완료

💡 Kafka UI 확인:
   1. http://localhost:8080
   2. user-events 토픽 선택
   3. 같은 user_id가 같은 파티션에 있는지 확인
```

---

## 📡 Part 4: Delivery Callback (전송 확인)

### Callback이란?

**문제 상황**:
- `produce()` 호출 후 메시지가 **실제로 전송되었는지** 어떻게 알까?
- 네트워크 오류, Broker 장애로 **전송 실패**하면?
- 어떤 파티션에 저장되었는지 확인하고 싶다면?

**해결책: Delivery Callback**
- 메시지 전송 성공/실패 시 **자동으로 호출되는 함수**
- 전송 결과를 확인하고 로깅, 재시도 등 처리 가능

### Callback 함수 형식

```python
def delivery_callback(err, msg):
    """
    메시지 전송 후 자동으로 호출됨

    Args:
        err: 에러 발생 시 에러 객체, 성공 시 None
        msg: 전송된 메시지 객체
    """
    if err:
        print(f"❌ 전송 실패: {err}")
    else:
        print(f"✅ 전송 성공: {msg.topic()} [{msg.partition()}] offset {msg.offset()}")
```

### Callback 사용 방법

```python
# produce() 호출 시 callback 함수 전달
producer.produce(
    topic='orders',
    value=b'message',
    callback=delivery_callback  # ← Callback 함수 지정
)
producer.flush()  # Callback이 호출됨
```

### msg 객체에서 얻을 수 있는 정보

- `msg.topic()`: 토픽 이름
- `msg.partition()`: 저장된 파티션 번호
- `msg.offset()`: 파티션 내 위치 (offset)
- `msg.key()`: 메시지 Key
- `msg.value()`: 메시지 내용

In [ ]:
# Delivery Callback 예제
from confluent_kafka import Producer
import json


# -----------------------------------------------------------------------------
# Callback 함수 정의
# -----------------------------------------------------------------------------
def delivery_callback(err, msg):
    """
    메시지 전송 결과를 처리하는 콜백 함수

    Args:
        err: 에러 객체 (성공 시 None)
        msg: 전송된 메시지 객체
    """
    if err:
        # 전송 실패
        print(f"❌ 전송 실패: {err}")
        print(f"   실패한 메시지: {msg.value().decode('utf-8')}")
    else:
        # 전송 성공
        print(f"✅ 전송 성공")
        print(f"   토픽: {msg.topic()}")
        print(f"   파티션: {msg.partition()}")
        print(f"   Offset: {msg.offset()}")
        if msg.key():
            print(f"   Key: {msg.key().decode('utf-8')}")


# -----------------------------------------------------------------------------
# Producer 생성 및 메시지 전송
# -----------------------------------------------------------------------------
producer = Producer({"bootstrap.servers": "broker:9092", "client.id": "callback-demo"})

print("📡 Delivery Callback 데모\n")

# 메시지 3개 전송 (각각 callback 함수 지정)
for i in range(1, 4):
    order = {"order_id": f"ORD-{i:03d}", "amount": 10000 * i}

    print(f"📤 메시지 {i} 전송 시도...")
    producer.produce(
        topic="orders",
        key=f"customer-{i}".encode("utf-8"),
        value=json.dumps(order, ensure_ascii=False).encode("utf-8"),
        callback=delivery_callback,  # ← Callback 함수 등록
    )
    print()  # 빈 줄

# flush() 호출 시 callback 함수들이 실행됨
print("⏳ flush() 호출 중...\n")
producer.flush()
print("\n🎉 모든 callback 실행 완료")

**예상 출력**:

```
📡 Delivery Callback 데모

📤 메시지 1 전송 시도...

📤 메시지 2 전송 시도...

📤 메시지 3 전송 시도...

⏳ flush() 호출 중...

✅ 전송 성공
   토픽: orders
   파티션: 2
   Offset: 20
   Key: customer-3
✅ 전송 성공
   토픽: orders
   파티션: 0
   Offset: 0
   Key: customer-1
✅ 전송 성공
   토픽: orders
   파티션: 0
   Offset: 1
   Key: customer-2

🎉 모든 callback 실행 완료
```

> **참고**: Offset 값은 토픽의 현재 상태에 따라 달라질 수 있습니다.

### 실전 패턴: Callback으로 전송 성공률 추적

**시나리오**: 100개 메시지 전송 후 성공/실패 통계 확인

In [ ]:
# 실전 패턴: 전송 통계 추적
from confluent_kafka import Producer
import json

# -----------------------------------------------------------------------------
# 전송 통계를 추적하는 Callback
# -----------------------------------------------------------------------------
# 전역 변수로 통계 저장
stats = {"success": 0, "failed": 0, "total": 0}


def stats_callback(err, msg):
    """전송 통계를 업데이트하는 Callback"""
    stats["total"] += 1

    if err:
        stats["failed"] += 1
        print(f"❌ [{stats['total']}] 실패: {err}")
    else:
        stats["success"] += 1
        # 성공 시에는 조용히 (로그 줄이기)


# -----------------------------------------------------------------------------
# 100개 메시지 전송
# -----------------------------------------------------------------------------
producer = Producer({"bootstrap.servers": "broker:9092", "client.id": "stats-tracker"})

print("📊 100개 메시지 전송 및 통계 추적")
print("=" * 60)

for i in range(1, 101):
    order = {"order_id": f"ORD-{i:03d}", "amount": 10000}

    producer.produce(
        topic="orders",
        value=json.dumps(order).encode("utf-8"),
        callback=stats_callback,  # ← 통계 Callback
    )

# 전송 완료 및 Callback 실행
print("⏳ 전송 중...")
producer.flush()

# -----------------------------------------------------------------------------
# 통계 출력
# -----------------------------------------------------------------------------
print("=" * 60)
print("📈 전송 통계")
print(f"   총 전송: {stats['total']}개")
print(f"   성공: {stats['success']}개")
print(f"   실패: {stats['failed']}개")
print(f"   성공률: {stats['success'] / stats['total'] * 100:.1f}%")

**예상 출력**:

```
📊 100개 메시지 전송 및 통계 추적
============================================================
⏳ 전송 중...
============================================================
📈 전송 통계
   총 전송: 100개
   성공: 100개
   실패: 0개
   성공률: 100.0%
```

---

## 🎯 종합 실습: E-commerce 주문 시스템

### 실습 목표

지금까지 배운 모든 내용을 활용하여 주문 시스템 구현:
1. ✅ Producer 설정
2. ✅ JSON 직렬화
3. ✅ Key 지정 (고객 ID)
4. ✅ Delivery Callback

### 요구사항

**10개 주문 전송**:
- 주문 ID: `ORD-001` ~ `ORD-010`
- 고객: `customer-1` ~ `customer-5` (고객당 2개 주문)
- 상품: `["노트북", "마우스", "키보드", "모니터", "헤드셋"]` 중 랜덤
- 수량: 1~3 랜덤
- Key: 고객 ID (같은 고객의 주문은 같은 파티션으로)
- Callback으로 성공 확인

**JSON 형식**:
```json
{
  "order_id": "ORD-001",
  "customer_id": "customer-1",
  "product": "노트북",
  "quantity": 2,
  "amount": 3000000,
  "timestamp": "2026-01-14 15:30:00"
}
```

In [ ]:
# 종합 실습 답안
from confluent_kafka import Producer
import json
import random
from datetime import datetime

# -----------------------------------------------------------------------------
# 설정 및 데이터
# -----------------------------------------------------------------------------
PRODUCTS = {
    "노트북": 1500000,
    "마우스": 30000,
    "키보드": 80000,
    "모니터": 300000,
    "헤드셋": 50000,
}

CUSTOMERS = ["customer-1", "customer-2", "customer-3", "customer-4", "customer-5"]


# -----------------------------------------------------------------------------
# Callback 함수
# -----------------------------------------------------------------------------
def order_callback(err, msg):
    """주문 전송 결과 처리"""
    if err:
        print(f"   ❌ 전송 실패: {err}")
    else:
        order_id = json.loads(msg.value().decode("utf-8"))["order_id"]
        print(f"   ✅ {order_id} → Partition {msg.partition()}, Offset {msg.offset()}")


# -----------------------------------------------------------------------------
# Producer 생성
# -----------------------------------------------------------------------------
producer = Producer(
    {"bootstrap.servers": "broker:9092", "client.id": "ecommerce-order-producer"}
)

# -----------------------------------------------------------------------------
# 10개 주문 생성 및 전송
# -----------------------------------------------------------------------------
print("🛒 E-commerce 주문 시스템 시작")
print("=" * 70)

for order_num in range(1, 11):
    # 주문 데이터 생성
    product = random.choice(list(PRODUCTS.keys()))
    quantity = random.randint(1, 3)
    customer_id = CUSTOMERS[(order_num - 1) % 5]  # 고객당 2개씩

    order = {
        "order_id": f"ORD-{order_num:03d}",
        "customer_id": customer_id,
        "product": product,
        "quantity": quantity,
        "amount": PRODUCTS[product] * quantity,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }

    # 주문 정보 출력
    print(
        f"📦 {order['order_id']}: {customer_id} | {product} x{quantity} | {order['amount']:,}원"
    )

    # Kafka로 전송
    producer.produce(
        topic="orders",
        key=customer_id.encode("utf-8"),  # Key: 고객 ID
        value=json.dumps(order, ensure_ascii=False).encode("utf-8"),
        callback=order_callback,  # Callback 등록
    )

# 전송 완료
print("\n⏳ 전송 중...")
producer.flush()
print("=" * 70)
print("✅ 10개 주문 전송 완료!")
print("\n💡 Kafka UI에서 확인:")
print("   http://localhost:8080")
print("   - orders 토픽 확인")
print("   - 같은 customer_id가 같은 파티션에 있는지 확인")

**예상 출력** (random 결과에 따라 상품은 달라질 수 있음):

```
🛒 E-commerce 주문 시스템 시작
======================================================================
📦 ORD-001: customer-1 | 노트북 x1 | 1,500,000원
📦 ORD-002: customer-2 | 키보드 x1 | 80,000원
📦 ORD-003: customer-3 | 마우스 x1 | 30,000원
📦 ORD-004: customer-4 | 노트북 x3 | 4,500,000원
📦 ORD-005: customer-5 | 헤드셋 x1 | 50,000원
📦 ORD-006: customer-1 | 헤드셋 x2 | 100,000원
📦 ORD-007: customer-2 | 노트북 x1 | 1,500,000원
📦 ORD-008: customer-3 | 노트북 x1 | 1,500,000원
📦 ORD-009: customer-4 | 마우스 x3 | 90,000원
📦 ORD-010: customer-5 | 헤드셋 x1 | 50,000원

⏳ 전송 중...
   ✅ ORD-005 → Partition 1, Offset 37
   ✅ ORD-010 → Partition 1, Offset 38
   ✅ ORD-003 → Partition 2, Offset 53
   ✅ ORD-008 → Partition 2, Offset 54
   ✅ ORD-001 → Partition 0, Offset 37
   ✅ ORD-002 → Partition 0, Offset 38
   ✅ ORD-004 → Partition 0, Offset 39
   ✅ ORD-006 → Partition 0, Offset 40
   ✅ ORD-007 → Partition 0, Offset 41
   ✅ ORD-009 → Partition 0, Offset 42
======================================================================
✅ 10개 주문 전송 완료!

💡 Kafka UI에서 확인:
   http://localhost:8080
   - orders 토픽 확인
   - 같은 customer_id가 같은 파티션에 있는지 확인
```

> **참고**: 같은 customer_id가 같은 파티션에 저장되는 것을 확인할 수 있습니다.
> - customer-5 → Partition 1
> - customer-3 → Partition 2
> - customer-1, customer-2, customer-4 → Partition 0

---

## ❓ FAQ

### Q1. produce()를 호출하면 즉시 전송되나요?

**A**: 아니요! Producer는 성능을 위해 메시지를 **버퍼에 모았다가** 배치로 전송합니다.

- `produce()`: 버퍼에 추가만 함
- `flush()`: 버퍼의 모든 메시지를 실제로 전송

**비유**: 택배 상자에 물건을 담기 (produce) vs 택배 발송하기 (flush)

### Q2. flush()를 매번 호출해야 하나요?

**A**: 상황에 따라 다릅니다.

- **실시간성이 중요**: 매번 `flush()` 호출
- **처리량이 중요**: 여러 메시지를 모아서 한번에 `flush()`
- **프로그램 종료 전**: 반드시 `flush()` 호출 (안하면 메시지 유실!)

### Q3. Key를 꼭 지정해야 하나요?

**A**: 아니요, 하지만 **순서 보장이 필요하면 필수**입니다.

- **Key 없음**: 메시지가 Round-robin으로 분산 → 순서 보장 안 됨
- **Key 있음**: 같은 Key는 항상 같은 파티션 → 순서 보장됨

**언제 Key 사용?**
- 같은 사용자의 이벤트는 순서대로 처리해야 할 때
- 같은 주문의 상태 변경을 순서대로 추적할 때
- 같은 디바이스의 센서 데이터를 순서대로 받을 때

### Q4. JSON 말고 다른 형식은 안 되나요?

**A**: 됩니다! Kafka는 bytes만 받으므로 어떤 형식이든 가능합니다.

- **Avro**: 스키마 기반 직렬화 (효율적, Schema Registry 필요)
- **Protobuf**: Google의 직렬화 형식
- **Pickle**: Python 객체 직렬화 (Python끼리만 가능)
- **Plain Text**: 단순 문자열

**JSON 장점**: 사람이 읽기 쉽고, 다양한 언어에서 지원

### Q5. Callback이 항상 호출되나요?

**A**: `flush()` 또는 자동 배치 전송 시 호출됩니다.

```python
producer.produce(..., callback=my_callback)  # Callback 등록만
producer.flush()  # ← 이 시점에 Callback 실행!
```

---

## 📝 핵심 요약

### Producer 생명 주기

```python
# 1. 생성
producer = Producer({'bootstrap.servers': 'broker:9092'})

# 2. 전송
producer.produce(
    topic='orders',
    key=b'customer-1',
    value=json.dumps(order).encode('utf-8'),
    callback=my_callback
)

# 3. 완료 대기
producer.flush()
```

### 핵심 개념

| 개념 | 설명 | 예시 |
|------|------|------|
| **bootstrap.servers** | Kafka 브로커 주소 | `'broker:9092'` |
| **topic** | 메시지 카테고리 | `'orders'`, `'logs'` |
| **key** | 파티션 결정 기준 | 고객 ID, 사용자 ID |
| **value** | 메시지 내용 (bytes) | JSON, Avro, Text |
| **callback** | 전송 결과 처리 함수 | 성공/실패 로깅 |
| **flush()** | 버퍼 강제 전송 | 프로그램 종료 전 필수 |

### JSON 직렬화 패턴 (실무)

```python
import json

# Python dict → JSON → bytes → Kafka
producer.produce(
    topic='orders',
    key=customer_id.encode('utf-8'),
    value=json.dumps(order, ensure_ascii=False).encode('utf-8'),
    callback=delivery_callback
)
producer.flush()
```

### CLI vs Python 대응표

| CLI | Python | 설명 |
|-----|--------|------|
| `--bootstrap-server` | `bootstrap.servers` | 브로커 주소 |
| `--topic orders` | `topic='orders'` | 토픽 이름 |
| `echo "msg"` | `value=b'msg'` | 메시지 내용 |
| Enter 입력 | `flush()` | 전송 완료 |

---

## 🎯 다음 교시 예고

**3교시: Consumer 깊이 있게 알기**
- Consumer 설정과 poll loop
- Signal handling (ctrl+C 문제 해결!)
- Consumer Group 실습
- 파티션 재분배 (Rebalancing)

이제 메시지를 **보내는** 방법을 마스터했습니다.
다음 시간에는 메시지를 **받는** 방법을 배웁니다!

---


# Day 8 - 3교시: Consumer 깊이 있게 알기

---

## 🎯 수업 목표

이 교시를 마치면 다음을 할 수 있습니다:

- ✅ Consumer 설정 옵션을 이해하고 적절히 사용할 수 있다
- ✅ poll() 메서드와 timeout의 동작 원리를 이해한다
- ✅ JSON 메시지를 역직렬화하여 처리할 수 있다
- ✅ Signal handling으로 ctrl+C 문제를 해결할 수 있다 (중요!)
- ✅ Consumer Group의 동작 원리를 이해한다
- ✅ 여러 Consumer가 파티션을 나눠서 처리하는 것을 확인할 수 있다
- ✅ Kafka UI에서 Consumer Lag을 모니터링할 수 있다

---

## 📋 Part 1: Consumer 기본 (30분)

### Consumer 설정 딕셔너리

**필수 설정**:

1. **`bootstrap.servers`**: Kafka 브로커 주소
   - Producer와 동일

2. **`group.id`**: Consumer Group ID
   - 같은 `group.id`를 가진 Consumer들은 **파티션을 나눠서 처리**
   - 예: `'order-processor'`, `'log-collector'`

3. **`auto.offset.reset`**: 처음 시작 위치
   - **`earliest`**: 토픽의 **처음부터** 읽기 (CLI의 `--from-beginning`)
   - **`latest`**: **최신 메시지부터** 읽기 (기본값)
   - **`none`**: Offset이 없으면 에러 발생

**선택 설정** (중요):

4. **`enable.auto.commit`**: 자동 Offset 커밋 여부
   - **`True`** (기본값): 자동으로 Offset 커밋
   - **`False`**: 수동으로 `commit()` 호출 필요 (더 안전)

5. **`auto.commit.interval.ms`**: 자동 커밋 간격
   - 기본값: 5000ms (5초)
   - `enable.auto.commit=True`일 때만 사용

### auto.offset.reset 상세 설명

**상황별 동작**:

```
토픽: orders (메시지 100개 있음)

Consumer가 처음 시작할 때:

1. earliest: 메시지 1번부터 읽음
   [1, 2, 3, ..., 100] 모두 읽음

2. latest: 메시지 101번부터 읽음
   [1~100] 무시, 새 메시지만 읽음

3. none: 에러 발생
   "Offset이 없습니다!" 에러
```

**실무 팁**:
- **개발/테스트**: `earliest` 사용 (모든 메시지 확인)
- **프로덕션**: `latest` 사용 (신규 메시지만 처리)

### group.id의 중요성

**같은 group.id를 가진 Consumer들**:
- 파티션을 **자동으로 분담**
- 한 파티션은 **한 Consumer만** 처리
- Consumer 추가/제거 시 **자동 재분배** (Rebalancing)

**다른 group.id를 가진 Consumer들**:
- 서로 **독립적**으로 동작
- 모든 메시지를 **각각** 읽음

```
토픽: orders (3개 파티션)

Consumer Group A (group.id='group-a'):
  Consumer 1 → Partition 0
  Consumer 2 → Partition 1
  Consumer 3 → Partition 2

Consumer Group B (group.id='group-b'):
  Consumer 1 → Partition 0, 1, 2 (모두 처리)

두 그룹은 서로 독립적!
```

In [ ]:
# Consumer 기본 설정 예제
from confluent_kafka import Consumer

# -----------------------------------------------------------------------------
# Consumer 설정
# -----------------------------------------------------------------------------
config = {
    # 필수: Kafka 브로커 주소
    "bootstrap.servers": "broker:9092",
    # 필수: Consumer Group ID
    # 같은 group.id를 가진 Consumer들은 파티션을 나눠서 처리
    "group.id": "order-consumer-group",
    # 필수: 처음 시작할 때 어디서부터 읽을지
    # 'earliest': 처음부터 (CLI의 --from-beginning)
    # 'latest': 최신 메시지부터 (기본값)
    "auto.offset.reset": "earliest",
    # 선택: 자동 Offset 커밋 (기본값: True)
    # True: 자동으로 "여기까지 읽었습니다" 기록
    # False: 수동으로 commit() 호출 필요 (더 안전)
    "enable.auto.commit": True,
    # 선택: 자동 커밋 간격 (밀리초)
    "auto.commit.interval.ms": 5000,  # 5초마다 자동 커밋
}

# -----------------------------------------------------------------------------
# Consumer 인스턴스 생성
# -----------------------------------------------------------------------------
consumer = Consumer(config)

print("✅ Consumer 생성 완료")
print(f"   브로커: {config['bootstrap.servers']}")
print(f"   Group ID: {config['group.id']}")
print(f"   시작 위치: {config['auto.offset.reset']}")

### poll() 메서드 깊이 파기

**poll()의 역할**:
- Kafka에서 메시지를 **하나씩** 가져옴
- timeout 시간 동안 **대기**
- 메시지가 있으면 **Message 객체** 반환
- 없으면 **None** 반환

**형식**:
```python
msg = consumer.poll(timeout=1.0)  # 1초 대기
```

**timeout 의미**:
- 메시지가 도착할 때까지 **최대 대기 시간** (초 단위)
- 메시지가 있으면 **즉시 반환** (timeout 무시)
- timeout 내에 메시지가 없으면 **None 반환**

**반환값 처리**:

```python
msg = consumer.poll(timeout=1.0)

if msg is None:
    # timeout 시간 내에 메시지가 없음
    print("메시지 없음")

elif msg.error():
    # 에러 발생 (네트워크 오류, 파티션 재분배 등)
    print(f"에러: {msg.error()}")

else:
    # 정상 메시지
    value = msg.value()        # bytes 타입
    key = msg.key()            # bytes 타입 (없으면 None)
    partition = msg.partition() # 파티션 번호
    offset = msg.offset()       # Offset 번호
```

### 메시지 역직렬화 (JSON)

**Producer에서 JSON으로 보냈다면**:

```python
# Producer가 보낸 방식
producer.produce(
    topic='orders',
    value=json.dumps(order).encode('utf-8')
)
```

**Consumer에서 역직렬화**:

```python
# 1단계: bytes → 문자열
message_str = msg.value().decode('utf-8')

# 2단계: 문자열 → dict
order = json.loads(message_str)

# 한 줄로
order = json.loads(msg.value().decode('utf-8'))
```

In [ ]:
# poll() 메서드 실습
from confluent_kafka import Consumer
import json

# Consumer 생성
consumer = Consumer(
    {
        "bootstrap.servers": "broker:9092",
        "group.id": "order-reader",
        "auto.offset.reset": "earliest",
    }
)

# 토픽 구독
consumer.subscribe(["orders"])

print("📨 메시지 읽기 시작 (5개 읽기)")
print("=" * 60)

# -----------------------------------------------------------------------------
# 메시지 5개 읽기
# -----------------------------------------------------------------------------
for i in range(5):
    # poll(): 메시지 하나 읽기 (최대 2초 대기)
    msg = consumer.poll(timeout=2.0)

    # 메시지 확인
    if msg is None:
        # timeout 내에 메시지가 없음
        print(f"{i + 1}. ⏰ 타임아웃: 메시지 없음")
        continue

    if msg.error():
        # 에러 발생
        print(f"{i + 1}. ❌ 에러: {msg.error()}")
        continue

    # -----------------------------------------------------------------------------
    # 정상 메시지 처리
    # -----------------------------------------------------------------------------
    # bytes → 문자열 → dict
    order = json.loads(msg.value().decode("utf-8"))

    # 메시지 정보 출력
    print(f"{i + 1}. ✅ 메시지 수신")
    print(f"   주문 ID: {order.get('order_id', 'N/A')}")
    print(f"   금액: {order.get('amount', 0):,}원")
    print(f"   파티션: {msg.partition()}")
    print(f"   Offset: {msg.offset()}")

    # Key가 있으면 출력
    if msg.key():
        print(f"   Key: {msg.key().decode('utf-8')}")
    print()

# Consumer 종료
consumer.close()
print("=" * 60)
print("✅ Consumer 종료 완료")

---

## 🛑 Part 2: Signal Handling (ctrl+C 문제 해결!) (30분)

### 문제 상황

**Consumer를 무한 루프로 실행하면**:

```python
while True:
    msg = consumer.poll(timeout=1.0)
    # 메시지 처리
```

**문제점**:
- ctrl+C를 눌러도 **즉시 종료 안 됨**
- Consumer가 제대로 **close()를 호출하지 못함**
- Kafka에 "나 종료했어요" 알림을 못 보냄
- 다음 실행 시 **Rebalancing 지연** 발생

### 올바른 종료 방법

**Signal handling 사용**:
1. ctrl+C (SIGINT) 신호를 **감지**
2. 무한 루프를 **종료**
3. `consumer.close()`를 **호출**

**코드 패턴**:

```python
import signal

# 1. 종료 플래그
running = True

# 2. Signal handler 함수
def signal_handler(sig, frame):
    global running
    print("\n종료 신호 수신...")
    running = False

# 3. Signal 등록
signal.signal(signal.SIGINT, signal_handler)

# 4. 무한 루프 (running 플래그 확인)
while running:
    msg = consumer.poll(timeout=1.0)
    # 메시지 처리

# 5. 정상 종료
consumer.close()
```

### signal 모듈 설명

**signal.SIGINT**:
- ctrl+C를 눌렀을 때 발생하는 **시그널**
- INT = Interrupt (중단)

**signal.signal()**:
- 특정 시그널이 발생하면 **함수를 호출**
- 형식: `signal.signal(시그널, 함수)`

**signal_handler 함수**:
- 시그널 발생 시 **자동으로 호출**
- 매개변수:
  - `sig`: 시그널 번호
  - `frame`: 현재 실행 프레임 (무시 가능)

In [ ]:
# Signal handling 예제 (올바른 Consumer 종료)
from confluent_kafka import Consumer
import json
import signal

# -----------------------------------------------------------------------------
# 1. 종료 플래그
# -----------------------------------------------------------------------------
# running: Consumer가 계속 실행될지 결정
# True: 계속 실행, False: 종료
running = True


# -----------------------------------------------------------------------------
# 2. Signal Handler 함수
# -----------------------------------------------------------------------------
def signal_handler(sig, frame):
    """
    ctrl+C (SIGINT) 발생 시 호출되는 함수

    Args:
        sig: 시그널 번호
        frame: 현재 실행 프레임 (무시)
    """
    global running
    print("\n🛑 종료 신호 수신 (ctrl+C)")
    print("   Consumer를 안전하게 종료 중...")
    running = False  # 무한 루프 종료


# -----------------------------------------------------------------------------
# 3. Signal 등록
# -----------------------------------------------------------------------------
# SIGINT (ctrl+C) 발생 시 signal_handler 함수를 호출하도록 등록
signal.signal(signal.SIGINT, signal_handler)

# -----------------------------------------------------------------------------
# 4. Consumer 생성 및 구독
# -----------------------------------------------------------------------------
consumer = Consumer(
    {
        "bootstrap.servers": "broker:9092",
        "group.id": "signal-demo-group",
        "auto.offset.reset": "earliest",
    }
)

consumer.subscribe(["orders"])

print("📨 Consumer 실행 중...")
print("   ctrl+C를 누르면 안전하게 종료됩니다")
print("=" * 60)

# -----------------------------------------------------------------------------
# 5. 무한 루프 (running 플래그 확인)
# -----------------------------------------------------------------------------
message_count = 0

# while True:  ← 이렇게 하면 ctrl+C가 안 먹힘!
while running:  # ✅ 이렇게 하면 ctrl+C가 먹힘!
    # poll(): 메시지 읽기 (timeout=1.0초)
    msg = consumer.poll(timeout=1.0)

    # 메시지 없으면 다음 루프
    if msg is None:
        continue

    # 에러 발생 시
    if msg.error():
        print(f"❌ 에러: {msg.error()}")
        continue

    # 메시지 처리
    message_count += 1
    order = json.loads(msg.value().decode("utf-8"))

    print(f"✅ [{message_count}] 주문 처리: {order.get('order_id', 'N/A')}")

# -----------------------------------------------------------------------------
# 6. 정상 종료
# -----------------------------------------------------------------------------
print("=" * 60)
print(f"📊 처리 완료: 총 {message_count}개 메시지")
print("   Consumer 종료 중...")

# close(): Kafka에 "나 종료했어요" 알림
# 이걸 호출해야 다음 실행 시 Rebalancing이 빠름!
consumer.close()
print("✅ Consumer 종료 완료")

### 실습: Signal Handling 비교

**잘못된 방법 vs 올바른 방법**:

**❌ 잘못된 방법** (ctrl+C 안 먹힘):
```python
while True:  # 무한 루프
    msg = consumer.poll(timeout=1.0)
    # 처리
# ctrl+C를 눌러도 즉시 종료 안 됨!
# consumer.close()를 호출하지 못함!
```

**✅ 올바른 방법** (ctrl+C 즉시 종료):
```python
running = True

def signal_handler(sig, frame):
    global running
    running = False

signal.signal(signal.SIGINT, signal_handler)

while running:  # running 플래그 확인
    msg = consumer.poll(timeout=1.0)
    # 처리

consumer.close()  # 안전하게 종료!
```

### close()를 호출하지 않으면?

**문제점**:
1. **Kafka가 Consumer 종료를 모름**
   - 30초 동안 (기본 session timeout) 기다림
   - 그동안 파티션이 **잠김** (다른 Consumer가 처리 못함)

2. **다음 실행 시 Rebalancing 지연**
   - "이 Consumer 죽었나?" 확인하는 시간 필요
   - 불필요한 대기 시간 발생

3. **Offset 커밋 유실 가능**
   - 마지막 처리한 위치가 기록 안 됨
   - 중복 처리 발생 가능

---

## 👥 Part 3: Consumer Group 실습 (30분)

### Consumer Group이란?

**정의**:
- 같은 `group.id`를 가진 **Consumer들의 집합**
- 파티션을 **자동으로 분담**하여 처리
- 한 파티션은 **한 Consumer만** 처리 (중복 방지)

**비유: 택배 분류 작업**:
```
택배 창고 (토픽):
  - 1번 구역 (파티션 0)
  - 2번 구역 (파티션 1)
  - 3번 구역 (파티션 2)

작업자 팀 (Consumer Group):
  - 작업자 A → 1번 구역 담당
  - 작업자 B → 2번 구역 담당
  - 작업자 C → 3번 구역 담당

작업자가 추가되면?
  → 자동으로 구역 재분배! (Rebalancing)
```

### 파티션 분배 규칙

**Consumer 수 ≤ 파티션 수** (정상):
```
토픽: orders (3개 파티션)
Consumer Group: group-a (3개 Consumer)

Consumer 1 → Partition 0
Consumer 2 → Partition 1
Consumer 3 → Partition 2
```

**Consumer 수 > 파티션 수** (비효율):
```
토픽: orders (3개 파티션)
Consumer Group: group-a (5개 Consumer)

Consumer 1 → Partition 0
Consumer 2 → Partition 1
Consumer 3 → Partition 2
Consumer 4 → (유휴)  ← 파티션 없음!
Consumer 5 → (유휴)  ← 파티션 없음!
```

**Consumer 수 < 파티션 수**:
```
토픽: orders (3개 파티션)
Consumer Group: group-a (2개 Consumer)

Consumer 1 → Partition 0, 1  ← 2개 처리
Consumer 2 → Partition 2
```

### Rebalancing (재분배)

**Rebalancing 발생 시점**:
- Consumer **추가** 시
- Consumer **제거** 시 (종료, 장애)
- 파티션 **추가** 시

**Rebalancing 과정**:
1. 모든 Consumer가 **일시 중지**
2. 파티션을 **재분배**
3. Consumer들이 **새 파티션** 할당받음
4. 처리 **재개**

**주의사항**:
- Rebalancing 중에는 **메시지 처리 안 됨** (일시 중지)
- 자주 발생하면 **성능 저하**
- Consumer를 함부로 종료하면 Rebalancing 발생!

In [ ]:
# Consumer Group 실습 - 단일 Consumer
from confluent_kafka import Consumer
import json
import signal

# 종료 플래그
running = True


def signal_handler(sig, frame):
    global running
    running = False


signal.signal(signal.SIGINT, signal_handler)

# -----------------------------------------------------------------------------
# Consumer 생성 (Group: demo-group)
# -----------------------------------------------------------------------------
consumer = Consumer(
    {
        "bootstrap.servers": "broker:9092",
        "group.id": "demo-group",  # ← Consumer Group ID
        "auto.offset.reset": "earliest",
    }
)

consumer.subscribe(["orders"])

print("📨 Consumer 1 실행 중 (Group: demo-group)")
print("   할당된 파티션 확인 중...")

# -----------------------------------------------------------------------------
# 첫 메시지 읽기 (파티션 할당 대기)
# -----------------------------------------------------------------------------
# 첫 poll() 호출 시 파티션 할당이 일어남
msg = consumer.poll(timeout=5.0)

# -----------------------------------------------------------------------------
# 할당된 파티션 확인
# -----------------------------------------------------------------------------
# assignment(): 현재 Consumer에 할당된 파티션 목록
assigned_partitions = consumer.assignment()

print("\n✅ 파티션 할당 완료")
print(f"   할당된 파티션: {len(assigned_partitions)}개")
for tp in assigned_partitions:
    # tp: TopicPartition 객체
    print(f"      - {tp.topic} [파티션 {tp.partition}]")

# -----------------------------------------------------------------------------
# 메시지 읽기
# -----------------------------------------------------------------------------
print("\n📨 메시지 읽기 시작 (ctrl+C로 종료)\n")

message_count = 0

while running:
    msg = consumer.poll(timeout=1.0)

    if msg is None:
        continue

    if msg.error():
        print(f"❌ 에러: {msg.error()}")
        continue

    # 메시지 처리
    message_count += 1
    order = json.loads(msg.value().decode("utf-8"))

    print(
        f"✅ [{message_count}] Partition {msg.partition()} | "
        f"Offset {msg.offset()} | "
        f"주문 {order.get('order_id', 'N/A')}"
    )

# 종료
consumer.close()
print(f"\n✅ 종료: 총 {message_count}개 메시지 처리")

### 실습: 여러 Consumer 동시 실행

**시나리오**: 같은 Consumer Group에 3개 Consumer 실행

**실행 방법**:

```bash
# Terminal 1: Consumer 1 실행
docker compose run --rm python-producer python consumer_group_1.py

# Terminal 2: Consumer 2 실행
docker compose run --rm python-producer python consumer_group_2.py

# Terminal 3: Consumer 3 실행
docker compose run --rm python-producer python consumer_group_3.py
```

**관찰 포인트**:
1. 각 Consumer가 **다른 파티션** 할당받음
2. 같은 메시지가 **한 Consumer에만** 전달됨 (중복 없음)
3. Consumer 종료 시 **Rebalancing** 발생

### Kafka UI에서 Consumer Group 모니터링

**접속**: `http://localhost:8080`

**확인 방법**:
1. 왼쪽 메뉴에서 **Consumers** 클릭
2. `demo-group` 선택
3. 다음 정보 확인:
   - **Members**: Consumer 수
   - **Lag**: 처리되지 않은 메시지 수
   - **Partition Assignment**: 파티션 할당 상태

**Lag이란?**:
- Producer가 보낸 메시지 - Consumer가 읽은 메시지
- Lag이 크면: Consumer가 느림 → Consumer 추가 필요
- Lag이 0이면: 실시간 처리 중

---

## ❓ FAQ

### Q1. poll()의 timeout은 왜 필요한가요?

**A**: 메시지가 없을 때 **무한정 기다리지 않도록** 하기 위함입니다.

**timeout 없으면**:
- 메시지가 올 때까지 **영원히 대기**
- ctrl+C를 눌러도 **반응 안 함** (poll()에서 멈춰있음)

**timeout 있으면**:
- 1초마다 `running` 플래그 확인 가능
- ctrl+C 신호를 **빠르게 감지**

**권장 timeout**:
- 1.0초: 일반적인 경우
- 0.1초: 빠른 응답이 필요한 경우
- 5.0초: 메시지가 드물게 오는 경우

### Q2. auto.offset.reset이 왜 중요한가요?

**A**: Consumer가 **처음 시작할 때** 어디서부터 읽을지 결정합니다.

**잘못 설정하면**:
- `latest`: 기존 메시지 **놓침** (테스트 시 문제!)
- `earliest`: 이미 처리한 메시지 **중복 처리**

**상황별 선택**:
- **개발/테스트**: `earliest` (모든 메시지 확인)
- **프로덕션 신규**: `latest` (신규 메시지만)
- **프로덕션 재시작**: Offset이 저장되어 있어서 상관없음

### Q3. Consumer Group을 왜 사용하나요?

**A**: **병렬 처리**로 처리량을 늘리기 위함입니다.

**Consumer 1개**:
- 3개 파티션 모두 처리
- 처리 속도: 100 msg/sec

**Consumer 3개 (같은 Group)**:
- 각자 1개 파티션씩 처리
- 처리 속도: 300 msg/sec (3배!)

**비유**: 택배 상자 정리
- 1명: 시간당 100개
- 3명: 시간당 300개 (각자 구역 분담)

### Q4. group.id를 같게 하면 메시지를 한 번만 읽나요?

**A**: 네, 같은 group.id를 가진 Consumer들은 **파티션을 나눠서** 처리합니다.

**같은 group.id** (`'group-a'`):
- 메시지가 **한 Consumer에만** 전달
- 중복 없음

**다른 group.id** (`'group-a'`, `'group-b'`):
- 메시지가 **각 그룹에 모두** 전달
- 독립적으로 처리

**예시**:
```
Producer → 메시지 1개 전송

Consumer A (group-a) → 받음
Consumer B (group-a) → 안 받음 (A가 받았으므로)
Consumer C (group-b) → 받음 (다른 그룹이므로)
```

### Q5. Consumer를 더 많이 띄우면 항상 빠른가요?

**A**: 아니요, **파티션 수만큼**만 효과가 있습니다.

**토픽: orders (3개 파티션)**:

| Consumer 수 | 활용도 | 처리 속도 |
|------------|--------|----------|
| 1개 | 100% (3개 파티션 처리) | 1x |
| 2개 | 100% (2개 + 1개) | 2x |
| 3개 | 100% (1개씩) | 3x ✅ |
| 4개 | 75% (1개 유휴) | 3x (효과 없음!) |
| 5개 | 60% (2개 유휴) | 3x (효과 없음!) |

**결론**: Consumer 수 = 파티션 수가 **최적**

---

## 📝 핵심 요약

### Consumer 필수 설정

| 설정 | 의미 | 예시 |
|------|------|------|
| **bootstrap.servers** | Kafka 브로커 주소 | `'broker:9092'` |
| **group.id** | Consumer Group ID | `'order-group'` |
| **auto.offset.reset** | 시작 위치 | `'earliest'` or `'latest'` |

### Signal Handling 패턴 (필수!)

```python
import signal

# 1. 플래그
running = True

# 2. Handler
def signal_handler(sig, frame):
    global running
    running = False

# 3. 등록
signal.signal(signal.SIGINT, signal_handler)

# 4. 루프
while running:
    msg = consumer.poll(timeout=1.0)
    # 처리

# 5. 종료
consumer.close()
```

### JSON 역직렬화

```python
# bytes → dict (한 줄)
order = json.loads(msg.value().decode('utf-8'))
```

### Consumer Group 핵심

- 같은 `group.id`: 파티션 **분담**
- 다른 `group.id`: 독립적으로 **모두 읽음**
- Consumer 수 ≤ 파티션 수 (권장)
- Rebalancing: Consumer 추가/제거 시 자동 재분배

### poll() vs close()

| 메서드 | 역할 | 호출 시점 |
|--------|------|----------|
| **poll()** | 메시지 1개 읽기 | 무한 루프 안 |
| **close()** | Consumer 종료 | 프로그램 종료 전 |

---

## 🎯 다음 교시 예고

**4교시: Flask 동기 → 비동기 전환**
- Day06의 Flask 동기 시스템 문제점 재확인
- Kafka로 전환한 비동기 시스템 구축
- 성능 측정: 4초 → 0.05초 (80배 향상!)
- 장애 격리 체험

이제 Consumer의 핵심을 마스터했습니다.
특히 Signal handling은 실무에서 **필수**이니 꼭 기억하세요!

---


# Day 07 - 3교시: confluent-kafka로 비동기 시스템 구축하기
> Python에서 Kafka를 활용하여 동기식 시스템을 비동기식으로 전환하는 실습

## 🎯 학습 목표
이 파트를 마치면 다음을 할 수 있습니다:

- **confluent-kafka-python**으로 Producer/Consumer를 구현할 수 있다
- **jq**를 활용하여 JSON 응답을 깔끔하게 처리할 수 있다
- 동기식 → 비동기식 전환의 효과를 직접 체험하고 설명할 수 있다
- Kafka 도입의 장점과 한계를 이해한다

---

## 📚 전체 학습 흐름

| 순서 | 내용 | 시간 |
|------|------|------|
| 1 | 포트 매핑 복습 & jq 도구 소개 | 10분 |
| 2 | confluent-kafka-python 소개 | 15분 |
| 3 | Producer/Consumer 기본 문법 | 20분 |
| 4 | 동기식 → 비동기식 전환 실습 | 40분 |
| 5 | 비교 실습 및 정리 | 20분 |

---
## 1. 포트 매핑 복습

### 왜 포트 매핑이 필요한가?

Docker 컨테이너는 호스트와 **별도의 네트워크 공간**을 가짐:
- 호스트의 `localhost:9092` ≠ 컨테이너의 `localhost:9092`
- 포트 매핑 없이는 호스트에서 컨테이너 내부 서비스에 접근 불가

### 포트 매핑 동작 원리

```
┌─────────────────────────────────────────────────────────────────────────┐
│                                                                         │
│  포트 매핑: -p 9092:9092                                                 │
│  ─────────────────────────────                                          │
│                                                                         │
│   호스트                           컨테이너                              │
│   ┌──────────────┐                ┌──────────────┐                      │
│   │              │      ✅        │              │                      │
│   │  :9092 ──────┼───────────────▶│  :9092 Kafka │                      │
│   │  (열림)       │   포트 포워딩     │              │                      │
│   └──────────────┘                └──────────────┘                      │
│                                                                         │
│   이제 호스트에서 localhost:9092 → 컨테이너 Kafka에 도달!                  │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

### compose.yml에서의 포트 매핑

```yaml
services:
  broker:
    image: apache/kafka:latest
    ports:
      - "9092:9092"    # 호스트:컨테이너
```

> 💡 **핵심**: 포트 매핑 덕분에 호스트의 Python 애플리케이션이 컨테이너의 Kafka에 연결 가능!

---
## 2. jq 설치 및 사용법

### jq란?

> **jq**: 커맨드라인에서 JSON 데이터를 파싱하고 가공하는 도구

- JSON을 예쁘게 포맷팅
- 특정 필드만 추출
- **유니코드(한글) 깨짐 문제 해결**

> 📖 **공식 문서**: [jq Manual](https://stedolan.github.io/jq/manual/) 참고

### WSL/Ubuntu에서 설치

```bash
# jq 설치
sudo apt-get update && sudo apt-get install -y jq

# 설치 확인
jq --version
```

### 기본 사용법

| 명령어 | 설명 | 예시 |
|--------|------|------|
| `jq .` | JSON 전체를 예쁘게 출력 | `echo '{"a":1}' \| jq .` |
| `jq '.필드명'` | 특정 필드 추출 | `echo '{"name":"홍길동"}' \| jq '.name'` |
| `jq -r` | 따옴표 없이 raw 출력 (유니코드 정상 출력) | `echo '{"name":"홍길동"}' \| jq -r '.name'` |
| `jq '.[]'` | 배열의 각 요소 출력 | `echo '[1,2,3]' \| jq '.[]'` |

### 유니코드 깨짐 해결

**문제 상황**: curl로 API 호출 시 한글이 `\uD55C\uAE00` 형태로 출력

```bash
# 유니코드가 깨진 출력
curl -s http://localhost:5000/order
# 출력: {"message": "\uc8fc\ubb38\uc774 \uc811\uc218\ub418\uc5c8\uc2b5\ub2c8\ub2e4"}

# jq로 해결 (예쁘게 포맷팅)
curl -s http://localhost:5000/order | jq .
# 출력:
# {
#   "message": "주문이 접수되었습니다"
# }

# 특정 필드만 추출 (-r: 따옴표 제거)
curl -s http://localhost:5000/order | jq -r '.message'
# 출력: 주문이 접수되었습니다
```

---
## 3. confluent-kafka-python 소개

### 왜 confluent-kafka인가?

| 라이브러리 | 특징 | 성능 |
|-----------|------|------|
| kafka-python | 순수 Python, 설치 쉬움 | 느림 |
| **confluent-kafka** | librdkafka 기반 (C 라이브러리) | **빠름** (10배 이상) |
| aiokafka | 비동기 (asyncio) 지원 | 중간 |

**confluent-kafka 선택 이유**:
- Confluent 공식 지원 (Kafka 창시자 회사)
- 프로덕션 검증된 성능
- 풍부한 기능 (트랜잭션, 정확히 한 번 전송 등)

> 📖 **공식 문서**: [confluent-kafka-python](https://docs.confluent.io/platform/current/clients/confluent-kafka-python/html/index.html) 참고

### 아키텍처 개요

```
┌─────────────────────────────────────────────────────────────────────────┐
│                        Kafka 기반 비동기 아키텍처                          │
│                                                                         │
│   ┌──────────┐         ┌──────────────────┐         ┌──────────────┐   │
│   │  Order   │         │      Kafka       │         │  Consumers   │   │
│   │ Service  │         │                  │         │              │   │
│   │          │  발행    │  ┌────────────┐  │  구독    │ [Inventory]  │   │
│   │ Producer │───────▶ │  │   orders   │  │  ◀──────│ [Shipping]   │   │
│   │          │         │  │   topic    │  │         │ [Notification]│   │
│   └──────────┘         │  └────────────┘  │         └──────────────┘   │
│        │               └──────────────────┘               │            │
│        │                                                  │            │
│   즉시 응답!                                         독립적으로 처리      │
│   (0.05초)                                          (각자 속도대로)      │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

### 설치 방법

```bash
# pip으로 설치
pip install confluent-kafka

# 또는 Docker 이미지에 포함 (Dockerfile에서)
RUN pip install confluent-kafka
```

---
## 4. Producer 기본 문법

### Producer 설정 옵션

| 설정 | 설명 | 권장값 |
|------|------|--------|
| `bootstrap.servers` | Kafka 브로커 주소 | `localhost:9092` |
| `client.id` | 클라이언트 식별자 (디버깅용) | `my-producer` |
| `acks` | 메시지 전송 확인 수준 | `all` (가장 안전) |
| `retries` | 전송 실패 시 재시도 횟수 | `3` |
| `linger.ms` | 배치 전송 대기 시간 | `0` (즉시 전송) |

### Producer 기본 코드

In [ ]:
# confluent-kafka Producer 기본 예제
from confluent_kafka import Producer
import json

# ------------------------------------------------------------
# 1. Producer 설정
# ------------------------------------------------------------
config = {
    "bootstrap.servers": "localhost:9092",  # Kafka 브로커 주소
    "client.id": "my-producer",  # 클라이언트 식별자
    "acks": "all",  # 모든 ISR이 복제 완료해야 성공
}

# Producer 인스턴스 생성
producer = Producer(config)


# ------------------------------------------------------------
# 2. 전송 완료 콜백 함수
# ------------------------------------------------------------
def delivery_callback(err, msg):
    """메시지 전송 결과를 처리하는 콜백 함수"""
    if err is not None:
        print(f"❌ 전송 실패: {err}")
    else:
        print(f"✅ 전송 성공: {msg.topic()} [{msg.partition()}] @ {msg.offset()}")


# ------------------------------------------------------------
# 3. 메시지 전송
# ------------------------------------------------------------
order_data = {
    "order_id": "abc123",
    "product": "노트북",
    "quantity": 1,
    "customer": "홍길동",
}

# produce(): 메시지를 전송 큐에 추가 (비동기)
producer.produce(
    topic="orders",
    key=order_data["order_id"].encode("utf-8"),  # 같은 키 → 같은 파티션
    value=json.dumps(order_data).encode("utf-8"),  # JSON → bytes
    callback=delivery_callback,
)

# ------------------------------------------------------------
# 4. 전송 완료 대기
# ------------------------------------------------------------
# flush(): 큐의 모든 메시지가 전송될 때까지 대기
producer.flush(timeout=10)

### Producer 주요 메서드

| 메서드 | 설명 |
|--------|------|
| `produce(topic, value, key, callback)` | 메시지를 전송 큐에 추가 (비동기) |
| `poll(timeout)` | 콜백 함수 처리, timeout=0이면 즉시 반환 |
| `flush(timeout)` | 큐의 모든 메시지 전송 완료 대기 |

---
## 5. Consumer 기본 문법

### Consumer 설정 옵션

| 설정 | 설명 | 권장값 |
|------|------|--------|
| `bootstrap.servers` | Kafka 브로커 주소 | `localhost:9092` |
| `group.id` | Consumer Group ID (필수!) | 서비스별로 다르게 |
| `auto.offset.reset` | 오프셋 없을 때 시작 위치 | `earliest` 또는 `latest` |
| `enable.auto.commit` | 오프셋 자동 커밋 여부 | `True` |
| `session.timeout.ms` | Consumer 죽음 판단 시간 | `30000` (30초) |

### Consumer 기본 코드

In [ ]:
# confluent-kafka Consumer 기본 예제
from confluent_kafka import Consumer, KafkaError
import json

# ------------------------------------------------------------
# 1. Consumer 설정
# ------------------------------------------------------------
config = {
    "bootstrap.servers": "localhost:9092",
    "group.id": "order-processors",  # Consumer Group ID (필수)
    "auto.offset.reset": "earliest",  # 처음부터 읽기
    "enable.auto.commit": True,
    "auto.commit.interval.ms": 5000,  # 5초마다 커밋
    "session.timeout.ms": 30000,
}

# Consumer 인스턴스 생성
consumer = Consumer(config)

# ------------------------------------------------------------
# 2. 토픽 구독
# ------------------------------------------------------------
consumer.subscribe(["orders"])

# ------------------------------------------------------------
# 3. 메시지 폴링
# ------------------------------------------------------------
try:
    while True:
        # poll(): 메시지 하나를 가져옴
        msg = consumer.poll(timeout=1.0)

        if msg is None:
            continue

        # 에러 체크
        if msg.error():
            if msg.error().code() == KafkaError._PARTITION_EOF:
                print(f"파티션 끝 도달: {msg.topic()}[{msg.partition()}]")
            else:
                print(f"에러 발생: {msg.error()}")
            continue

        # 메시지 처리
        key = msg.key().decode("utf-8") if msg.key() else None
        value = json.loads(msg.value().decode("utf-8"))

        print(f"📨 메시지 수신:")
        print(f"   토픽: {msg.topic()}")
        print(f"   파티션: {msg.partition()}")
        print(f"   오프셋: {msg.offset()}")
        print(f"   값: {value}")

except KeyboardInterrupt:
    print("종료 요청...")

finally:
    # ------------------------------------------------------------
    # 4. 정리
    # ------------------------------------------------------------
    consumer.close()

### Consumer 주요 메서드

| 메서드 | 설명 |
|--------|------|
| `subscribe(topics)` | 토픽 구독 (리스트로 여러 개 가능) |
| `poll(timeout)` | 메시지 하나를 가져옴, timeout 동안 대기 |
| `commit()` | 현재까지 처리한 오프셋을 수동 커밋 |
| `close()` | Consumer 종료 및 리소스 정리 |

---
## 6. 동기식 → 비동기식 전환 실습

### 6.1 폴더 구조 준비

```bash
mkdir -p kafka-migration/sync-version/{order,inventory,shipping,notification}
mkdir -p kafka-migration/async-version/{order,inventory,shipping,notification}
cd kafka-migration
```

**최종 구조**:
```
kafka-migration/
├── sync-version/          # 동기식 버전 (기존)
│   ├── compose.yml
│   ├── order/
│   ├── inventory/
│   ├── shipping/
│   └── notification/
│
└── async-version/         # 비동기식 버전 (Kafka)
    ├── compose.yml
    ├── order/
    ├── inventory/
    ├── shipping/
    └── notification/
```

### 6.2 동기식 버전 (문제 상황 체험)

**동기식의 문제점**:
- 모든 서비스를 순차적으로 호출 (직렬 처리)
- 하나라도 느리면 전체가 느려짐
- 하나라도 실패하면 전체 실패

#### sync-version/order/app.py

```python
# ============================================================
# 동기식 주문 서비스
# 문제점: 모든 서비스 순차 호출 → 느리고 장애 전파
# ============================================================
from flask import Flask, jsonify
import requests
import time
import uuid

app = Flask(__name__)

@app.route('/order', methods=['POST'])
def create_order():
    start = time.time()
    order_id = str(uuid.uuid4())[:8]

    print(f"\n{'='*50}")
    print(f"🛒 [동기식] 주문 접수: {order_id}")
    print(f"{'='*50}")

    # Step 1: 재고 확인 (0.5초)
    print("📞 [1/3] 재고 서비스 호출 중...")
    try:
        response = requests.get(
            "http://inventory:5001/check",
            params={"order_id": order_id},
            timeout=5
        )
        print(f"   ✅ 재고 확인 완료: {response.text}")
    except requests.exceptions.RequestException as e:
        print(f"   ❌ 재고 서비스 실패: {e}")
        return jsonify({"error": "재고 서비스 연결 실패"}), 500

    # Step 2: 배송 예약 (3초) - 느림!
    print("📞 [2/3] 배송 서비스 호출 중... (오래 걸림)")
    try:
        response = requests.get(
            "http://shipping:5002/schedule",
            params={"order_id": order_id},
            timeout=10
        )
        print(f"   ✅ 배송 예약 완료: {response.text}")
    except requests.exceptions.RequestException as e:
        print(f"   ❌ 배송 서비스 실패: {e}")
        return jsonify({"error": "배송 서비스 연결 실패"}), 500

    # Step 3: 알림 발송 (0.3초)
    print("📞 [3/3] 알림 서비스 호출 중...")
    try:
        response = requests.get(
            "http://notification:5003/send",
            params={"order_id": order_id},
            timeout=5
        )
        print(f"   ✅ 알림 발송 완료: {response.text}")
    except requests.exceptions.RequestException as e:
        print(f"   ❌ 알림 서비스 실패: {e}")
        return jsonify({"error": "알림 서비스 연결 실패"}), 500

    elapsed = time.time() - start
    print(f"\n⏱️ 총 처리 시간: {elapsed:.2f}초")

    return jsonify({
        "status": "completed",
        "order_id": order_id,
        "elapsed_seconds": round(elapsed, 2),
        "message": f"주문 완료! (처리 시간: {elapsed:.1f}초)"
    })

if __name__ == '__main__':
    print("🚀 [동기식] 주문 서비스 시작 (포트: 5000)")
    app.run(host='0.0.0.0', port=5000)
```

#### sync-version/inventory/app.py

```python
# 재고 서비스 (처리 시간: 0.5초)
from flask import Flask, request
import time

app = Flask(__name__)

@app.route('/check')
def check_inventory():
    order_id = request.args.get('order_id', 'unknown')
    print(f"📦 재고 확인 중... (주문: {order_id})")
    time.sleep(0.5)  # DB 조회 시뮬레이션
    print(f"📦 재고 확인 완료! (주문: {order_id})")
    return f"재고 OK (주문: {order_id})"

if __name__ == '__main__':
    print("📦 재고 서비스 시작 (포트: 5001)")
    app.run(host='0.0.0.0', port=5001)
```

#### sync-version/shipping/app.py

```python
# 배송 서비스 (처리 시간: 3초 - 의도적으로 느림!)
from flask import Flask, request
import time

app = Flask(__name__)

@app.route('/schedule')
def schedule_shipping():
    order_id = request.args.get('order_id', 'unknown')
    print(f"🚚 배송 예약 중... (주문: {order_id}) - 3초 소요")
    time.sleep(3)  # 외부 배송사 API 시뮬레이션 (느림!)
    print(f"🚚 배송 예약 완료! (주문: {order_id})")
    return f"배송 OK (주문: {order_id})"

if __name__ == '__main__':
    print("🚚 배송 서비스 시작 (포트: 5002)")
    app.run(host='0.0.0.0', port=5002)
```

#### sync-version/notification/app.py

```python
# 알림 서비스 (처리 시간: 0.3초)
from flask import Flask, request
import time

app = Flask(__name__)

@app.route('/send')
def send_notification():
    order_id = request.args.get('order_id', 'unknown')
    print(f"📱 알림 발송 중... (주문: {order_id})")
    time.sleep(0.3)  # SMS/푸시 발송 시뮬레이션
    print(f"📱 알림 발송 완료! (주문: {order_id})")
    return f"알림 OK (주문: {order_id})"

if __name__ == '__main__':
    print("📱 알림 서비스 시작 (포트: 5003)")
    app.run(host='0.0.0.0', port=5003)
```

#### sync-version/Dockerfile (모든 서비스 공통)

```dockerfile
FROM python:3.9-slim
WORKDIR /app
RUN pip install flask requests
COPY app.py .
CMD ["python", "app.py"]
```

#### sync-version/compose.yml

```yaml
# ============================================================
# 동기식 주문 시스템
# 문제: 모든 서비스 순차 호출 → 총 처리 시간 ~4초
# ============================================================

services:
  order:
    build: ./order
    ports:
      - "5000:5000"
    depends_on:
      - inventory
      - shipping
      - notification
    environment:
      - PYTHONUNBUFFERED=1

  inventory:
    build: ./inventory
    environment:
      - PYTHONUNBUFFERED=1

  shipping:
    build: ./shipping
    environment:
      - PYTHONUNBUFFERED=1

  notification:
    build: ./notification
    environment:
      - PYTHONUNBUFFERED=1
```

> 💡 **PYTHONUNBUFFERED=1**: Python의 print 출력이 버퍼링 없이 즉시 로그에 표시됨

#### 동기식 버전 실행 및 테스트

```bash
cd sync-version

# 빌드 및 실행
docker compose up --build -d

# 주문 테스트 (jq로 결과 확인)
curl -s -X POST http://localhost:5000/order | jq .

# 로그 확인
docker compose logs -f order

# 정리
docker compose down
```

**예상 결과** (~4초 소요):

```json
{
  "elapsed_seconds": 3.99,
  "message": "주문 완료! (처리 시간: 4.0초)",
  "order_id": "07336704",
  "status": "completed"
}
```

**order 서비스 로그 예시**:

```
==================================================
[동기식] 주문 접수: 07336704
==================================================
[1/3] 재고 서비스 호출 중...
   재고 확인 완료: 재고 OK (주문: 07336704)
[2/3] 배송 서비스 호출 중... (오래 걸림)
   배송 예약 완료: 배송 OK (주문: 07336704)
[3/3] 알림 서비스 호출 중...
   알림 발송 완료: 알림 OK (주문: 07336704)

총 처리 시간: 3.99초
```

### 6.3 비동기식 버전 (Kafka 적용)

**비동기식의 핵심 변화**:
- 주문 서비스는 Kafka에 메시지 발행 후 **즉시 응답**
- 각 Consumer는 **독립적**으로 자기 속도대로 처리
- 하나가 느려도/죽어도 다른 서비스에 영향 없음

#### async-version/order/app.py (Producer)

```python
# ============================================================
# 비동기식 주문 서비스 (Kafka Producer)
# 핵심: HTTP 호출 대신 Kafka 메시지 발행 → 즉시 응답
# ============================================================
from flask import Flask, jsonify, request
from confluent_kafka import Producer
import json
import time
import uuid

app = Flask(__name__)

# Kafka Producer 설정
kafka_config = {
    'bootstrap.servers': 'kafka:9092',
    'client.id': 'order-service-producer',
    'acks': 'all',
    'retries': 3,
    'retry.backoff.ms': 100,
    'linger.ms': 0,  # 즉시 전송
}

producer = Producer(kafka_config)

def delivery_report(err, msg):
    if err is not None:
        print(f'❌ 메시지 전송 실패: {err}')
    else:
        print(f'✅ 메시지 전송 성공: partition={msg.partition()}, offset={msg.offset()}')

@app.route('/order', methods=['POST'])
def create_order():
    start = time.time()
    order_id = str(uuid.uuid4())[:8]

    print(f"\n{'='*50}")
    print(f"🛒 [비동기식] 주문 접수: {order_id}")
    print(f"{'='*50}")

    # 주문 데이터 준비
    order_data = {
        "order_id": order_id,
        "timestamp": time.time(),
        "status": "received",
        "product": request.json.get('product', '샘플 상품') if request.is_json else '샘플 상품',
        "quantity": request.json.get('quantity', 1) if request.is_json else 1,
    }

    # Kafka로 메시지 발행
    try:
        print(f"📤 Kafka로 메시지 발행 중...")
        producer.produce(
            topic='orders',
            key=order_id.encode('utf-8'),
            value=json.dumps(order_data).encode('utf-8'),
            callback=delivery_report
        )
        producer.poll(0)
        producer.flush(timeout=5)
        print(f"✅ 메시지 발행 완료!")
    except Exception as e:
        print(f"❌ Kafka 발행 실패: {e}")
        return jsonify({"status": "error", "message": "주문 접수 실패"}), 500

    elapsed = time.time() - start
    print(f"\n⏱️ API 응답 시간: {elapsed:.3f}초")

    # 핵심: "completed"가 아니라 "accepted"
    return jsonify({
        "status": "accepted",
        "order_id": order_id,
        "elapsed_seconds": round(elapsed, 3),
        "message": f"주문이 접수되었습니다. (처리 시간: {elapsed:.3f}초)",
        "note": "실제 처리는 백그라운드에서 진행됩니다."
    })

@app.route('/health')
def health_check():
    return jsonify({"status": "healthy", "service": "order"})

if __name__ == '__main__':
    print("🚀 [비동기식] 주문 서비스 시작 (포트: 5000)")
    app.run(host='0.0.0.0', port=5000)
```

#### async-version/inventory/app.py (Consumer)

```python
# ============================================================
# 재고 서비스 (Kafka Consumer)
# 독립적으로 동작, 자기 속도대로 처리
# ============================================================
from confluent_kafka import Consumer, KafkaError
import json
import time
import signal

kafka_config = {
    'bootstrap.servers': 'kafka:9092',
    'group.id': 'inventory-service-group',
    'client.id': 'inventory-consumer-1',
    'auto.offset.reset': 'earliest',
    'enable.auto.commit': True,
    'session.timeout.ms': 30000,
}

consumer = Consumer(kafka_config)
running = True

def signal_handler(signum, frame):
    global running
    print("\n🛑 종료 신호 수신...")
    running = False

signal.signal(signal.SIGINT, signal_handler)
signal.signal(signal.SIGTERM, signal_handler)

def process_order(order_data):
    order_id = order_data.get('order_id', 'unknown')
    print(f"📦 재고 확인 시작: 주문={order_id}")
    time.sleep(0.5)  # DB 조회 시뮬레이션
    print(f"📦 ✅ 재고 확인 완료: 주문={order_id}")

def main():
    print("="*60)
    print("📦 재고 서비스 (Kafka Consumer) 시작")
    print("="*60)

    consumer.subscribe(['orders'])

    try:
        while running:
            msg = consumer.poll(timeout=1.0)
            if msg is None:
                continue
            if msg.error():
                if msg.error().code() != KafkaError._PARTITION_EOF:
                    print(f"❌ Consumer 에러: {msg.error()}")
                continue

            value = json.loads(msg.value().decode('utf-8'))
            print(f"\n📨 메시지 수신: offset={msg.offset()}")
            process_order(value)
    finally:
        consumer.close()
        print("✅ Consumer 종료 완료")

if __name__ == '__main__':
    main()
```

#### async-version/shipping/app.py (Consumer - 느린 서비스)

```python
# ============================================================
# 배송 서비스 (Kafka Consumer) - 3초 걸리는 느린 서비스
# 핵심: 이 서비스가 느려도 주문 서비스와 다른 Consumer에 영향 없음!
# ============================================================
from confluent_kafka import Consumer, KafkaError
import json
import time
import signal

kafka_config = {
    'bootstrap.servers': 'kafka:9092',
    'group.id': 'shipping-service-group',
    'client.id': 'shipping-consumer-1',
    'auto.offset.reset': 'earliest',
    'enable.auto.commit': True,
    'session.timeout.ms': 30000,
}

consumer = Consumer(kafka_config)
running = True

def signal_handler(signum, frame):
    global running
    print("\n🛑 종료 신호 수신...")
    running = False

signal.signal(signal.SIGINT, signal_handler)
signal.signal(signal.SIGTERM, signal_handler)

def process_order(order_data):
    order_id = order_data.get('order_id', 'unknown')
    print(f"🚚 배송 예약 시작: 주문={order_id}")
    print(f"🚚 ⏳ 외부 배송사 API 호출 중... (3초 소요)")
    time.sleep(3)  # 느린 외부 API 시뮬레이션
    print(f"🚚 ✅ 배송 예약 완료: 주문={order_id}")

def main():
    print("="*60)
    print("🚚 배송 서비스 (Kafka Consumer) 시작")
    print("⚠️  주의: 메시지당 3초 소요")
    print("="*60)

    consumer.subscribe(['orders'])

    try:
        while running:
            msg = consumer.poll(timeout=1.0)
            if msg is None:
                continue
            if msg.error():
                if msg.error().code() != KafkaError._PARTITION_EOF:
                    print(f"❌ Consumer 에러: {msg.error()}")
                continue

            value = json.loads(msg.value().decode('utf-8'))
            print(f"\n📨 메시지 수신: offset={msg.offset()}")
            process_order(value)
    finally:
        consumer.close()
        print("✅ Consumer 종료 완료")

if __name__ == '__main__':
    main()
```

#### async-version/notification/app.py (Consumer)

```python
# ============================================================
# 알림 서비스 (Kafka Consumer)
# 빠른 처리 (0.3초), 실패해도 주문 자체는 성공
# ============================================================
from confluent_kafka import Consumer, KafkaError
import json
import time
import signal

kafka_config = {
    'bootstrap.servers': 'kafka:9092',
    'group.id': 'notification-service-group',
    'client.id': 'notification-consumer-1',
    'auto.offset.reset': 'earliest',
    'enable.auto.commit': True,
    'session.timeout.ms': 30000,
}

consumer = Consumer(kafka_config)
running = True

def signal_handler(signum, frame):
    global running
    print("\n🛑 종료 신호 수신...")
    running = False

signal.signal(signal.SIGINT, signal_handler)
signal.signal(signal.SIGTERM, signal_handler)

def process_order(order_data):
    order_id = order_data.get('order_id', 'unknown')
    print(f"📱 알림 발송 시작: 주문={order_id}")
    time.sleep(0.3)  # SMS/푸시 발송 시뮬레이션
    print(f"📱 ✅ 알림 발송 완료: 주문={order_id}")

def main():
    print("="*60)
    print("📱 알림 서비스 (Kafka Consumer) 시작")
    print("="*60)

    consumer.subscribe(['orders'])

    try:
        while running:
            msg = consumer.poll(timeout=1.0)
            if msg is None:
                continue
            if msg.error():
                if msg.error().code() != KafkaError._PARTITION_EOF:
                    print(f"❌ Consumer 에러: {msg.error()}")
                continue

            value = json.loads(msg.value().decode('utf-8'))
            print(f"\n📨 메시지 수신: offset={msg.offset()}")
            process_order(value)
    finally:
        consumer.close()
        print("✅ Consumer 종료 완료")

if __name__ == '__main__':
    main()
```

#### async-version/Dockerfile (모든 서비스 공통)

```dockerfile
FROM python:3.9-slim
WORKDIR /app

# confluent-kafka 설치를 위한 시스템 의존성
RUN apt-get update && apt-get install -y \
    gcc \
    librdkafka-dev \
    && rm -rf /var/lib/apt/lists/*

# Python 패키지 설치
RUN pip install flask confluent-kafka

COPY app.py .
CMD ["python", "-u", "app.py"]
```

#### async-version/compose.yml

```yaml
# ============================================================
# 비동기식 주문 시스템 (Kafka 기반)
#
# 핵심 변화:
# - order 서비스: Kafka에 발행 후 즉시 응답 (~0.05초)
# - 각 Consumer: 독립적으로 자기 속도대로 처리
# ============================================================

services:
  # Kafka 브로커 (KRaft 모드)
  kafka:
    image: apache/kafka:latest
    container_name: kafka-broker
    ports:
      - "9092:9092"
    environment:
      KAFKA_NODE_ID: 1
      KAFKA_PROCESS_ROLES: broker,controller
      KAFKA_CONTROLLER_QUORUM_VOTERS: 1@kafka:9093
      KAFKA_LISTENERS: PLAINTEXT://0.0.0.0:9092,CONTROLLER://0.0.0.0:9093
      KAFKA_ADVERTISED_LISTENERS: PLAINTEXT://kafka:9092
      KAFKA_LISTENER_SECURITY_PROTOCOL_MAP: PLAINTEXT:PLAINTEXT,CONTROLLER:PLAINTEXT
      KAFKA_CONTROLLER_LISTENER_NAMES: CONTROLLER
      KAFKA_INTER_BROKER_LISTENER_NAME: PLAINTEXT
      KAFKA_AUTO_CREATE_TOPICS_ENABLE: "true"
      KAFKA_NUM_PARTITIONS: 3
      KAFKA_DEFAULT_REPLICATION_FACTOR: 1
      KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR: 1
      KAFKA_TRANSACTION_STATE_LOG_REPLICATION_FACTOR: 1
      KAFKA_TRANSACTION_STATE_LOG_MIN_ISR: 1
    healthcheck:
      test: ["CMD-SHELL", "/opt/kafka/bin/kafka-broker-api-versions.sh --bootstrap-server localhost:9092 || exit 1"]
      interval: 10s
      timeout: 10s
      retries: 5
      start_period: 30s

  # 주문 서비스 (Producer)
  order:
    build: ./order
    container_name: order-service
    ports:
      - "5000:5000"
    depends_on:
      kafka:
        condition: service_healthy
    environment:
      - KAFKA_BOOTSTRAP_SERVERS=kafka:9092
      - PYTHONUNBUFFERED=1

  # 재고 서비스 (Consumer)
  inventory:
    build: ./inventory
    container_name: inventory-consumer
    depends_on:
      kafka:
        condition: service_healthy
    environment:
      - KAFKA_BOOTSTRAP_SERVERS=kafka:9092
      - PYTHONUNBUFFERED=1

  # 배송 서비스 (Consumer) - 느림!
  shipping:
    build: ./shipping
    container_name: shipping-consumer
    depends_on:
      kafka:
        condition: service_healthy
    environment:
      - KAFKA_BOOTSTRAP_SERVERS=kafka:9092
      - PYTHONUNBUFFERED=1

  # 알림 서비스 (Consumer)
  notification:
    build: ./notification
    container_name: notification-consumer
    depends_on:
      kafka:
        condition: service_healthy
    environment:
      - KAFKA_BOOTSTRAP_SERVERS=kafka:9092
      - PYTHONUNBUFFERED=1

  # Kafka UI (모니터링용)
  kafka-ui:
    image: provectuslabs/kafka-ui:latest
    container_name: kafka-ui
    ports:
      - "8080:8080"
    depends_on:
      kafka:
        condition: service_healthy
    environment:
      KAFKA_CLUSTERS_0_NAME: local
      KAFKA_CLUSTERS_0_BOOTSTRAPSERVERS: kafka:9092
```

#### 비동기식 버전 실행 및 테스트

```bash
cd async-version

# 빌드 및 실행 (Kafka 준비에 시간 소요)
docker compose up --build -d

# 주문 테스트 (jq로 결과 확인)
curl -s -X POST http://localhost:5000/order | jq .

# 각 서비스 로그 확인
docker compose logs -f order
docker compose logs -f inventory
docker compose logs -f shipping
docker compose logs -f notification

# Kafka UI 접속: http://localhost:8080

# 정리
docker compose down
```

**예상 결과** (~0.01초 소요):

```json
{
  "elapsed_seconds": 0.006,
  "message": "주문이 접수되었습니다. (처리 시간: 0.006초)",
  "note": "실제 처리는 백그라운드에서 진행됩니다.",
  "order_id": "4c49f7a3",
  "status": "accepted"
}
```

**order 서비스 로그 예시**:

```
==================================================
[비동기식] 주문 접수: 4c49f7a3
==================================================
Kafka로 메시지 발행 중...
메시지 전송 성공: partition=2, offset=0
메시지 발행 완료!

API 응답 시간: 0.006초
```

**Consumer 서비스 로그 예시**:

```
# inventory-consumer
메시지 수신: offset=0
[재고] 확인 시작: 주문=4c49f7a3
[재고] 확인 완료: 주문=4c49f7a3

# notification-consumer
메시지 수신: offset=0
[알림] 발송 시작: 주문=4c49f7a3
[알림] 발송 완료: 주문=4c49f7a3

# shipping-consumer (3초 소요)
메시지 수신: offset=0
[배송] 예약 시작: 주문=4c49f7a3
[배송] 외부 배송사 API 호출 중... (3초 소요)
[배송] 예약 완료: 주문=4c49f7a3
```

---
## 7. jq 실전 활용

### 동기식 vs 비동기식 응답 비교

```bash
# 동기식 응답 (jq로 예쁘게 출력)
curl -s -X POST http://localhost:5000/order | jq .
```

```json
{
  "elapsed_seconds": 3.99,
  "message": "주문 완료! (처리 시간: 4.0초)",
  "order_id": "07336704",
  "status": "completed"
}
```

```bash
# 비동기식 응답
curl -s -X POST http://localhost:5000/order | jq .
```

```json
{
  "elapsed_seconds": 0.006,
  "message": "주문이 접수되었습니다. (처리 시간: 0.006초)",
  "note": "실제 처리는 백그라운드에서 진행됩니다.",
  "order_id": "ef76173c",
  "status": "accepted"
}
```

### 특정 필드만 추출

```bash
# 응답 시간만 추출
curl -s -X POST http://localhost:5000/order | jq '.elapsed_seconds'

# 메시지만 추출 (따옴표 없이)
curl -s -X POST http://localhost:5000/order | jq -r '.message'

# 여러 필드 추출
curl -s -X POST http://localhost:5000/order | jq '{status, elapsed_seconds}'
```

### 시간 측정과 함께 사용

```bash
# time 명령어와 함께 사용
time curl -s -X POST http://localhost:5000/order | jq .
```

---
## 8. 비교 실습 및 정리

### 8.1 실습 A: 응답 시간 비교

```bash
# === 동기식 테스트 ===
cd sync-version
docker compose up -d
sleep 5

echo "=== 동기식 주문 테스트 ==="
time curl -s -X POST http://localhost:5000/order | jq .

docker compose down

# === 비동기식 테스트 ===
cd ../async-version
docker compose up -d
sleep 30  # Kafka 준비 대기

echo "=== 비동기식 주문 테스트 ==="
time curl -s -X POST http://localhost:5000/order | jq .

docker compose logs -f
```

### 8.2 결과 비교표

| 항목 | 동기식 | 비동기식 |
|------|--------|----------|
| **API 응답 시간** | ~4초 | ~0.05초 |
| **사용자 대기 시간** | 4초 | 즉시 |
| **응답 의미** | "모든 처리 완료" | "접수됨 (처리 중)" |
| **status 값** | `completed` | `accepted` |

### 8.3 실습 B: 장애 격리 비교

```bash
# === 동기식에서 배송 장애 ===
cd sync-version
docker compose up -d
docker compose stop shipping
curl -s -X POST http://localhost:5000/order | jq .
```

**동기식 결과** (전체 실패):

```json
{
  "error": "배송 서비스 연결 실패"
}
```

```bash
# === 비동기식에서 배송 장애 ===
cd ../async-version
docker compose up -d
sleep 30
docker compose stop shipping
curl -s -X POST http://localhost:5000/order | jq .
```

**비동기식 결과** (주문 접수 성공):

```json
{
  "elapsed_seconds": 0.007,
  "message": "주문이 접수되었습니다. (처리 시간: 0.007초)",
  "note": "실제 처리는 백그라운드에서 진행됩니다.",
  "order_id": "7d3661a8",
  "status": "accepted"
}
```

```bash
# 배송 서비스 복구
docker compose start shipping
docker compose logs -f shipping
```

**배송 서비스 복구 후 로그** (밀린 메시지 자동 처리):

```
============================================================
[배송] 서비스 (Kafka Consumer) 시작
주의: 메시지당 3초 소요
============================================================

메시지 수신: offset=2
[배송] 예약 시작: 주문=7d3661a8
[배송] 외부 배송사 API 호출 중... (3초 소요)
[배송] 예약 완료: 주문=7d3661a8
```

### 8.4 장애 격리 비교표

| 상황 | 동기식 | 비동기식 |
|------|--------|----------|
| **배송 장애 시 주문** | ❌ 전체 실패 | ✅ 접수 성공 |
| **장애 중 메시지** | 유실 | Kafka에 보관 |
| **복구 후** | 수동 재처리 필요 | 자동으로 처리 |

### 8.5 비동기식의 한계점

| 한계 | 설명 | 해결책 |
|------|------|--------|
| **"완료" 의미 변화** | API 응답 = "접수됨" (처리 완료 아님) | 별도 상태 조회 API, 웹소켓/SSE |
| **에러 핸들링 복잡** | Consumer 실패 시 클라이언트가 모름 | Dead Letter Queue, 보상 트랜잭션 |
| **순서 보장 어려움** | 재고/배송이 순서대로 처리 안될 수 있음 | 단일 Consumer, 이벤트 체이닝 |
| **트랜잭션 처리** | 부분 실패 가능 | Saga 패턴 |

### 8.6 언제 무엇을 사용할까?

**동기식이 적합한 경우**:
- 즉각적인 결과 확인 필요 (결제 승인, 로그인)
- 트랜잭션 무결성 중요
- 서비스 수가 적고 안정적

**비동기식(Kafka)이 적합한 경우**:
- 처리 결과를 나중에 확인해도 됨
- 높은 처리량 필요
- 서비스 간 느슨한 결합 필요
- 장애 격리 중요

> 💡 **실제 서비스에서는 혼합 사용!** 예: 결제(동기) + 배송/알림(비동기)

---
## 📚 FAQ

**Q1. confluent-kafka와 kafka-python 중 어떤 것을 사용해야 하나요?**

A1. 프로덕션 환경에서는 `confluent-kafka`를 권장합니다. librdkafka C 라이브러리 기반으로 10배 이상 빠르고, Confluent 공식 지원을 받습니다. `kafka-python`은 순수 Python으로 설치가 쉽지만 성능이 낮아 학습용으로만 적합합니다.

**Q2. `PYTHONUNBUFFERED=1`은 왜 필요한가요?**

A2. Python은 기본적으로 stdout을 버퍼링합니다. Docker 환경에서 `print()` 출력이 즉시 로그에 나타나지 않을 수 있습니다. `PYTHONUNBUFFERED=1`을 설정하면 버퍼링 없이 즉시 출력됩니다.

**Q3. Consumer의 `group.id`를 왜 서비스마다 다르게 설정하나요?**

A3. 같은 `group.id`를 가진 Consumer들은 메시지를 **분담**해서 처리합니다. 다른 `group.id`를 사용하면 **모든 Consumer가 모든 메시지**를 받습니다. 재고/배송/알림이 각각 모든 주문을 처리해야 하므로 서로 다른 그룹을 사용합니다.

**Q4. jq에서 `-r` 옵션은 언제 사용하나요?**

A4. 기본적으로 jq는 문자열을 따옴표로 감싸서 출력합니다. `-r` (raw output) 옵션을 사용하면 따옴표 없이 순수 문자열로 출력됩니다. 스크립트에서 값을 변수에 저장할 때 유용합니다.

**Q5. 비동기식에서 "주문 완료"를 어떻게 확인하나요?**

A5. 별도의 상태 조회 API (`GET /order/{id}/status`)를 만들거나, 웹소켓/SSE로 실시간 알림을 보내거나, 모든 처리 완료 후 별도 이벤트를 발행하는 방식을 사용합니다.

---
## 📝 퀴즈

### Q1. confluent-kafka의 특징으로 올바른 것은?

- A) 순수 Python으로 작성되어 설치가 간편하다
- B) asyncio를 기본 지원하여 비동기 처리에 최적화되어 있다
- C) librdkafka C 라이브러리 기반으로 고성능을 제공한다
- D) Apache 재단에서 공식 지원하는 라이브러리이다

<details>
<summary>정답 확인</summary>

**정답: C**

confluent-kafka는 librdkafka C 라이브러리를 기반으로 하여 kafka-python보다 10배 이상 빠른 성능을 제공합니다. Confluent(Kafka 창시자 회사)에서 공식 지원합니다.
</details>

---

### Q2. 다음 jq 명령어 중 한글이 정상 출력되는 것은?

- A) `curl http://api | cat`
- B) `curl http://api | jq`
- C) `curl http://api | python -c "import json; print(json.load(sys.stdin))"`
- D) `curl http://api | grep message`

<details>
<summary>정답 확인</summary>

**정답: B**

jq는 JSON을 파싱하면서 유니코드 이스케이프 시퀀스(`\uXXXX`)를 실제 문자로 변환합니다. 다른 방법들은 유니코드가 그대로 출력되거나 추가 처리가 필요합니다.
</details>

---

### Q3. Kafka Consumer에서 `group.id`의 역할은?

- A) 메시지의 키를 지정하여 파티션을 결정한다
- B) 같은 그룹의 Consumer들이 메시지를 분담 처리하도록 한다
- C) Producer와 Consumer를 연결하는 식별자이다
- D) 토픽의 파티션 수를 결정한다

<details>
<summary>정답 확인</summary>

**정답: B**

같은 `group.id`를 가진 Consumer들은 Consumer Group을 형성하여 파티션을 나눠서 처리합니다. 다른 그룹의 Consumer들은 같은 메시지를 각각 독립적으로 처리합니다.
</details>

---

### Q4. 비동기식 주문 시스템에서 `status: "accepted"` 응답의 의미는?

- A) 주문이 완료되어 배송이 시작되었다
- B) 모든 서비스(재고, 배송, 알림)가 처리를 완료했다
- C) 주문이 Kafka에 저장되었고, 백그라운드 처리가 진행 중이다
- D) 결제가 승인되어 재고가 차감되었다

<details>
<summary>정답 확인</summary>

**정답: C**

비동기식에서 `accepted`는 "주문 메시지가 Kafka에 저장됨"을 의미합니다. 실제 처리(재고 확인, 배송 예약, 알림)는 각 Consumer가 독립적으로 진행합니다.
</details>

---
## ✏️ 과제

### 과제 1: 동기식 vs 비동기식 응답 시간 비교 (난이도: ⭐)

1. 동기식 버전 실행 후 주문 API 호출, 응답 시간 기록
2. 비동기식 버전 실행 후 주문 API 호출, 응답 시간 기록
3. 두 결과를 비교하고 차이점 분석

<details>
<summary>💡 힌트</summary>

```bash
# 시간 측정과 jq 함께 사용
time curl -s -X POST http://localhost:5000/order | jq .
```
</details>

---

### 과제 2: 장애 격리 테스트 (난이도: ⭐⭐)

1. 비동기식 버전 실행
2. 배송 서비스 중지: `docker compose stop shipping`
3. 주문 3개 생성
4. 재고/알림 서비스 로그 확인 (처리 완료 여부)
5. 배송 서비스 재시작 후 밀린 메시지 처리 확인

<details>
<summary>💡 힌트</summary>

```bash
# 특정 서비스 로그만 확인
docker compose logs -f inventory
docker compose logs -f shipping
```
</details>

---

### 과제 3: jq 활용 스크립트 작성 (난이도: ⭐⭐)

연속 5개 주문을 생성하고, 각 응답의 `elapsed_seconds`만 추출하여 평균 계산

<details>
<summary>💡 힌트</summary>

```bash
# 반복문으로 주문 생성, 응답 시간만 추출
for i in {1..5}; do
  curl -s -X POST http://localhost:5000/order | jq '.elapsed_seconds'
done
```
</details>

---

### 보너스 과제: Kafka UI에서 메시지 확인 (난이도: ⭐⭐⭐)

1. 비동기식 버전 실행 후 `http://localhost:8080` 접속
2. `orders` 토픽 확인
3. 주문 생성 후 메시지 내용 확인
4. 각 Consumer Group의 오프셋 상태 확인
5. Consumer 하나를 중지했다 재시작하며 Lag 변화 관찰

---
## 🎯 핵심 요약

### 1. 포트 매핑
- Docker 컨테이너와 호스트는 별도 네트워크
- `-p 호스트:컨테이너`로 연결

### 2. jq 도구
- JSON 포맷팅 및 필드 추출
- 유니코드 깨짐 해결: `curl ... | jq .`

### 3. confluent-kafka
- librdkafka 기반 고성능 라이브러리
- Producer: `produce()` → `flush()`
- Consumer: `subscribe()` → `poll()` → `close()`

### 4. 동기식 vs 비동기식

| 항목 | 동기식 | 비동기식 |
|------|--------|----------|
| 응답 시간 | ~4초 | ~0.05초 |
| 응답 의미 | 완료 | 접수됨 |
| 장애 영향 | 전파 | 격리 |
| 메시지 보존 | 없음 | Kafka에 저장 |

### 5. 실제 적용
- 동기식: 결제, 로그인 등 즉각 확인 필요한 경우
- 비동기식: 배송, 알림 등 나중에 처리해도 되는 경우
- **실무에서는 혼합 사용!**

---


# 📝 실습 과제: 주문 알림 시스템 구축

> **소요 시간**: 30분
> **난이도**: ⭐⭐☆☆☆

## 🎯 학습 목표

이번 실습 과제의 목표:

- 지금까지 배운 Producer/Consumer 개념을 **직접 구현**하며 체화
- JSON 직렬화/역직렬화 과정을 손으로 작성하며 이해
- Signal handling으로 안전한 종료 구현
- 전체 메시지 흐름 파악 (Producer → Kafka → Consumer)

**핵심**: 강사의 도움 없이 스스로 구현해보는 것이 가장 중요합니다!

## 📋 과제 설명

### 시나리오: 간단한 주문 알림 시스템

온라인 쇼핑몰에서 새 주문이 들어오면, 주문 정보를 Kafka로 전송하고, 알림 서비스가 이를 받아 콘솔에 출력하는 시스템을 구축합니다.

```
[주문 접수 시스템 (Producer)]
         │
         │ JSON 주문 정보 전송
         ▼
   [Kafka - orders 토픽]
         │
         │ 주문 정보 읽기
         ▼
[알림 서비스 (Consumer)]
         │
         ▼
   [콘솔에 주문 출력]
```

## 📝 요구사항

### 1. Producer (order_producer.py)

**역할**: 주문 정보를 JSON 형태로 Kafka에 전송

**주문 정보 형식**:
```json
{
  "order_id": "ORD-001",
  "customer": "홍길동",
  "product": "노트북",
  "quantity": 1,
  "price": 1500000
}
```

**구현해야 할 기능**:
- Kafka Producer 설정 및 생성
- 주문 딕셔너리를 JSON 문자열로 변환
- JSON 문자열을 bytes로 인코딩
- `orders` 토픽으로 메시지 전송
- flush()로 전송 완료 대기
- 전송 완료 메시지 출력

### 2. Consumer (order_consumer.py)

**역할**: Kafka에서 주문 정보를 읽고 콘솔에 예쁘게 출력

**구현해야 할 기능**:
- Kafka Consumer 설정 및 생성
- `orders` 토픽 구독
- while 루프로 메시지 수신 (poll)
- 메시지 value를 decode하고 JSON 파싱
- 주문 정보를 보기 좋게 출력
- Signal handling으로 ctrl+C 시 깔끔하게 종료

## 💡 힌트

### Producer 힌트

```python
# 1단계: 필요한 라이브러리 임포트
import json
from confluent_kafka import Producer

# 2단계: Producer 설정 딕셔너리 생성
config = {
    'bootstrap.servers': '??',  # Kafka 브로커 주소는?
    'client.id': '??'            # Producer 이름은?
}

# 3단계: Producer 객체 생성
producer = Producer(??)

# 4단계: 주문 딕셔너리 생성
order = {
    "order_id": "ORD-001",
    "customer": "홍길동",
    "product": "노트북",
    "quantity": 1,
    "price": 1500000
}

# 5단계: 딕셔너리 → JSON 문자열
order_json = json.dumps(??, ensure_ascii=False)

# 6단계: JSON 문자열 → bytes
order_bytes = order_json.encode('??')  # 인코딩은?

# 7단계: Kafka로 전송
producer.produce(
    topic='??',      # 토픽 이름은?
    value=??         # 전송할 데이터는?
)

# 8단계: 전송 완료 대기
producer.??()  # flush 메서드 호출

# 9단계: 완료 메시지 출력
print(f"✅ 주문 전송 완료: {order['order_id']}")
```

### Consumer 힌트

```python
# 1단계: 필요한 라이브러리 임포트
import json
import signal
from confluent_kafka import Consumer

# 2단계: 종료 플래그 (Signal handling용)
running = True

def signal_handler(sig, frame):
    """ctrl+C 처리 함수"""
    global running
    print("\n🛑 종료 신호 수신. Consumer 종료 중...")
    running = ??  # True? False?

# 3단계: Signal 등록
signal.signal(signal.SIGINT, signal_handler)

# 4단계: Consumer 설정 딕셔너리
config = {
    'bootstrap.servers': '??',
    'group.id': '??',                    # Consumer Group ID는?
    'auto.offset.reset': '??'            # earliest? latest?
}

# 5단계: Consumer 생성 및 구독
consumer = Consumer(??)
consumer.subscribe(['??'])  # 구독할 토픽은?

print("📨 주문 대기 중...")

# 6단계: 메시지 수신 루프
while ??:  # running? True?
    msg = consumer.poll(timeout=1.0)

    if msg is None:
        continue
    if msg.error():
        print(f"❌ 에러: {msg.error()}")
        continue

    # 7단계: 메시지 처리
    # bytes → str
    order_str = msg.value().decode('??')

    # str → dict
    order = json.loads(??)

    # 8단계: 예쁘게 출력
    print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📦 새 주문 접수!
주문번호: {order['order_id']}
고객명: {order['customer']}
상품: {order['product']}
수량: {order['quantity']}개
금액: {order['price']:,}원
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    """)

# 9단계: 정리
consumer.close()
print("✅ Consumer 종료 완료")
```

## 🚀 실행 방법

### 1. 파일 생성

두 개의 Python 파일을 생성합니다:
- `order_producer.py`: Producer 코드
- `order_consumer.py`: Consumer 코드

### 2. Docker 환경 확인

```bash
# Kafka가 실행 중인지 확인
docker compose ps

# 실행 중이 아니라면 시작
docker compose up -d
```

### 3. Consumer 실행 (Terminal 1)

```bash
# Consumer를 먼저 실행하여 메시지를 기다립니다
docker compose run python-producer python order_consumer.py
```

### 4. Producer 실행 (Terminal 2)

```bash
# 다른 터미널에서 Producer를 실행하여 메시지를 전송합니다
docker compose run python-producer python order_producer.py

# 여러 번 실행하여 여러 주문을 전송해보세요!
docker compose run python-producer python order_producer.py
docker compose run python-producer python order_producer.py
```

### 5. Kafka UI에서 확인

브라우저에서 http://localhost:8080 접속하여 `orders` 토픽의 메시지를 확인해보세요!

## ✅ 체크포인트

다음 항목들을 모두 달성했는지 확인하세요:

- [ ] Producer가 주문 정보를 JSON 형태로 전송
- [ ] Consumer가 메시지를 읽고 예쁘게 출력
- [ ] ctrl+C로 Consumer가 깔끔하게 종료 (에러 없이!)
- [ ] Kafka UI에서 메시지 확인 가능
- [ ] 여러 주문을 전송했을 때 모두 Consumer에서 출력됨

**추가 도전 과제** (빨리 끝낸 학생들을 위해):
- 주문마다 랜덤한 order_id 생성하기 (uuid 라이브러리 사용)
- 여러 상품 중에서 랜덤으로 선택하기 (random.choice)
- delivery_callback 함수 추가하여 전송 확인 메시지 출력하기

---
## 🎓 솔루션

> **주의**: 먼저 스스로 구현해보고, 막히는 부분이 있을 때만 솔루션을 참고하세요!

### order_producer.py (솔루션)

In [ ]:
# order_producer.py - 주문 정보를 Kafka로 전송하는 Producer

import json
from confluent_kafka import Producer

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Step 1: Kafka Producer 설정
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

config = {
    'bootstrap.servers': 'broker:9092',  # Kafka 브로커 주소 (Docker 내부에서는 서비스 이름 사용)
    'client.id': 'order-producer'        # Producer 식별용 이름
}

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Step 2: Producer 객체 생성
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

producer = Producer(config)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Step 3: 주문 데이터 준비
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

order = {
    "order_id": "ORD-001",      # 주문 번호
    "customer": "홍길동",        # 고객 이름
    "product": "노트북",         # 상품명
    "quantity": 1,              # 수량
    "price": 1500000            # 가격 (원)
}

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Step 4: 딕셔너리 → JSON 문자열 → bytes 변환
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# ensure_ascii=False: 한글이 제대로 표시되도록 설정
order_json = json.dumps(order, ensure_ascii=False)

# Kafka는 bytes만 받으므로 UTF-8로 인코딩
order_bytes = order_json.encode('utf-8')

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Step 5: Kafka로 메시지 전송
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

producer.produce(
    topic='orders',      # 전송할 토픽 이름
    value=order_bytes    # 전송할 데이터 (bytes)
)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Step 6: 전송 완료 대기
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# flush(): 모든 메시지가 Kafka로 전송될 때까지 대기
# 이 메서드를 호출하지 않으면 메시지가 전송되지 않을 수 있음!
producer.flush()

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Step 7: 완료 메시지 출력
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print(f"✅ 주문 전송 완료: {order['order_id']}")
print(f"   - 고객: {order['customer']}")
print(f"   - 상품: {order['product']}")
print(f"   - 금액: {order['price']:,}원")

**실행 결과**:

```
✅ 주문 전송 완료: ORD-001
   - 고객: 홍길동
   - 상품: 노트북
   - 금액: 1,500,000원
```

### order_consumer.py (솔루션)

In [ ]:
# order_consumer.py - Kafka에서 주문 정보를 읽고 출력하는 Consumer

import json
import signal
from confluent_kafka import Consumer

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Step 1: Signal Handling 설정 (ctrl+C 처리)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

running = True  # 실행 상태를 나타내는 플래그

def signal_handler(sig, frame):
    """
    ctrl+C (SIGINT) 신호를 받으면 실행되는 함수

    역할:
    - running 플래그를 False로 변경
    - while 루프를 종료하고 consumer.close() 호출하도록 유도
    """
    global running
    print("\n🛑 종료 신호 수신 (ctrl+C). Consumer 종료 중...")
    running = False

# SIGINT (ctrl+C) 신호를 받으면 signal_handler 함수 실행
signal.signal(signal.SIGINT, signal_handler)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Step 2: Kafka Consumer 설정
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

config = {
    'bootstrap.servers': 'broker:9092',     # Kafka 브로커 주소
    'group.id': 'order-consumer-group',     # Consumer Group ID (같은 그룹끼리 파티션 분담)
    'auto.offset.reset': 'earliest'         # 처음부터 읽기 (CLI의 --from-beginning)
}

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Step 3: Consumer 생성 및 구독
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

consumer = Consumer(config)

# 'orders' 토픽 구독 (여러 토픽을 리스트로 지정 가능)
consumer.subscribe(['orders'])

print("📨 주문 대기 중... (ctrl+C로 종료)")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Step 4: 메시지 수신 루프
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

while running:  # running이 True인 동안 계속 실행
    # poll(): Kafka에서 메시지를 가져옴
    # timeout=1.0: 1초 동안 메시지가 없으면 None 반환
    msg = consumer.poll(timeout=1.0)

    # 메시지가 없으면 다시 시도
    if msg is None:
        continue

    # 에러가 있으면 출력하고 다시 시도
    if msg.error():
        print(f"❌ 에러: {msg.error()}")
        continue

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # Step 5: 메시지 처리 (bytes → dict)
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

    # bytes → str
    order_str = msg.value().decode('utf-8')

    # str → dict
    order = json.loads(order_str)

    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    # Step 6: 주문 정보 예쁘게 출력
    # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

    print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📦 새 주문 접수!
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
주문번호: {order['order_id']}
고객명: {order['customer']}
상품: {order['product']}
수량: {order['quantity']}개
금액: {order['price']:,}원
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    """)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Step 7: 정리 (Consumer 종료)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# close(): Consumer를 깔끔하게 종료
# - Consumer Group에서 탈퇴
# - 오프셋 커밋
# - 연결 종료
consumer.close()
print("✅ Consumer 종료 완료")

**실행 결과**:

```
📨 주문 대기 중... (ctrl+C로 종료)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📦 새 주문 접수!
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
주문번호: ORD-001
고객명: 홍길동
상품: 노트북
수량: 1개
금액: 1,500,000원
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🛑 종료 신호 수신 (ctrl+C). Consumer 종료 중...
✅ Consumer 종료 완료
```

## 🎉 완료!

축하합니다! Producer와 Consumer를 직접 구현하셨습니다.

### 핵심 포인트 복습

1. **Producer**:
   - `config` → `Producer(config)` → `produce()` → `flush()`
   - JSON 직렬화: `dict` → `json.dumps()` → `encode()`

2. **Consumer**:
   - `config` → `Consumer(config)` → `subscribe()` → `poll()` 루프
   - JSON 역직렬화: `bytes` → `decode()` → `json.loads()` → `dict`
   - Signal handling: `running` 플래그 + `signal.signal()`

3. **전체 흐름**:
   ```
   Producer: dict → JSON → bytes → Kafka
   Kafka: 메시지 저장
   Consumer: Kafka → bytes → JSON → dict → 처리
   ```

### 다음 단계

이제 Flask와 Kafka를 연동하여 실제 웹 애플리케이션에서 비동기 처리를 구현해봅시다!

---


# 🏗️ 데이터 파이프라인 구축

> **소요 시간**: 1.5시간 (90분)
> **난이도**: ⭐⭐⭐☆☆

## 🎯 학습 목표

이 파트의 목적:

- 데이터 엔지니어링 관점에서 Kafka 활용법 이해
- **로그 수집 → Kafka → DB 저장** 파이프라인 구축
- 실시간 데이터 적재 경험
- SQL과 Kafka 연동 (Day03-05 복습)
- 데이터 엔지니어의 핵심 업무 체험

### 구축할 파이프라인

```
[웹 서버 로그] → [Log Producer] → [Kafka] → [Log Consumer] → [PostgreSQL]
                                                                     ↓
                                                                 [SQL 분석]
```

## 📊 Part 1: 데이터 파이프라인이란?

### 1.1 데이터 파이프라인 개념

**데이터 파이프라인**(Data Pipeline): 데이터를 한 시스템에서 다른 시스템으로 이동시키고 변환하는 일련의 과정

**구성 요소**:
- **Source** (소스): 데이터가 생성되는 곳
  - 웹 서버 로그, API 응답, IoT 센서, 데이터베이스 등
- **Pipeline** (파이프라인): 데이터를 전송하는 통로
  - Kafka, RabbitMQ, AWS Kinesis 등
- **Sink** (싱크): 데이터가 저장되는 곳
  - PostgreSQL, MySQL, S3, Elasticsearch 등
- **Processing** (처리): 데이터를 변환하거나 정제
  - 필터링, 집계, 포맷 변환 등

### 1.2 실시간 vs 배치 파이프라인

| 구분 | 배치 파이프라인 | 실시간 파이프라인 |
|------|---------------|-----------------|
| **처리 방식** | 일정 주기마다 처리 (예: 매일 새벽) | 데이터 생성 즉시 처리 |
| **지연 시간** | 몇 시간 ~ 하루 | 초 단위 ~ 분 단위 |
| **기술** | Cron, Airflow | Kafka, Spark Streaming |
| **사용 사례** | 일일 리포트, 배치 분석 | 실시간 대시보드, 알림 |

**Kafka는 실시간 파이프라인의 핵심 기술입니다!**

### 1.3 오늘 구축할 파이프라인

**시나리오**: 전자상거래 웹사이트의 접속 로그를 실시간으로 수집하고 분석

1. **웹 서버**: 사용자가 페이지를 방문할 때마다 로그 생성
2. **Log Producer**: 로그를 Kafka로 전송
3. **Kafka**: 로그를 안정적으로 전달
4. **Log Consumer**: Kafka에서 로그를 읽어 PostgreSQL에 저장
5. **SQL 분석**: 저장된 로그를 SQL로 분석

**실무 활용**:
- 실시간 방문자 통계 대시보드
- 페이지 성능 모니터링
- 사용자 행동 분석
- 이상 트래픽 탐지

## 🐘 Part 2: PostgreSQL 설정 (10분)

### 2.1 Docker Compose에 PostgreSQL 추가

`compose.yml` 파일에 PostgreSQL 서비스를 추가합니다:

```yaml
services:
  broker:
    image: apache/kafka:latest
    ports:
      - "9092:9092"
    environment:
      KAFKA_NODE_ID: 1
      KAFKA_PROCESS_ROLES: broker,controller
      KAFKA_LISTENERS: PLAINTEXT://0.0.0.0:9092,CONTROLLER://0.0.0.0:9093
      KAFKA_ADVERTISED_LISTENERS: PLAINTEXT://broker:9092
      KAFKA_CONTROLLER_QUORUM_VOTERS: 1@broker:9093
      KAFKA_AUTO_CREATE_TOPICS_ENABLE: "true"

  kafka-ui:
    image: provectuslabs/kafka-ui:latest
    ports:
      - "8080:8080"
    environment:
      KAFKA_CLUSTERS_0_NAME: local
      KAFKA_CLUSTERS_0_BOOTSTRAPSERVERS: broker:9092

  postgres:
    image: postgres:15
    ports:
      - "5432:5432"
    environment:
      POSTGRES_DB: logs_db
      POSTGRES_USER: admin
      POSTGRES_PASSWORD: admin
    volumes:
      - postgres_data:/var/lib/postgresql/data

volumes:
  postgres_data:
```

### 2.2 PostgreSQL 시작

```bash
# Docker Compose로 전체 환경 시작
docker compose up -d

# PostgreSQL 실행 확인
docker compose ps
```

### 2.3 psycopg2 설치 및 연결 테스트

Python에서 PostgreSQL을 사용하려면 `psycopg2` 라이브러리가 필요합니다.

**Dockerfile에 추가**:
```dockerfile
FROM python:3.11-slim

COPY --from=ghcr.io/astral-sh/uv:latest /uv /usr/local/bin/uv

WORKDIR /app

RUN apt-get update && apt-get install -y \
    gcc \
    librdkafka-dev \
    libpq-dev \
    && rm -rf /var/lib/apt/lists/*

RUN uv pip install --system \
    confluent-kafka \
    psycopg2-binary

COPY . .
```

In [ ]:
# PostgreSQL 연결 테스트

import psycopg2

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# PostgreSQL 연결 정보
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

db_config = {
    'host': 'postgres',      # Docker 서비스 이름
    'port': 5432,
    'database': 'logs_db',
    'user': 'admin',
    'password': 'admin'
}

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 연결 테스트
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

try:
    # PostgreSQL 연결
    conn = psycopg2.connect(**db_config)
    cursor = conn.cursor()

    # 버전 확인 쿼리
    cursor.execute("SELECT version();")
    version = cursor.fetchone()

    print("✅ PostgreSQL 연결 성공!")
    print(f"   버전: {version[0]}")

    # 연결 종료
    cursor.close()
    conn.close()

except Exception as e:
    print(f"❌ 연결 실패: {e}")

**실행 결과**:

```
✅ PostgreSQL 연결 성공!
   버전: PostgreSQL 15.x (Debian 15.x-x.pgdg120+1) on aarch64-unknown-linux-gnu...
```

### 2.4 테이블 생성

웹 액세스 로그를 저장할 테이블을 생성합니다.

In [ ]:
# access_logs 테이블 생성

import psycopg2

db_config = {
    'host': 'postgres',
    'port': 5432,
    'database': 'logs_db',
    'user': 'admin',
    'password': 'admin'
}

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 테이블 생성 SQL
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

create_table_query = """
CREATE TABLE IF NOT EXISTS access_logs (
    id SERIAL PRIMARY KEY,
    timestamp TIMESTAMP NOT NULL,
    user_id VARCHAR(50),
    page VARCHAR(100),
    method VARCHAR(10),
    status_code INTEGER,
    duration_ms INTEGER,
    ip_address VARCHAR(50),
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
"""

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 테이블 생성
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

conn = psycopg2.connect(**db_config)
cursor = conn.cursor()

cursor.execute(create_table_query)
conn.commit()

print("✅ access_logs 테이블 생성 완료!")

# 테이블 확인
cursor.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_name = 'access_logs';
""")

columns = cursor.fetchall()
print("\n📋 테이블 구조:")
for col_name, col_type in columns:
    print(f"   - {col_name}: {col_type}")

cursor.close()
conn.close()

**실행 결과**:

```
✅ access_logs 테이블 생성 완료!

📋 테이블 구조:
   - id: integer
   - timestamp: timestamp without time zone
   - user_id: character varying
   - page: character varying
   - method: character varying
   - status_code: integer
   - duration_ms: integer
   - ip_address: character varying
   - created_at: timestamp without time zone
```

## 📤 Part 3: 로그 생성 Producer (30분)

### 3.1 웹 액세스 로그란?

**웹 액세스 로그**: 웹 서버에서 생성되는 로그로, 사용자의 페이지 방문 정보를 기록

**포함 정보**:
- **timestamp**: 접속 시각
- **user_id**: 사용자 ID
- **page**: 방문한 페이지 경로 (예: `/home`, `/products/123`)
- **method**: HTTP 메서드 (GET, POST 등)
- **status_code**: HTTP 상태 코드 (200, 404 등)
- **duration_ms**: 페이지 로딩 시간 (밀리초)
- **ip_address**: 사용자 IP 주소

### 3.2 로그 시뮬레이션 Producer

In [ ]:
# log_producer.py - 웹 액세스 로그를 Kafka로 전송

import json
import random
import time
from datetime import datetime
from confluent_kafka import Producer

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Kafka Producer 설정
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

config = {
    'bootstrap.servers': 'broker:9092',
    'client.id': 'log-producer'
}

producer = Producer(config)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 로그 데이터 생성을 위한 샘플 데이터
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# 방문 가능한 페이지 목록
PAGES = [
    '/home',
    '/products',
    '/products/laptop',
    '/products/phone',
    '/cart',
    '/checkout',
    '/about',
    '/contact'
]

# 사용자 ID 목록 (10명)
USER_IDS = [f"user_{i:03d}" for i in range(1, 11)]

# HTTP 메서드
METHODS = ['GET', 'POST']

# HTTP 상태 코드와 비율
STATUS_CODES = [
    (200, 0.85),  # 85% 성공
    (404, 0.10),  # 10% Not Found
    (500, 0.05)   # 5% Server Error
]

# IP 주소 풀
IP_ADDRESSES = [f"192.168.1.{i}" for i in range(1, 51)]

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 랜덤 로그 생성 함수
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def generate_access_log():
    """
    랜덤한 웹 액세스 로그를 생성

    Returns:
        dict: 로그 데이터 딕셔너리
    """
    # 상태 코드 선택 (가중치 적용)
    codes, weights = zip(*STATUS_CODES)
    status_code = random.choices(codes, weights=weights)[0]

    # 로그 데이터 생성
    log = {
        'timestamp': datetime.now().isoformat(),
        'user_id': random.choice(USER_IDS),
        'page': random.choice(PAGES),
        'method': random.choice(METHODS),
        'status_code': status_code,
        'duration_ms': random.randint(50, 2000),  # 50ms ~ 2초
        'ip_address': random.choice(IP_ADDRESSES)
    }

    return log

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Delivery callback (전송 결과 확인)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def delivery_callback(err, msg):
    """메시지 전송 결과를 확인하는 콜백"""
    if err:
        print(f"❌ 전송 실패: {err}")
    else:
        # 성공 시 간단히 출력
        log = json.loads(msg.value().decode('utf-8'))
        print(f"✅ [{log['timestamp']}] {log['user_id']} → {log['page']} ({log['status_code']})")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 로그 생성 및 전송 루프
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("🚀 로그 생성 시작... (ctrl+C로 중단)")

try:
    while True:
        # 로그 생성
        log = generate_access_log()

        # JSON 직렬화
        log_json = json.dumps(log, ensure_ascii=False)

        # Kafka로 전송
        producer.produce(
            topic='access-logs',
            value=log_json.encode('utf-8'),
            callback=delivery_callback
        )

        # poll(0): 비동기 전송을 위해 콜백 처리
        producer.poll(0)

        # 1초 대기 (실제로는 로그가 훨씬 빠르게 발생)
        time.sleep(1)

except KeyboardInterrupt:
    print("\n🛑 로그 생성 중단")

finally:
    # 남은 메시지 전송
    print("📤 남은 메시지 전송 중...")
    producer.flush()
    print("✅ Producer 종료 완료")

### 3.3 Producer 실행

```bash
# Terminal 1: Producer 실행
docker compose run python-producer python log_producer.py
```

**출력 예시**:
```
🚀 로그 생성 시작... (ctrl+C로 중단)
✅ [2025-01-14T10:30:15] user_003 → /products/laptop (200)
✅ [2025-01-14T10:30:16] user_007 → /home (200)
✅ [2025-01-14T10:30:17] user_001 → /cart (404)
...
```

## 📥 Part 4: 로그 저장 Consumer (30분)

### 4.1 Consumer 구현

In [ ]:
# log_consumer.py - Kafka에서 로그를 읽어 PostgreSQL에 저장

import json
import signal
import psycopg2
from datetime import datetime
from confluent_kafka import Consumer

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Signal Handling
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

running = True

def signal_handler(sig, frame):
    global running
    print("\n🛑 종료 신호 수신. Consumer 종료 중...")
    running = False

signal.signal(signal.SIGINT, signal_handler)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# PostgreSQL 연결
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

db_config = {
    'host': 'postgres',
    'port': 5432,
    'database': 'logs_db',
    'user': 'admin',
    'password': 'admin'
}

# PostgreSQL 연결
db_conn = psycopg2.connect(**db_config)
db_cursor = db_conn.cursor()

print("✅ PostgreSQL 연결 성공")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Kafka Consumer 설정
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

config = {
    'bootstrap.servers': 'broker:9092',
    'group.id': 'log-storage-group',
    'auto.offset.reset': 'earliest'
}

consumer = Consumer(config)
consumer.subscribe(['access-logs'])

print("✅ Kafka Consumer 구독 시작")
print("📊 로그를 PostgreSQL에 저장 중... (ctrl+C로 중단)")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 로그 저장 함수
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def save_log_to_db(log):
    """
    로그 데이터를 PostgreSQL에 저장

    Args:
        log (dict): 로그 데이터
    """
    try:
        # INSERT 쿼리
        insert_query = """
        INSERT INTO access_logs
        (timestamp, user_id, page, method, status_code, duration_ms, ip_address)
        VALUES (%s, %s, %s, %s, %s, %s, %s)
        """

        # 데이터 준비
        values = (
            datetime.fromisoformat(log['timestamp']),
            log['user_id'],
            log['page'],
            log['method'],
            log['status_code'],
            log['duration_ms'],
            log['ip_address']
        )

        # 실행
        db_cursor.execute(insert_query, values)
        db_conn.commit()

        # 성공 메시지
        print(f"💾 저장: {log['user_id']} → {log['page']} ({log['status_code']})")

    except Exception as e:
        print(f"❌ 저장 실패: {e}")
        db_conn.rollback()

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 메시지 수신 및 저장 루프
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

message_count = 0

while running:
    msg = consumer.poll(timeout=1.0)

    if msg is None:
        continue
    if msg.error():
        print(f"❌ 에러: {msg.error()}")
        continue

    # 메시지 파싱
    log = json.loads(msg.value().decode('utf-8'))

    # PostgreSQL에 저장
    save_log_to_db(log)

    message_count += 1

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 정리
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

consumer.close()
db_cursor.close()
db_conn.close()

print(f"\n✅ Consumer 종료 완료 (총 {message_count}개 로그 저장)")

### 4.2 Consumer 실행

```bash
# Terminal 2: Consumer 실행
docker compose run python-producer python log_consumer.py
```

**출력 예시**:
```
✅ PostgreSQL 연결 성공
✅ Kafka Consumer 구독 시작
📊 로그를 PostgreSQL에 저장 중... (ctrl+C로 중단)
💾 저장: user_003 → /products/laptop (200)
💾 저장: user_007 → /home (200)
💾 저장: user_001 → /cart (404)
...
```

## 📊 Part 5: SQL 분석 (20분)

이제 PostgreSQL에 쌓인 로그를 SQL로 분석해봅시다!

### 5.1 PostgreSQL 접속

```bash
# PostgreSQL 컨테이너에 접속
docker compose exec postgres psql -U admin -d logs_db
```

### 5.2 기본 조회

In [ ]:
# SQL 쿼리 실행 예제

import psycopg2

db_config = {
    'host': 'postgres',
    'port': 5432,
    'database': 'logs_db',
    'user': 'admin',
    'password': 'admin'
}

conn = psycopg2.connect(**db_config)
cursor = conn.cursor()

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 1. 총 로그 개수
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

cursor.execute("SELECT COUNT(*) FROM access_logs;")
total_logs = cursor.fetchone()[0]
print(f"📊 총 로그 개수: {total_logs:,}개")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 2. 가장 많이 방문한 페이지 TOP 5
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n📈 가장 많이 방문한 페이지 TOP 5:")
cursor.execute("""
    SELECT page, COUNT(*) as visit_count
    FROM access_logs
    GROUP BY page
    ORDER BY visit_count DESC
    LIMIT 5;
""")

for page, count in cursor.fetchall():
    print(f"   {page}: {count:,}회")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3. 페이지별 평균 로딩 시간
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n⏱️  페이지별 평균 로딩 시간:")
cursor.execute("""
    SELECT page, AVG(duration_ms) as avg_duration
    FROM access_logs
    GROUP BY page
    ORDER BY avg_duration DESC;
""")

for page, avg_duration in cursor.fetchall():
    print(f"   {page}: {avg_duration:.0f}ms")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4. HTTP 상태 코드별 분포
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n🔢 HTTP 상태 코드 분포:")
cursor.execute("""
    SELECT status_code, COUNT(*) as count,
           ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM access_logs), 2) as percentage
    FROM access_logs
    GROUP BY status_code
    ORDER BY count DESC;
""")

for status_code, count, percentage in cursor.fetchall():
    print(f"   {status_code}: {count:,}회 ({percentage}%)")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 5. 가장 활동적인 사용자 TOP 5
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n👥 가장 활동적인 사용자 TOP 5:")
cursor.execute("""
    SELECT user_id, COUNT(*) as visit_count
    FROM access_logs
    GROUP BY user_id
    ORDER BY visit_count DESC
    LIMIT 5;
""")

for user_id, count in cursor.fetchall():
    print(f"   {user_id}: {count:,}회")

cursor.close()
conn.close()

**실행 결과 예시** (30개 로그 기준):

```
📊 총 로그 개수: 30개

📈 가장 많이 방문한 페이지 TOP 5:
   /products: 8회
   /products/laptop: 6회
   /home: 5회
   /cart: 4회
   /checkout: 3회

⏱️  페이지별 평균 로딩 시간:
   /checkout: 1,523ms
   /products/phone: 1,205ms
   /cart: 987ms
   /home: 654ms
   /products: 432ms

🔢 HTTP 상태 코드 분포:
   200: 25회 (83.33%)
   404: 4회 (13.33%)
   500: 1회 (3.33%)

👥 가장 활동적인 사용자 TOP 5:
   user_007: 5회
   user_003: 4회
   user_001: 4회
   user_009: 3회
   user_005: 3회
```

### 5.3 고급 분석 쿼리

Day03-05에서 배운 SQL 기술을 활용합니다!

In [ ]:
# 고급 SQL 분석 예제

conn = psycopg2.connect(**db_config)
cursor = conn.cursor()

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 1. 시간대별 트래픽 분석 (Window Function 활용)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("🕐 시간대별 트래픽 분석:")
cursor.execute("""
    SELECT
        EXTRACT(HOUR FROM timestamp) as hour,
        COUNT(*) as request_count,
        AVG(duration_ms) as avg_duration
    FROM access_logs
    GROUP BY hour
    ORDER BY hour;
""")

for hour, count, avg_duration in cursor.fetchall():
    print(f"   {int(hour):02d}시: {count:,}회 (평균 {avg_duration:.0f}ms)")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 2. 사용자별 페이지 방문 순서 분석 (LAG 함수)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n🔄 사용자별 페이지 이동 패턴 (최근 5개):")
cursor.execute("""
    WITH user_journey AS (
        SELECT
            user_id,
            page,
            LAG(page) OVER (PARTITION BY user_id ORDER BY timestamp) as previous_page,
            timestamp
        FROM access_logs
        ORDER BY timestamp DESC
        LIMIT 5
    )
    SELECT user_id, previous_page, page
    FROM user_journey
    WHERE previous_page IS NOT NULL;
""")

for user_id, prev_page, page in cursor.fetchall():
    print(f"   {user_id}: {prev_page} → {page}")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3. 에러가 많이 발생하는 페이지 찾기
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n⚠️  에러가 많은 페이지 (4xx, 5xx):")
cursor.execute("""
    SELECT
        page,
        COUNT(*) as error_count,
        ROUND(COUNT(*) * 100.0 /
            (SELECT COUNT(*) FROM access_logs WHERE page = a.page), 2) as error_rate
    FROM access_logs a
    WHERE status_code >= 400
    GROUP BY page
    ORDER BY error_count DESC
    LIMIT 5;
""")

for page, error_count, error_rate in cursor.fetchall():
    print(f"   {page}: {error_count:,}회 ({error_rate}%)")

cursor.close()
conn.close()

## 🎓 Part 6: 정리 및 FAQ

### 6.1 배운 내용 정리

#### 구축한 파이프라인

```
[웹 로그] → [Producer] → [Kafka] → [Consumer] → [PostgreSQL] → [SQL 분석]
```

#### 핵심 개념

1. **실시간 데이터 파이프라인**
   - Source (웹 서버) → Pipeline (Kafka) → Sink (PostgreSQL)
   - 데이터가 생성되는 즉시 처리

2. **Producer**
   - 로그 데이터 생성 및 JSON 직렬화
   - Kafka로 전송

3. **Consumer**
   - Kafka에서 메시지 수신
   - PostgreSQL에 INSERT

4. **SQL 분석**
   - 집계 함수 (COUNT, AVG)
   - GROUP BY
   - Window Function (LAG)
   - Day03-05 내용 복습

### 6.2 실무 활용

이 파이프라인은 실제로 이렇게 사용됩니다:

1. **실시간 대시보드**
   - Grafana나 Kibana로 로그 시각화
   - 실시간 방문자 수, 페이지 응답 시간 모니터링

2. **이상 탐지**
   - 에러율이 급증하면 알림
   - 비정상적인 트래픽 패턴 감지

3. **A/B 테스트**
   - 사용자 행동 분석
   - 페이지 성능 비교

4. **데이터 레이크 구축**
   - Kafka → S3 (장기 보관)
   - Spark로 배치 분석

### 6.3 FAQ

**Q1. Producer와 Consumer를 같은 컨테이너에서 실행할 수 있나요?**

A. 테스트 환경에서는 가능하지만, 프로덕션에서는 분리합니다:
- Producer: 웹 서버와 함께 배포
- Consumer: 별도의 서비스로 배포 (확장 가능)

---

**Q2. Consumer가 다운되면 데이터가 유실되나요?**

A. 아니요! Kafka가 메시지를 보관하고 있습니다:
- Consumer가 다시 시작하면 마지막 오프셋부터 재개
- `enable.auto.commit: True`가 기본 설정

---

**Q3. PostgreSQL 대신 다른 DB를 사용할 수 있나요?**

A. 네, 원하는 DB를 자유롭게 선택할 수 있습니다:
- MySQL: `mysql-connector-python` 사용
- MongoDB: `pymongo` 사용
- Elasticsearch: `elasticsearch` 사용
- 핵심은 Kafka가 DB와 독립적이라는 점!

---

**Q4. 로그가 너무 많으면 어떻게 하나요?**

A. 여러 방법이 있습니다:
1. **파티션 증가**: 병렬 처리 향상
2. **Consumer 추가**: 같은 group.id로 여러 Consumer 실행
3. **배치 INSERT**: 여러 개를 모아서 한 번에 INSERT
4. **샘플링**: 중요한 로그만 저장

---

**Q5. 데이터 순서가 보장되나요?**

A. 같은 파티션 내에서는 순서가 보장됩니다:
- Key를 지정하면 같은 Key는 같은 파티션으로
- 같은 파티션은 순서 보장
- 다른 파티션 간에는 순서 보장 안 됨

## 🔌 Part 7: Kafka Connect 소개 (15분)

지금까지 Producer와 Consumer를 **직접 코드로 구현**했습니다.
하지만 실무에서는 **Kafka Connect**를 사용하면 코드 없이 데이터 파이프라인을 구축할 수 있습니다!

### 7.1 Kafka Connect란?

**Kafka Connect**: 코드 작성 없이 **설정만으로** 데이터를 Kafka로 가져오거나 내보내는 프레임워크

```
[Source System] → [Source Connector] → [Kafka] → [Sink Connector] → [Target System]
```

**핵심 개념**:
- **Source Connector**: 외부 시스템 → Kafka (데이터 수집)
  - DB, 파일, API 등에서 데이터를 읽어 Kafka로 전송
- **Sink Connector**: Kafka → 외부 시스템 (데이터 적재)
  - Kafka의 데이터를 DB, S3, Elasticsearch 등에 저장

### 7.2 왜 Kafka Connect를 사용할까?

| 구분 | 직접 구현 (Part 3-4) | Kafka Connect |
|------|---------------------|---------------|
| **개발** | Python 코드 작성 | JSON 설정 파일 |
| **에러 처리** | 직접 구현 | 내장 (재시도, 오프셋 관리) |
| **확장성** | 코드 수정 필요 | Worker 추가만으로 확장 |
| **모니터링** | 직접 구현 | REST API 제공 |

### 7.3 주요 Connector 종류

**Source Connectors** (데이터 수집):
- `JDBC Source`: DB 테이블 → Kafka
- `Debezium`: DB CDC (Change Data Capture)
- `File Source`: 파일 → Kafka

**Sink Connectors** (데이터 적재):
- `JDBC Sink`: Kafka → DB 테이블
- `S3 Sink`: Kafka → AWS S3
- `Elasticsearch Sink`: Kafka → Elasticsearch

### 7.4 실행 모드

| 모드 | 설명 | 사용 환경 |
|------|------|----------|
| **Standalone** | 단일 프로세스 | 개발/테스트 |
| **Distributed** | 여러 Worker가 협력 | 프로덕션 |

## 🛠️ Part 8: JDBC Sink Connector 실습 (30분)

Part 4에서 직접 구현한 Consumer를 **JDBC Sink Connector로 대체**해봅시다!

```
[Producer] → [Kafka] → [JDBC Sink Connector] → [PostgreSQL]
                             ↑
                    Consumer 코드 대신 설정으로!
```

### 8.1 Docker Compose에 Kafka Connect 추가

`compose.yml` 파일을 수정합니다:

```yaml
services:
  broker:
    image: apache/kafka:latest
    ports:
      - "9092:9092"
    environment:
      KAFKA_NODE_ID: 1
      KAFKA_PROCESS_ROLES: broker,controller
      KAFKA_LISTENERS: PLAINTEXT://0.0.0.0:9092,CONTROLLER://0.0.0.0:9093
      KAFKA_ADVERTISED_LISTENERS: PLAINTEXT://broker:9092
      KAFKA_CONTROLLER_QUORUM_VOTERS: 1@broker:9093
      KAFKA_CONTROLLER_LISTENER_NAMES: CONTROLLER
      KAFKA_LISTENER_SECURITY_PROTOCOL_MAP: CONTROLLER:PLAINTEXT,PLAINTEXT:PLAINTEXT
      KAFKA_AUTO_CREATE_TOPICS_ENABLE: "true"

  kafka-ui:
    image: provectuslabs/kafka-ui:latest
    ports:
      - "8080:8080"
    environment:
      KAFKA_CLUSTERS_0_NAME: local
      KAFKA_CLUSTERS_0_BOOTSTRAPSERVERS: broker:9092
    depends_on:
      - broker

  postgres:
    image: postgres:15
    ports:
      - "5432:5432"
    environment:
      POSTGRES_DB: logs_db
      POSTGRES_USER: admin
      POSTGRES_PASSWORD: admin
    volumes:
      - postgres_data:/var/lib/postgresql/data

  # Kafka Connect 추가!
  kafka-connect:
    image: confluentinc/cp-kafka-connect:7.5.0
    ports:
      - "8083:8083"
    environment:
      CONNECT_BOOTSTRAP_SERVERS: broker:9092
      CONNECT_REST_PORT: 8083
      CONNECT_GROUP_ID: connect-cluster
      CONNECT_CONFIG_STORAGE_TOPIC: connect-configs
      CONNECT_OFFSET_STORAGE_TOPIC: connect-offsets
      CONNECT_STATUS_STORAGE_TOPIC: connect-status
      CONNECT_CONFIG_STORAGE_REPLICATION_FACTOR: 1
      CONNECT_OFFSET_STORAGE_REPLICATION_FACTOR: 1
      CONNECT_STATUS_STORAGE_REPLICATION_FACTOR: 1
      CONNECT_KEY_CONVERTER: org.apache.kafka.connect.json.JsonConverter
      CONNECT_VALUE_CONVERTER: org.apache.kafka.connect.json.JsonConverter
      CONNECT_KEY_CONVERTER_SCHEMAS_ENABLE: "false"
      CONNECT_VALUE_CONVERTER_SCHEMAS_ENABLE: "false"
      CONNECT_REST_ADVERTISED_HOST_NAME: kafka-connect
      CONNECT_PLUGIN_PATH: /usr/share/java,/usr/share/confluent-hub-components
    command:
      - bash
      - -c
      - |
        # JDBC 드라이버 설치
        confluent-hub install --no-prompt confluentinc/kafka-connect-jdbc:10.7.4
        # Connect 시작
        /etc/confluent/docker/run
    depends_on:
      - broker
      - postgres

volumes:
  postgres_data:
```

### 8.2 환경 시작 및 확인

```bash
# 환경 시작
docker compose up -d

# Kafka Connect 상태 확인 (시작까지 1-2분 소요)
curl http://localhost:8083/

# 설치된 플러그인 확인
curl http://localhost:8083/connector-plugins | jq
```

**출력 예시**:
```json
[
  {"class": "io.confluent.connect.jdbc.JdbcSinkConnector", ...},
  {"class": "io.confluent.connect.jdbc.JdbcSourceConnector", ...}
]
```

### 8.3 access_logs_connect 테이블 생성

JDBC Sink Connector가 사용할 별도 테이블을 만듭니다.
(기존 테이블과 비교하기 위해)

In [ ]:
# Connect용 테이블 생성

import psycopg2

db_config = {
    'host': 'postgres',
    'port': 5432,
    'database': 'logs_db',
    'user': 'admin',
    'password': 'admin'
}

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Connect용 테이블 생성 SQL
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

create_table_query = """
CREATE TABLE IF NOT EXISTS access_logs_connect (
    timestamp VARCHAR(50),
    user_id VARCHAR(50),
    page VARCHAR(100),
    method VARCHAR(10),
    status_code INTEGER,
    duration_ms INTEGER,
    ip_address VARCHAR(50)
);
"""

conn = psycopg2.connect(**db_config)
cursor = conn.cursor()

cursor.execute(create_table_query)
conn.commit()

print("✅ access_logs_connect 테이블 생성 완료!")

cursor.close()
conn.close()

### 8.4 JDBC Sink Connector 등록

REST API로 Connector를 등록합니다.

In [ ]:
# JDBC Sink Connector 등록

import requests
import json

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Connector 설정
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

connector_config = {
    "name": "jdbc-sink-access-logs",
    "config": {
        "connector.class": "io.confluent.connect.jdbc.JdbcSinkConnector",
        "tasks.max": "1",

        # Kafka 토픽
        "topics": "access-logs",

        # PostgreSQL 연결 정보
        "connection.url": "jdbc:postgresql://postgres:5432/logs_db",
        "connection.user": "admin",
        "connection.password": "admin",

        # 테이블 설정
        "table.name.format": "access_logs_connect",
        "auto.create": "false",
        "insert.mode": "insert",

        # 키/값 변환
        "key.converter": "org.apache.kafka.connect.storage.StringConverter",
        "value.converter": "org.apache.kafka.connect.json.JsonConverter",
        "value.converter.schemas.enable": "false"
    }
}

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Connector 등록
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

connect_url = "http://kafka-connect:8083/connectors"

response = requests.post(
    connect_url,
    headers={"Content-Type": "application/json"},
    data=json.dumps(connector_config)
)

if response.status_code == 201:
    print("✅ JDBC Sink Connector 등록 성공!")
    print(json.dumps(response.json(), indent=2))
else:
    print(f"❌ 등록 실패: {response.status_code}")
    print(response.text)

### 8.5 Connector 상태 확인

```bash
# 등록된 Connector 목록
curl http://localhost:8083/connectors

# Connector 상태 확인
curl http://localhost:8083/connectors/jdbc-sink-access-logs/status | jq
```

**정상 상태**:
```json
{
  "name": "jdbc-sink-access-logs",
  "connector": {"state": "RUNNING", ...},
  "tasks": [{"state": "RUNNING", ...}]
}
```

In [ ]:
# Connector 상태 확인 코드

import requests

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Connector 상태 조회
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

status_url = "http://kafka-connect:8083/connectors/jdbc-sink-access-logs/status"

response = requests.get(status_url)

if response.status_code == 200:
    status = response.json()
    print(f"📊 Connector 상태: {status['connector']['state']}")
    for task in status.get('tasks', []):
        print(f"   Task {task['id']}: {task['state']}")
else:
    print(f"❌ 조회 실패: {response.status_code}")

### 8.6 테스트: Producer 실행 후 결과 확인

이제 **Consumer 코드 없이** Producer만 실행하면 데이터가 자동으로 DB에 저장됩니다!

```bash
# Terminal 1: Producer 실행 (Part 3에서 만든 코드 재사용)
docker compose run python-producer python log_producer.py

# 몇 개 메시지 전송 후 Ctrl+C로 중단
```

### 8.7 결과 확인

In [ ]:
# Connect로 저장된 데이터 확인

import psycopg2

db_config = {
    'host': 'postgres',
    'port': 5432,
    'database': 'logs_db',
    'user': 'admin',
    'password': 'admin'
}

conn = psycopg2.connect(**db_config)
cursor = conn.cursor()

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 두 테이블 비교
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# 직접 구현한 Consumer로 저장된 데이터
cursor.execute("SELECT COUNT(*) FROM access_logs;")
manual_count = cursor.fetchone()[0]

# Kafka Connect로 저장된 데이터
cursor.execute("SELECT COUNT(*) FROM access_logs_connect;")
connect_count = cursor.fetchone()[0]

print("📊 테이블별 저장된 로그 수:")
print(f"   - access_logs (직접 구현): {manual_count:,}개")
print(f"   - access_logs_connect (Kafka Connect): {connect_count:,}개")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Connect로 저장된 최근 데이터 확인
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n📋 Kafka Connect로 저장된 최근 5개 로그:")
cursor.execute("""
    SELECT user_id, page, status_code
    FROM access_logs_connect
    ORDER BY timestamp DESC
    LIMIT 5;
""")

for user_id, page, status_code in cursor.fetchall():
    print(f"   {user_id} -> {page} ({status_code})")

cursor.close()
conn.close()

### 8.8 Connector 관리 명령어

```bash
# Connector 일시 중지
curl -X PUT http://localhost:8083/connectors/jdbc-sink-access-logs/pause

# Connector 재개
curl -X PUT http://localhost:8083/connectors/jdbc-sink-access-logs/resume

# Connector 삭제
curl -X DELETE http://localhost:8083/connectors/jdbc-sink-access-logs

# Connector 설정 조회
curl http://localhost:8083/connectors/jdbc-sink-access-logs/config | jq
```

## 🤔 Part 9: 언제 Connect를 쓸까? (10분)

### 9.1 직접 구현 vs Kafka Connect 비교

| 구분 | 직접 구현 (Part 3-4) | Kafka Connect (Part 8) |
|------|---------------------|------------------------|
| **코드량** | Producer + Consumer 코드 | JSON 설정 파일 |
| **개발 시간** | 수 시간 ~ 수 일 | 수 분 ~ 수 시간 |
| **유연성** | 무한대 (원하는 대로 구현) | Connector가 지원하는 범위 |
| **에러 처리** | 직접 구현 (try/except) | 내장 (재시도, DLQ) |
| **오프셋 관리** | 직접 구현 또는 auto commit | 자동 관리 |
| **확장** | 코드 수정 + 배포 | Worker 추가 |
| **모니터링** | 직접 구현 | REST API + JMX |
| **학습 곡선** | Python/Kafka 지식 | Connect 설정 문법 |

### 9.2 판단 기준 체크리스트

**Kafka Connect를 선택해야 할 때**:

- [ ] 표준적인 ETL 작업 (DB → Kafka → DB)
- [ ] 이미 존재하는 Connector가 요구사항을 충족
- [ ] 빠른 개발이 필요
- [ ] 운영팀이 코드보다 설정 관리를 선호
- [ ] 여러 데이터 소스/싱크를 동일한 방식으로 관리

**직접 구현을 선택해야 할 때**:

- [ ] 복잡한 데이터 변환이 필요 (예: ML 모델 적용)
- [ ] 특수한 비즈니스 로직 (예: 조건부 라우팅)
- [ ] Connector가 없는 시스템과 연동
- [ ] 세밀한 제어가 필요 (메시지별 처리 로직)
- [ ] 이미 Python/Java 기반 시스템이 있음

### 9.3 실무에서의 조합

실제로는 **둘 다 함께** 사용하는 경우가 많습니다:

```
[Source DB] → [Debezium Source] → [Kafka] → [Python Consumer] → [ML 처리]
                                     ↓
                             [JDBC Sink] → [Data Warehouse]
```

- **Connect**: 단순한 데이터 이동 (CDC, 적재)
- **직접 구현**: 복잡한 처리 (ML, 실시간 알림)

### 9.4 정리

| 상황 | 추천 |
|------|------|
| DB 테이블 → Kafka | Debezium/JDBC Source |
| Kafka → DB/S3/ES | JDBC/S3/ES Sink |
| 복잡한 변환/필터링 | 직접 구현 (Python) |
| 실시간 알림/액션 | 직접 구현 |
| 빠른 PoC | Kafka Connect |
| 특수 시스템 연동 | 직접 구현 |

## 🚀 다음 단계

이제 Kafka를 활용한 실시간 데이터 파이프라인을 구축할 수 있습니다!

**다음 교시 예고**:
- Producer `acks` 설정과 신뢰성
- 성능과 안정성의 트레이드오프 이해

지금까지:
- ✅ Producer 기초
- ✅ Consumer 기초
- ✅ Flask 비동기 전환
- ✅ 실습 과제
- ✅ 데이터 파이프라인 구축
- ✅ Kafka Connect 소개 및 실습

다음:
- ⏭️ acks 설정
- ⏭️ 종합 복습

---


# ⚙️ Producer acks 설정과 신뢰성

> **소요 시간**: 30분
> **난이도**: ⭐⭐⭐☆☆

## 🎯 학습 목표

이 파트의 목적:

- Producer `acks` 설정의 의미 이해
- 성능과 안정성의 **트레이드오프** 파악
- 실무에서 어떤 설정을 선택할지 판단 기준 습득

**핵심 질문**: "데이터를 얼마나 안전하게 보낼 것인가?"

## 📚 Part 1: acks 설정이란? (15분)

### 1.1 acks 개념

**acks**(Acknowledgment): Producer가 메시지를 전송한 후, Kafka 브로커로부터 **몇 개의 응답**을 받아야 "전송 완료"로 간주할지 결정하는 설정

```
[Producer] ──메시지 전송──▶ [Leader Broker]
                                   │
                                   ├──복제──▶ [Follower 1]
                                   └──복제──▶ [Follower 2]

acks=0: 응답 안 기다림
acks=1: Leader만 응답 기다림
acks=all: Leader + 모든 Follower 응답 기다림
```

### 1.2 세 가지 acks 설정

#### acks=0 (Fire and Forget)

**동작 방식**:
- Producer가 메시지를 전송하고 **응답을 기다리지 않음**
- 브로커가 메시지를 받았는지 확인하지 않음

**장점**:
- ⚡ 가장 빠른 성능
- 네트워크 대역폭 절약

**단점**:
- ⚠️ 메시지 유실 가능성 높음
- 브로커가 다운되어도 모름
- 네트워크 문제로 전송 실패해도 모름

**사용 사례**:
- 메트릭, 로그 수집 (일부 유실 허용)
- IoT 센서 데이터 (초 단위로 계속 들어오는 데이터)
- 실시간 위치 추적

---

#### acks=1 (Leader Acknowledgment) - **기본값**

**동작 방식**:
- Leader 브로커가 메시지를 받으면 응답
- Follower 복제는 기다리지 않음

**장점**:
- 중간 수준의 성능
- 기본적인 안정성 보장

**단점**:
- ⚠️ Leader가 응답 후 즉시 다운되면 메시지 유실 가능
- Follower로 복제되기 전에 장애 발생 시 문제

**사용 사례**:
- 일반적인 이벤트 처리
- 사용자 활동 로그
- 알림 메시지

---

#### acks=all (또는 acks=-1) (All In-Sync Replicas)

**동작 방식**:
- Leader와 **모든 In-Sync Replica (ISR)**가 메시지를 받아야 응답
- ISR: Leader와 동기화된 Follower 목록

**장점**:
- ✅ 가장 높은 안정성
- 데이터 유실 거의 불가능
- Leader 다운되어도 Follower가 가지고 있음

**단점**:
- ⏱️ 가장 느린 성능
- 네트워크 지연 시 큰 영향

**사용 사례**:
- 금융 거래
- 주문 처리
- 결제 정보
- 절대 유실되면 안 되는 데이터

### 1.3 비교표

| 설정 | 속도 | 안정성 | 유실 가능성 | 사용 사례 |
|------|------|--------|------------|----------|
| **acks=0** | ⭐⭐⭐ 매우 빠름 | ❌ 낮음 | 높음 | 로그, 메트릭 |
| **acks=1** | ⭐⭐ 중간 | ⚠️ 중간 | 낮음 | 일반 이벤트 |
| **acks=all** | ⭐ 느림 | ✅ 높음 | 거의 없음 | 주문, 결제 |

## 💻 Part 2: acks 설정 실습 (15분)

### 2.1 벤치마크 코드

각 acks 설정으로 1000개 메시지를 전송하고 시간을 측정합니다.

In [ ]:
# acks_benchmark.py - acks 설정별 성능 비교

import time
import json
from confluent_kafka import Producer

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 벤치마크 함수
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def benchmark_acks(acks_value, message_count=1000):
    """
    특정 acks 설정으로 메시지를 전송하고 시간을 측정

    Args:
        acks_value: acks 설정 값 (0, 1, 'all')
        message_count: 전송할 메시지 개수

    Returns:
        float: 전송에 걸린 시간 (초)
    """
    # Producer 설정
    config = {
        'bootstrap.servers': 'broker:9092',
        'client.id': f'benchmark-acks-{acks_value}',
        'acks': acks_value  # 핵심 설정!
    }

    producer = Producer(config)

    # 전송 시작 시간
    start_time = time.time()

    # 메시지 전송
    for i in range(message_count):
        message = {
            'id': i,
            'acks': str(acks_value),
            'timestamp': time.time()
        }

        producer.produce(
            topic='benchmark',
            value=json.dumps(message).encode('utf-8')
        )

        # 매 100개마다 poll() 호출 (버퍼 관리)
        if i % 100 == 0:
            producer.poll(0)

    # 모든 메시지 전송 완료 대기
    producer.flush()

    # 전송 종료 시간
    end_time = time.time()
    elapsed_time = end_time - start_time

    return elapsed_time

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 벤치마크 실행
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("🏁 acks 설정별 성능 벤치마크 시작")
print("=" * 60)

MESSAGE_COUNT = 1000

# acks=0 테스트
print("\n⚡ acks=0 (Fire and Forget) 테스트 중...")
time_acks_0 = benchmark_acks(acks_value=0, message_count=MESSAGE_COUNT)
print(f"   완료: {time_acks_0:.3f}초")

# acks=1 테스트
print("\n⚡ acks=1 (Leader Only) 테스트 중...")
time_acks_1 = benchmark_acks(acks_value=1, message_count=MESSAGE_COUNT)
print(f"   완료: {time_acks_1:.3f}초")

# acks=all 테스트
print("\n⚡ acks=all (All ISR) 테스트 중...")
time_acks_all = benchmark_acks(acks_value='all', message_count=MESSAGE_COUNT)
print(f"   완료: {time_acks_all:.3f}초")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 결과 비교
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("\n" + "=" * 60)
print("📊 벤치마크 결과 비교")
print("=" * 60)

print(f"\nacks=0:   {time_acks_0:.3f}초  (기준: 1.00x)")
print(f"acks=1:   {time_acks_1:.3f}초  (기준: {time_acks_1/time_acks_0:.2f}x)")
print(f"acks=all: {time_acks_all:.3f}초  (기준: {time_acks_all/time_acks_0:.2f}x)")

print("\n💡 해석:")
if time_acks_all > time_acks_1 * 1.5:
    print("   acks=all이 acks=1보다 50% 이상 느립니다.")
    print("   → 안정성을 위해 성능을 희생하는 트레이드오프!")
else:
    print("   acks=all과 acks=1의 성능 차이가 크지 않습니다.")
    print("   → 안정성이 중요하다면 acks=all 사용 권장!")

### 2.2 실행 결과 예시

**프로덕션 환경 (3개 브로커, 복제 팩터 3)에서의 예상 결과**:

```
🏁 acks 설정별 성능 벤치마크 시작
============================================================

⚡ acks=0 (Fire and Forget) 테스트 중...
   완료: 0.523초

⚡ acks=1 (Leader Only) 테스트 중...
   완료: 1.234초

⚡ acks=all (All ISR) 테스트 중...
   완료: 2.456초

============================================================
📊 벤치마크 결과 비교
============================================================

acks=0:   0.523초  (기준: 1.00x)
acks=1:   1.234초  (기준: 2.36x)
acks=all: 2.456초  (기준: 4.70x)

💡 해석:
   acks=all이 acks=1보다 50% 이상 느립니다.
   → 안정성을 위해 성능을 희생하는 트레이드오프!
```

**로컬 Docker 환경 (단일 브로커)에서의 실제 결과**:

```
⚡ acks=0 (Fire and Forget) 테스트 중...
   완료: 0.107초

⚡ acks=1 (Leader Only) 테스트 중...
   완료: 0.019초

⚡ acks=all (All ISR) 테스트 중...
   완료: 0.005초
```

> **참고**: 로컬 Docker 환경에서는 단일 브로커만 있고 복제가 없기 때문에
> `acks=1`과 `acks=all`의 차이가 거의 없습니다. 프로덕션 환경에서
> 3개 이상의 브로커와 복제 팩터 설정이 있을 때 위 표의 차이가 명확하게 나타납니다.

### 2.3 메시지 유실 테스트 (선택)

**주의**: 이 테스트는 실제 브로커를 중단시키므로 테스트 환경에서만 수행하세요!

In [ ]:
# message_loss_test.py - acks=0 vs acks=all 유실 테스트

import time
import json
from confluent_kafka import Producer

def send_critical_message(acks_value):
    """
    중요한 메시지를 특정 acks 설정으로 전송

    Args:
        acks_value: acks 설정 값
    """
    config = {
        'bootstrap.servers': 'broker:9092',
        'client.id': f'loss-test-{acks_value}',
        'acks': acks_value
    }

    producer = Producer(config)

    message = {
        'transaction_id': 'TRX-12345',
        'amount': 1000000,
        'acks': str(acks_value),
        'note': '이 메시지는 절대 유실되면 안 됩니다!'
    }

    print(f"\n💰 중요한 거래 메시지 전송 (acks={acks_value})...")

    try:
        producer.produce(
            topic='critical-transactions',
            value=json.dumps(message, ensure_ascii=False).encode('utf-8')
        )
        producer.flush(timeout=5)
        print(f"   ✅ 전송 완료!")

    except Exception as e:
        print(f"   ❌ 전송 실패: {e}")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 테스트 시나리오
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("🧪 메시지 유실 테스트")
print("=" * 60)
print("\n📝 시나리오:")
print("   1. acks=0으로 메시지 전송 (응답 안 기다림)")
print("   2. acks=all로 메시지 전송 (모든 복제 완료)")
print("   3. 브로커 중단 시 어떤 메시지가 살아남는지 확인")
print("\n⚠️  주의: 이 테스트는 실제 브로커를 중단시킵니다!")
print("=" * 60)

# acks=0 테스트
send_critical_message(acks_value=0)

# 잠시 대기
time.sleep(1)

# acks=all 테스트
send_critical_message(acks_value='all')

print("\n" + "=" * 60)
print("✅ 테스트 완료")
print("\n💡 확인 방법:")
print("   1. Kafka UI (http://localhost:8080)에서 'critical-transactions' 토픽 확인")
print("   2. docker compose stop broker 로 브로커 중단")
print("   3. docker compose start broker 로 브로커 재시작")
print("   4. acks=all 메시지는 남아있지만, acks=0 메시지는 유실될 수 있음")

## 🎓 Part 3: 실무 가이드

### 3.1 어떤 설정을 선택할까?

#### 질문 1: "이 데이터가 유실되면 어떻게 되나요?"

- **문제없음** → `acks=0`
  - 예: 웹사이트 방문 로그, 센서 데이터

- **약간 문제** → `acks=1`
  - 예: 사용자 활동 로그, 알림 메시지

- **큰 문제** → `acks=all`
  - 예: 주문, 결제, 금융 거래

#### 질문 2: "처리량이 더 중요한가요, 안정성이 더 중요한가요?"

- **처리량 > 안정성** → `acks=0` 또는 `acks=1`
- **안정성 > 처리량** → `acks=all`

### 3.2 실무 권장 설정

#### E-commerce 주문 시스템

```python
# 주문 Producer (절대 유실 불가!)
config = {
    'bootstrap.servers': 'broker:9092',
    'acks': 'all',              # 모든 복제 완료
    'retries': 10,              # 실패 시 10번 재시도
    'max.in.flight.requests.per.connection': 1  # 순서 보장
}
```

#### 로그 수집 시스템

```python
# 로그 Producer (빠른 처리 중요)
config = {
    'bootstrap.servers': 'broker:9092',
    'acks': 0,                  # 응답 안 기다림
    'compression.type': 'snappy'  # 압축으로 대역폭 절약
}
```

#### 실시간 분석 파이프라인

```python
# 이벤트 Producer (중간 수준)
config = {
    'bootstrap.servers': 'broker:9092',
    'acks': 1,                  # Leader만 확인
    'linger.ms': 10,            # 10ms 동안 배치 대기
    'batch.size': 32768         # 32KB 배치
}
```

### 3.3 FAQ

**Q1. acks=all을 사용하면 항상 안전한가요?**

A. 거의 그렇지만, 추가 설정이 필요합니다:
- `min.insync.replicas`: 최소 ISR 개수 설정 (기본 1, 권장 2)
- `replication.factor`: 복제 개수 (최소 3 권장)
- 예: Leader + 2 Follower (총 3개)

---

**Q2. acks=0을 사용하면 정말 빠른가요?**

A. 네, 하지만:
- 네트워크가 느리면 효과가 적음
- 메시지가 큰 경우 `compression.type` 설정도 고려
- 배치 설정(`linger.ms`, `batch.size`)과 함께 사용

---

**Q3. acks=1이 기본값인 이유는?**

A. 대부분의 사용 사례에 적합한 균형:
- 합리적인 성능
- 기본적인 안정성
- Leader 장애 시 문제가 될 수 있지만, 보통 빠르게 복구됨

---

**Q4. 운영 중에 acks 설정을 변경할 수 있나요?**

A. 네, Producer 설정이므로 언제든지 변경 가능:
- 코드에서 config 변경
- 재배포
- 브로커 설정 변경 불필요

---

**Q5. acks=all을 써도 메시지가 유실될 수 있나요?**

A. 극히 드문 경우:
- 모든 ISR이 동시에 다운 (매우 드묾)
- `min.insync.replicas`가 1이고 Follower가 모두 다운
- 해결: `min.insync.replicas=2` + `replication.factor=3` 설정

## 🎉 정리

### 핵심 내용 요약

1. **acks 설정의 의미**
   - `acks=0`: 응답 안 기다림 (빠름, 불안정)
   - `acks=1`: Leader만 확인 (중간, 기본값)
   - `acks=all`: 모든 ISR 확인 (느림, 안전)

2. **트레이드오프**
   - 속도 ⬆ = 안정성 ⬇
   - 안정성 ⬆ = 속도 ⬇

3. **실무 가이드**
   - 유실 허용: `acks=0` (로그, 메트릭)
   - 일반적: `acks=1` (이벤트)
   - 중요 데이터: `acks=all` (주문, 결제)

### 다음 단계

지금까지 배운 내용:
- ✅ Kafka 개념 및 환경 설정
- ✅ Producer 기초
- ✅ Consumer 기초
- ✅ Flask 비동기 전환
- ✅ 실습 과제
- ✅ 데이터 파이프라인
- ✅ acks 설정과 신뢰성

**다음 교시**: 전체 내용 복습 및 Q&A

**선택 학습**: 종합 프로젝트 (시간 여유가 있는 경우)